In [1]:
# ============================================================
# DATA-DRIVEN MULTI-BAND DRIFT GROUPING + TIME/FREQUENCY HYBRID SIMILARITY
#
# This version keeps the original time-based drift peak logic, but changes
# the band merging stage so that clustering is driven by a combined similarity:
#
#   S_total = alpha * S_time + beta * S_freq + gamma * S_adjacency
#
# where:
#   - S_time comes from the existing drift-based similarity
#   - S_freq comes from stable per-band spectral descriptors
#   - S_adjacency mildly prefers merging nearby bands
#
# The rest of the downstream pipeline is kept close to the original baseline.
# ============================================================

import os
import glob
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from scipy.stats import pearsonr, spearmanr
from sklearn.mixture import GaussianMixture

# -----------------------------
# INPUTS
# -----------------------------
# Batch mode inputs
# DATA_ROOT should contain one folder per site/session/recording set.
# Each inner folder should contain one recordings/audio folder and one
# Raven Pro annotation/selection-table folder, for example:
#
# DATA_ROOT/
#   Site_1/
#     recordings/
#       file1.wav
#     annotations/
#       file1.txt
#   Site_2/
#     audio/
#       file2.wav
#     raven/
#       file2_SelectionTable.txt
#
# The folder names are flexible. See RECORDING_SUBDIR_CANDIDATES and
# ANNOTATION_SUBDIR_CANDIDATES below.
DATA_ROOT = "D:\WABAD"
OUTPUT_DIR = "D:\WABAD/batch_gmm_outputs"

# Single-file fallback. Used only if no nested datasets are found.
WAV_PATH = "ARD_20211027_072000.wav"
GT_PATH = None  # Optional Raven-style annotation txt/csv for confirmatory box metrics

# Inner folder name candidates. Add your exact folder names here if needed.
RECORDING_SUBDIR_CANDIDATES = (
    "Recordings",
    "recording",
    "audio",
    "audios",
    "wav",
    "wavs",
    "sounds",
)

ANNOTATION_SUBDIR_CANDIDATES = (
    "annotations",
    "annotation",
    "raven",
    "raven_annotations",
    "Raven Pro annotations",
    "raven_pro",
    "selection_tables",
    "selection tables",
    "gt",
    "ground_truth",
)

AUDIO_EXTENSIONS = ("*.wav", "*.WAV", "*.flac", "*.FLAC")
GT_EXTENSIONS = (".txt", ".csv", ".tsv")

# If True, audio and annotation files can be inside subfolders under the
# discovered recordings/annotations folders.
RECURSIVE_AUDIO_SEARCH = True
RECURSIVE_GT_SEARCH = True

# If True, skip files that already have completed output from a previous run.
# This makes the script restart-safe for large folders.
SKIP_COMPLETED_FILES = True

# Completion is checked using both the predicted boxes CSV and a small done marker.
# The done marker is written only after the file finishes successfully.
# If you have older outputs from before this marker existed, set this to False
# so the script can skip based on the predicted boxes CSV alone.
REQUIRE_DONE_MARKER_FOR_SKIP = False

# Write a per-file summary CSV inside each output folder. This helps preserve
# metrics for completed files even if the full batch later crashes.
SAVE_PER_FILE_SUMMARY = True

# If True, optimise GMM post-cluster parameters per file. Useful for exploration
# and picking promising examples, but it uses GT when available, so report fixed
# parameter results separately for generalisability claims.
RUN_PER_FILE_GRID_SEARCH = False
SAVE_PER_FILE_PLOTS = True
SAVE_CLUSTER_DRIFT_PLOTS = True
SAVE_PREDICTED_BOX_CSVS = True

# GT matching
GT_IOU_THRESHOLD = 0.30

# Grid-search values used only when RUN_PER_FILE_GRID_SEARCH=True
GRID_GMM_MIN_EVENT_POSTERIOR = [0.35, 0.40, 0.45, 0.50, 0.55]
GRID_GMM_SMOOTH_LABELS_K = [1, 3, 5]
GRID_GMM_MERGE_GAP_WINDOWS = [1, 2, 3, 5]
GRID_GMM_MIN_EVENT_WINDOWS = [1, 2]
GRID_GMM_USE_LOG_DRIFT = [False, True]
GRID_GMM_MIN_EVENT_AREA_Q = [None, 0.05, 0.10]

# When no GT exists, rank grid-search configurations using a proxy objective.
PROXY_CORR_WEIGHT = 0.80
PROXY_PEAK_F1_WEIGHT = 0.20
PROXY_BOX_DENSITY_PENALTY = 0.03

TARGET_SR = 22050

# Micro-windowing
WINDOW_S = 0.25
HOP_S = 0.125

# Spectral representation used for per-band drift
N_FFT = 1024
HOP_LENGTH = 256
N_MELS_FULL = 128
FMIN = 200.0
FMAX = 10000.0

# Fine band decomposition
N_FINE_BANDS = 20
BAND_SCALE = "mel"   # "mel" or "linear"

# Drift smoothing
DRIFT_SMOOTH_K = 7

# Peak / boundary evidence extraction
PEAK_K_MAD = 2.5
PEAK_MIN_SEP_WINDOWS = 2

# Time similarity
SIMILARITY_MODE = "lagged_corr"   # "jaccard_binary" or "lagged_corr"
MAX_LAG_WINDOWS = 2

# New hybrid similarity weights
ALPHA_TIME = 0.70
BETA_FREQ = 0.20
GAMMA_ADJ = 0.10
ADJ_TAU = 2.5

# Frequency descriptor controls
FREQ_DESC_USE_ACTIVITY = True
FREQ_DESC_ACTIVITY_K_MAD = 2.0
FREQ_DESC_NORMALIZE_SUBBAND = True

# Hierarchical clustering
LINKAGE_METHOD = "average"
N_GROUPS = None
DISTANCE_THRESHOLD = 0.55

# Merged cluster drift aggregation
CLUSTER_AGG = "median"   # "median", "mean", "max"

# Plotting
PLOT_SECONDS_LIMIT = None
FIG_DPI = 120

# Spectrogram-event comparison
EVENT_SMOOTH_K = 7
EVENT_PEAK_K_MAD = 2.0
EVENT_PEAK_MIN_SEP_WINDOWS = 2
PEAK_MATCH_TOL_WINDOWS = 2

# Peak -> box controls
BOX_EXPAND_WINDOWS = 0
BOX_MIN_DURATION_WINDOWS = 1
BOX_SMOOTH_ACTIVITY_K = 5

# Post-cluster state segmentation controls
# This replaces threshold/peak-only detection for final boxes.
# Each merged cluster drift is treated as a 2-state problem:
# background vs event. A 2-component GMM is fitted per merged cluster.
POST_CLUSTER_DETECTOR = "gmm_state"
GMM_RANDOM_STATE = 0
GMM_MIN_EVENT_POSTERIOR = 0.50      # 0.50 = standard GMM decision boundary; raise to 0.60-0.70 for stricter boxes
GMM_SMOOTH_LABELS_K = 3             # odd integer; set 1 to disable short flicker smoothing
GMM_MERGE_GAP_WINDOWS = 2           # merge events separated by <= this many windows
GMM_MIN_EVENT_WINDOWS = 2           # remove very short events
GMM_MIN_EVENT_AREA_Q = None         # e.g. 0.10 to remove weakest 10% of event regions, None disables
GMM_USE_LOG_DRIFT = False           # True can help when one cluster has extreme spikes

# Optional preprocessing / denoising controls
DENOISE_METHOD = "hpss_percussive"
# options:
# "none"
# "spectral_gate"
# "hpss_percussive"
# "hpss_residual"
# "median_specsub"

USE_PCEN = True

# spectral gate params
NOISE_PROFILE_SEC = 2.0
SPEC_SUB_FLOOR = 0.1

# HPSS params
HPSS_KERNEL_SIZE = 31
HPSS_POWER = 2.0
HPSS_MARGIN = (1.0, 5.0)

# median spectral subtraction params
SPEC_FLOOR_MODE = "median"
SPEC_FLOOR_STRENGTH = 1.0

# -----------------------------
# UTILS
# -----------------------------

def moving_average(x: np.ndarray, k: int) -> np.ndarray:
    if k <= 1:
        return x.copy()
    pad = k // 2
    xp = np.pad(x, (pad, pad), mode="edge")
    w = np.ones(k, dtype=float) / k
    return np.convolve(xp, w, mode="valid")


def robust_mad(x: np.ndarray) -> float:
    med = np.median(x)
    return float(np.median(np.abs(x - med)) + 1e-12)


def robust_zscore(x: np.ndarray) -> np.ndarray:
    med = np.median(x)
    mad = robust_mad(x)
    return (x - med) / mad


def minmax01(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    mn = np.min(x)
    mx = np.max(x)
    if mx - mn < 1e-12:
        return np.zeros_like(x)
    return (x - mn) / (mx - mn)


def safe_pearson(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if np.std(x) < 1e-12 or np.std(y) < 1e-12:
        return 0.0
    r = pearsonr(x, y)[0]
    if np.isnan(r):
        return 0.0
    return float(r)


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na < 1e-12 or nb < 1e-12:
        return 0.0
    s = float(np.dot(a, b) / (na * nb))
    return max(0.0, min(1.0, 0.5 * (s + 1.0)))


def make_fixed_windows(duration_s: float, window_s: float, hop_s: float):
    starts = []
    ends = []
    t = 0.0
    while t + window_s <= duration_s + 1e-12:
        starts.append(t)
        ends.append(t + window_s)
        t += hop_s
    return np.array(starts, dtype=float), np.array(ends, dtype=float)


def nonmax_suppress_peaks(peaks, scores, radius):
    if radius <= 0 or len(peaks) <= 1:
        return sorted(peaks)
    peaks_sorted = sorted(peaks, key=lambda i: scores[i], reverse=True)
    blocked = np.zeros(len(scores), dtype=bool)
    kept = []
    for p in peaks_sorted:
        if blocked[p]:
            continue
        kept.append(p)
        lo = max(0, p - radius)
        hi = min(len(scores), p + radius + 1)
        blocked[lo:hi] = True
    return sorted(kept)


def detect_peaks_from_drift(drift: np.ndarray, k_mad: float, min_sep_windows: int):
    z = robust_zscore(drift)
    thr = np.median(z) + k_mad * robust_mad(z)

    peaks = []
    for i in range(1, len(z) - 1):
        if z[i] >= thr and z[i] >= z[i - 1] and z[i] >= z[i + 1]:
            peaks.append(i)

    peaks = nonmax_suppress_peaks(peaks, z, radius=min_sep_windows)

    evidence = np.zeros(len(z), dtype=int)
    evidence[peaks] = 1
    return z, evidence, peaks, thr


def band_edges_hz(fmin: float, fmax: float, n_bands: int, scale: str = "mel") -> np.ndarray:
    if scale == "mel":
        m0 = librosa.hz_to_mel(fmin)
        m1 = librosa.hz_to_mel(fmax)
        m_edges = np.linspace(m0, m1, n_bands + 1)
        return librosa.mel_to_hz(m_edges)
    elif scale == "linear":
        return np.linspace(fmin, fmax, n_bands + 1)
    else:
        raise ValueError("scale must be 'mel' or 'linear'")


# -----------------------------
# PREPROCESSING / DENOISING
# -----------------------------

def denoise_spectral_gate(y, sr, n_fft=1024, hop_length=256,
                          noise_profile_sec=2.0, floor=0.1):
    S = librosa.stft(y, n_fft=n_fft, hop_length=hop_length)
    mag = np.abs(S)
    phase = np.angle(S)

    noise_frames = max(1, int(round((noise_profile_sec * sr) / hop_length)))
    noise_frames = min(noise_frames, mag.shape[1])

    noise_mag = np.median(mag[:, :noise_frames], axis=1, keepdims=True)
    mag_denoised = mag - noise_mag
    mag_denoised = np.maximum(mag_denoised, floor * mag)

    S_denoised = mag_denoised * np.exp(1j * phase)
    y_denoised = librosa.istft(S_denoised, hop_length=hop_length, length=len(y))
    return y_denoised


def denoise_hpss_percussive(y, kernel_size=31, power=2.0, margin=(1.0, 5.0)):
    y_h, y_p = librosa.effects.hpss(
        y,
        kernel_size=kernel_size,
        power=power,
        margin=margin
    )
    return y_p


def denoise_hpss_residual(y, kernel_size=31, power=2.0, margin=(1.0, 5.0)):
    y_h, y_p = librosa.effects.hpss(
        y,
        kernel_size=kernel_size,
        power=power,
        margin=margin
    )
    residual = y - y_h
    return residual


def subtract_stationary_band_floor(X, strength=1.0, floor_mode="median"):
    X = np.asarray(X, dtype=float)

    if floor_mode == "median":
        bg = np.median(X, axis=0, keepdims=True)
    elif floor_mode == "mean":
        bg = np.mean(X, axis=0, keepdims=True)
    else:
        raise ValueError("floor_mode must be 'median' or 'mean'")

    out = X - strength * bg
    return np.maximum(out, 0.0)


def apply_waveform_preprocessing(y, sr):
    if DENOISE_METHOD == "none":
        return y

    if DENOISE_METHOD == "spectral_gate":
        print("Applying spectral gate denoising...")
        return denoise_spectral_gate(
            y=y,
            sr=sr,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            noise_profile_sec=NOISE_PROFILE_SEC,
            floor=SPEC_SUB_FLOOR
        )

    if DENOISE_METHOD == "hpss_percussive":
        print("Applying HPSS percussive emphasis...")
        return denoise_hpss_percussive(
            y,
            kernel_size=HPSS_KERNEL_SIZE,
            power=HPSS_POWER,
            margin=HPSS_MARGIN
        )

    if DENOISE_METHOD == "hpss_residual":
        print("Applying HPSS residual emphasis...")
        return denoise_hpss_residual(
            y,
            kernel_size=HPSS_KERNEL_SIZE,
            power=HPSS_POWER,
            margin=HPSS_MARGIN
        )

    if DENOISE_METHOD == "median_specsub":
        return y

    raise ValueError(f"Unsupported DENOISE_METHOD: {DENOISE_METHOD}")


# -----------------------------
# FEATURE EXTRACTION
# -----------------------------

def compute_window_repr_vectors(y, sr, win_starts, win_ends, n_mels, fmin, fmax,
                                use_pcen=False):
    out = []

    for t0, t1 in zip(win_starts, win_ends):
        s0 = int(round(t0 * sr))
        s1 = int(round(t1 * sr))
        seg = y[s0:s1]

        if len(seg) < 16:
            out.append(np.zeros(n_mels, dtype=np.float32))
            continue

        n_fft_eff = min(N_FFT, 2 ** int(np.floor(np.log2(len(seg)))))
        n_fft_eff = max(n_fft_eff, 128)
        hop_eff = min(HOP_LENGTH, max(32, n_fft_eff // 4))

        S = librosa.feature.melspectrogram(
            y=seg,
            sr=sr,
            n_fft=n_fft_eff,
            hop_length=hop_eff,
            n_mels=n_mels,
            fmin=fmin,
            fmax=fmax,
            power=2.0,
        )

        if use_pcen:
            rep = librosa.pcen(S * (2.0 ** 16), sr=sr, hop_length=hop_eff)
        else:
            rep = librosa.power_to_db(S, ref=np.max)

        vec = rep.mean(axis=1)
        out.append(vec.astype(np.float32))

    return np.stack(out, axis=0)


def split_mel_vectors_into_bands(logmel_vectors, mel_freqs, edges_hz):
    band_vectors = []
    for bi in range(len(edges_hz) - 1):
        lo, hi = edges_hz[bi], edges_hz[bi + 1]
        if bi < len(edges_hz) - 2:
            mask = (mel_freqs >= lo) & (mel_freqs < hi)
        else:
            mask = (mel_freqs >= lo) & (mel_freqs <= hi)

        if np.sum(mask) == 0:
            band_vectors.append(np.zeros((logmel_vectors.shape[0], 1), dtype=np.float32))
        else:
            band_vectors.append(logmel_vectors[:, mask])
    return band_vectors


def compute_direct_band_drifts(band_vectors, smooth_k):
    drifts = []
    for Xb in band_vectors:
        d = np.zeros(Xb.shape[0], dtype=float)
        for t in range(1, Xb.shape[0]):
            d[t] = np.linalg.norm(Xb[t] - Xb[t - 1])
        ds = moving_average(d, smooth_k)
        drifts.append(ds)
    return drifts


def compute_band_frequency_descriptors(band_vectors, k_mad_activity=2.0,
                                       normalize_subband=True,
                                       use_activity=True):
    """
    Stable per-band descriptor.

    Each band gets a compact summary vector built from:
    - median energy per sub-bin
    - variability per sub-bin (MAD)
    - global stats: median, MAD, max, flatness proxy, activity proportion,
      burstiness, temporal sparsity, mean pairwise sub-bin correlation
    """
    descs = []

    for Xb in band_vectors:
        X = np.asarray(Xb, dtype=float)
        if X.ndim == 1:
            X = X[:, None]

        if normalize_subband:
            Xn = np.zeros_like(X)
            for j in range(X.shape[1]):
                col = X[:, j]
                Xn[:, j] = robust_zscore(col)
            X_work = Xn
        else:
            X_work = X

        med_sub = np.median(X_work, axis=0)
        mad_sub = np.median(np.abs(X_work - med_sub[None, :]), axis=0)

        collapsed = np.mean(X_work, axis=1)
        collapsed_z = robust_zscore(collapsed)
        thr = np.median(collapsed_z) + k_mad_activity * robust_mad(collapsed_z)
        active = (collapsed_z >= thr).astype(float)

        if use_activity:
            activity_prop = float(np.mean(active))
            runs = np.diff(np.pad(active.astype(int), (1, 1), mode='constant'))
            starts = np.where(runs == 1)[0]
            ends = np.where(runs == -1)[0]
            durations = ends - starts
            mean_run = float(np.mean(durations)) if len(durations) > 0 else 0.0
            n_runs = float(len(durations))
        else:
            activity_prop = 0.0
            mean_run = 0.0
            n_runs = 0.0

        global_median = float(np.median(collapsed))
        global_mad = float(robust_mad(collapsed))
        global_max = float(np.max(collapsed))
        global_mean = float(np.mean(collapsed))
        global_std = float(np.std(collapsed))
        burstiness = float(global_std / (abs(global_mean) + 1e-12))
        sparsity = float(np.mean(np.abs(collapsed_z) < 0.5))

        # Flatness-like ratio over positive-shifted data
        xp = collapsed - np.min(collapsed) + 1e-6
        gmean = float(np.exp(np.mean(np.log(xp))))
        amean = float(np.mean(xp))
        flatness = float(gmean / (amean + 1e-12))

        # Correlation between sub-bins inside the band
        if X_work.shape[1] >= 2:
            C = np.corrcoef(X_work.T)
            vals = C[np.triu_indices_from(C, k=1)]
            vals = vals[np.isfinite(vals)]
            mean_subcorr = float(np.mean(vals)) if len(vals) > 0 else 0.0
        else:
            mean_subcorr = 1.0

        desc = np.concatenate([
            med_sub,
            mad_sub,
            np.array([
                global_median,
                global_mad,
                global_max,
                burstiness,
                sparsity,
                flatness,
                activity_prop,
                mean_run,
                n_runs,
                mean_subcorr,
            ], dtype=float)
        ])
        descs.append(desc)

    max_len = max(len(d) for d in descs)
    out = []
    for d in descs:
        if len(d) < max_len:
            pad = np.zeros(max_len - len(d), dtype=float)
            d = np.concatenate([d, pad])
        out.append(d)
    return np.stack(out, axis=0)


# -----------------------------
# SIMILARITY / CLUSTERING
# -----------------------------

def jaccard_similarity_binary(a: np.ndarray, b: np.ndarray) -> float:
    inter = np.sum((a > 0) & (b > 0))
    union = np.sum((a > 0) | (b > 0))
    if union == 0:
        return 0.0
    return float(inter / union)


def lagged_corr_similarity(a: np.ndarray, b: np.ndarray, max_lag: int = 2) -> float:
    a = robust_zscore(a)
    b = robust_zscore(b)
    best = -1.0
    for lag in range(-max_lag, max_lag + 1):
        if lag < 0:
            aa = a[-lag:]
            bb = b[:len(aa)]
        elif lag > 0:
            aa = a[:-lag]
            bb = b[lag:]
        else:
            aa = a
            bb = b

        if len(aa) < 3:
            continue

        c = np.corrcoef(aa, bb)[0, 1]
        if np.isnan(c):
            c = 0.0
        best = max(best, c)

    return float(max(best, 0.0))


def build_time_similarity_matrix(drifts, evidences, mode="jaccard_binary", max_lag=2):
    n = len(drifts)
    S = np.eye(n, dtype=float)

    for i in range(n):
        for j in range(i + 1, n):
            if mode == "jaccard_binary":
                sim = jaccard_similarity_binary(evidences[i], evidences[j])
            elif mode == "lagged_corr":
                sim = lagged_corr_similarity(drifts[i], drifts[j], max_lag=max_lag)
            else:
                raise ValueError("Unsupported similarity mode")

            S[i, j] = sim
            S[j, i] = sim

    return S


def build_frequency_similarity_matrix(band_descs):
    n = band_descs.shape[0]
    S = np.eye(n, dtype=float)

    # robust standardization across bands
    X = np.asarray(band_descs, dtype=float)
    Xs = np.zeros_like(X)
    for j in range(X.shape[1]):
        col = X[:, j]
        Xs[:, j] = robust_zscore(col)

    for i in range(n):
        for j in range(i + 1, n):
            sim = cosine_sim(Xs[i], Xs[j])
            S[i, j] = sim
            S[j, i] = sim
    return S


def build_adjacency_similarity_matrix(n_bands, tau=2.5):
    S = np.eye(n_bands, dtype=float)
    for i in range(n_bands):
        for j in range(i + 1, n_bands):
            d = abs(i - j)
            sim = float(np.exp(-d / tau))
            S[i, j] = sim
            S[j, i] = sim
    return S


def combine_similarity_matrices(S_time, S_freq, S_adj,
                                alpha=0.7, beta=0.2, gamma=0.1):
    wsum = alpha + beta + gamma
    if wsum <= 0:
        raise ValueError("alpha + beta + gamma must be > 0")
    St = alpha / wsum
    Sf = beta / wsum
    Sa = gamma / wsum
    S = St * S_time + Sf * S_freq + Sa * S_adj
    S = np.clip(S, 0.0, 1.0)
    np.fill_diagonal(S, 1.0)
    return S


def cluster_bands_from_similarity(S, linkage_method="average", n_groups=None, distance_threshold=0.55):
    D = 1.0 - S
    np.fill_diagonal(D, 0.0)
    condensed = squareform(D, checks=False)
    Z = linkage(condensed, method=linkage_method)

    if n_groups is not None:
        labels = fcluster(Z, n_groups, criterion="maxclust")
    else:
        labels = fcluster(Z, distance_threshold, criterion="distance")

    return Z, labels


def aggregate_cluster_drifts(drifts, cluster_labels, agg="median"):
    uniq = sorted(np.unique(cluster_labels))
    cluster_drifts = []
    cluster_members = []

    for c in uniq:
        idx = np.where(cluster_labels == c)[0]
        M = np.stack([drifts[i] for i in idx], axis=0)

        if agg == "median":
            cd = np.median(M, axis=0)
        elif agg == "mean":
            cd = np.mean(M, axis=0)
        elif agg == "max":
            cd = np.max(M, axis=0)
        else:
            raise ValueError("agg must be median, mean, or max")

        cluster_drifts.append(cd)
        cluster_members.append(idx.tolist())

    return uniq, cluster_drifts, cluster_members


def reorder_by_dendrogram(Z):
    dend = dendrogram(Z, no_plot=True)
    return dend["leaves"]


def cluster_band_ranges(edges_hz, cluster_labels):
    uniq = sorted(np.unique(cluster_labels))
    out = {}
    for c in uniq:
        idx = np.where(cluster_labels == c)[0]
        lo = edges_hz[np.min(idx)]
        hi = edges_hz[np.max(idx) + 1]
        out[c] = (lo, hi, idx.tolist())
    return out


# -----------------------------
# EVENT CURVE FROM SPECTROGRAM
# -----------------------------

def compute_global_repr_vectors(y, sr, win_starts, win_ends, n_mels, fmin, fmax):
    return compute_window_repr_vectors(
        y=y,
        sr=sr,
        win_starts=win_starts,
        win_ends=win_ends,
        n_mels=n_mels,
        fmin=fmin,
        fmax=fmax,
        use_pcen=USE_PCEN
    )


def compute_spectrogram_event_curve(global_repr_vectors, smooth_k=7):
    d = np.zeros(global_repr_vectors.shape[0], dtype=float)
    for t in range(1, global_repr_vectors.shape[0]):
        diff = global_repr_vectors[t] - global_repr_vectors[t - 1]
        d[t] = np.linalg.norm(diff)
    d = moving_average(d, smooth_k)
    return d


def match_peaks_with_tolerance(ref_peaks, test_peaks, tol=2):
    ref_peaks = sorted(ref_peaks)
    test_peaks = sorted(test_peaks)

    matched_ref = set()
    matched_test = set()

    for i, rp in enumerate(ref_peaks):
        best_j = None
        best_dist = None
        for j, tp in enumerate(test_peaks):
            if j in matched_test:
                continue
            dist = abs(rp - tp)
            if dist <= tol and (best_dist is None or dist < best_dist):
                best_j = j
                best_dist = dist
        if best_j is not None:
            matched_ref.add(i)
            matched_test.add(best_j)

    tp = len(matched_test)
    fp = len(test_peaks) - tp
    fn = len(ref_peaks) - tp

    precision = tp / (tp + fp + 1e-12)
    recall = tp / (tp + fn + 1e-12)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def best_lagged_corr(a, b, max_lag=2):
    a = robust_zscore(a)
    b = robust_zscore(b)

    best_corr = -1.0
    best_lag = 0

    for lag in range(-max_lag, max_lag + 1):
        if lag < 0:
            aa = a[-lag:]
            bb = b[:len(aa)]
        elif lag > 0:
            aa = a[:-lag]
            bb = b[lag:]
        else:
            aa = a
            bb = b

        if len(aa) < 3:
            continue

        c = np.corrcoef(aa, bb)[0, 1]
        if np.isnan(c):
            c = 0.0

        if c > best_corr:
            best_corr = c
            best_lag = lag

    return float(best_corr), int(best_lag)


def compute_cluster_vs_event_metrics(cluster_drifts, event_curve, max_lag=2,
                                     drift_peak_k_mad=2.5, drift_peak_sep=2,
                                     event_peak_k_mad=2.0, event_peak_sep=2,
                                     peak_match_tol=2):
    metrics = []

    ev_z, ev_bin, ev_peaks, ev_thr = detect_peaks_from_drift(
        event_curve,
        k_mad=event_peak_k_mad,
        min_sep_windows=event_peak_sep
    )

    for i, cd in enumerate(cluster_drifts):
        x = robust_zscore(cd)
        y = robust_zscore(event_curve)

        if np.std(x) < 1e-12 or np.std(y) < 1e-12:
            pear = 0.0
            spear = 0.0
        else:
            pear = pearsonr(x, y)[0]
            spear = spearmanr(x, y).statistic
            if np.isnan(pear):
                pear = 0.0
            if np.isnan(spear):
                spear = 0.0

        lagcorr, bestlag = best_lagged_corr(cd, event_curve, max_lag=max_lag)

        cd_z, cd_bin, cd_peaks, cd_thr = detect_peaks_from_drift(
            cd,
            k_mad=drift_peak_k_mad,
            min_sep_windows=drift_peak_sep
        )

        peak_scores = match_peaks_with_tolerance(
            ref_peaks=ev_peaks,
            test_peaks=cd_peaks,
            tol=peak_match_tol
        )

        metrics.append({
            "cluster_index": i,
            "pearson_r": float(pear),
            "spearman_rho": float(spear),
            "max_lag_corr": float(lagcorr),
            "best_lag_windows": int(bestlag),
            "n_cluster_peaks": int(len(cd_peaks)),
            "n_event_peaks": int(len(ev_peaks)),
            "peak_precision": float(peak_scores["precision"]),
            "peak_recall": float(peak_scores["recall"]),
            "peak_f1": float(peak_scores["f1"]),
            "cluster_peaks": cd_peaks,
            "event_peaks": ev_peaks,
            "cluster_drift_z": cd_z,
            "event_curve_z": ev_z,
        })

    return metrics


# -----------------------------
# PEAK -> BOX LOGIC
# -----------------------------

def find_peak_support(z, peak_idx, thr, expand_windows=0):
    n = len(z)

    left = peak_idx
    while left > 0 and z[left] >= thr:
        left -= 1
    if z[left] < thr and left < peak_idx:
        left += 1

    right = peak_idx
    while right < n - 1 and z[right] >= thr:
        right += 1
    if z[right] < thr and right > peak_idx:
        right -= 1

    left = max(0, left - expand_windows)
    right = min(n - 1, right + expand_windows)

    if right < left:
        left = max(0, peak_idx - 1)
        right = min(n - 1, peak_idx + 1)

    return left, right


def merge_overlapping_time_boxes(boxes):
    if len(boxes) == 0:
        return []

    boxes = sorted(boxes, key=lambda b: (b["cluster_id"], b["t0"], b["t1"]))
    merged = [boxes[0].copy()]

    for b in boxes[1:]:
        prev = merged[-1]
        same_cluster = (prev["cluster_id"] == b["cluster_id"])
        overlapping = b["t0"] <= prev["t1"]

        if same_cluster and overlapping:
            prev["t1"] = max(prev["t1"], b["t1"])
        else:
            merged.append(b.copy())

    return merged



def smooth_binary_labels(labels, k=3):
    """Simple majority filter for binary labels."""
    labels = np.asarray(labels, dtype=int)
    if k <= 1:
        return labels.copy()
    if k % 2 == 0:
        k += 1
    pad = k // 2
    padded = np.pad(labels, (pad, pad), mode="edge")
    out = np.zeros_like(labels)
    for i in range(len(labels)):
        window = padded[i:i + k]
        out[i] = 1 if np.mean(window) >= 0.5 else 0
    return out


def binary_labels_to_regions(labels, merge_gap_windows=2, min_event_windows=2):
    """Convert 0/1 labels into contiguous event regions."""
    labels = np.asarray(labels, dtype=int)
    regions = []
    in_event = False
    start = None

    for i, lab in enumerate(labels):
        if lab == 1 and not in_event:
            in_event = True
            start = i
        elif lab == 0 and in_event:
            regions.append([start, i - 1])
            in_event = False

    if in_event:
        regions.append([start, len(labels) - 1])

    # Merge nearby regions
    if merge_gap_windows is not None and merge_gap_windows >= 0 and len(regions) > 1:
        merged = [regions[0]]
        for st, en in regions[1:]:
            prev = merged[-1]
            gap = st - prev[1] - 1
            if gap <= merge_gap_windows:
                prev[1] = en
            else:
                merged.append([st, en])
        regions = merged

    # Drop very short regions
    regions = [[st, en] for st, en in regions if (en - st + 1) >= min_event_windows]
    return regions


def fit_gmm_event_state(drift,
                        min_event_posterior=0.50,
                        smooth_labels_k=3,
                        merge_gap_windows=2,
                        min_event_windows=2,
                        min_event_area_q=None,
                        use_log_drift=False,
                        random_state=0):
    """
    Per-cluster 2-state segmentation using a Gaussian mixture model.

    The lower-mean component is interpreted as background and the higher-mean
    component as event. This estimates the non-event/event state from each
    post-cluster drift curve rather than using a manually chosen drift threshold.
    """
    raw = np.asarray(drift, dtype=float)

    # Optional compression for highly spiky clusters.
    if use_log_drift:
        shift = np.min(raw)
        x = np.log1p(raw - shift)
    else:
        x = raw.copy()

    if len(x) < 5 or np.std(x) < 1e-12:
        labels = np.zeros(len(x), dtype=int)
        return {
            "labels": labels,
            "regions": [],
            "threshold": float(np.inf),
            "means": np.array([float(np.mean(x)), float(np.mean(x))]),
            "weights": np.array([1.0, 0.0]),
            "event_component": 1,
            "posterior": np.zeros(len(x), dtype=float),
            "raw_event_areas": [],
        }

    X = x.reshape(-1, 1)
    gmm = GaussianMixture(
        n_components=2,
        covariance_type="full",
        random_state=random_state,
        reg_covar=1e-6,
        n_init=5,
    )
    gmm.fit(X)

    means = gmm.means_.flatten()
    weights = gmm.weights_.flatten()
    event_component = int(np.argmax(means))
    bg_component = 1 - event_component

    posterior = gmm.predict_proba(X)[:, event_component]
    labels = (posterior >= min_event_posterior).astype(int)
    labels = smooth_binary_labels(labels, k=smooth_labels_k)

    regions = binary_labels_to_regions(
        labels,
        merge_gap_windows=merge_gap_windows,
        min_event_windows=min_event_windows,
    )

    # Optional region-level area filtering. This is intentionally weak by default.
    raw_event_areas = []
    for st, en in regions:
        local_base = min(raw[st], raw[en]) if en > st else raw[st]
        area = float(np.sum(np.maximum(raw[st:en + 1] - local_base, 0.0)))
        raw_event_areas.append(area)

    if min_event_area_q is not None and len(regions) > 0:
        area_gate = float(np.quantile(raw_event_areas, min_event_area_q))
        kept_regions = []
        kept_areas = []
        for region, area in zip(regions, raw_event_areas):
            if area >= area_gate:
                kept_regions.append(region)
                kept_areas.append(area)
        regions = kept_regions
        raw_event_areas = kept_areas

        labels2 = np.zeros_like(labels)
        for st, en in regions:
            labels2[st:en + 1] = 1
        labels = labels2

    # Approximate threshold in the same space used by the GMM.
    # This is diagnostic only; boxes are built from posterior labels/regions.
    threshold = float(np.mean(means))
    if use_log_drift:
        threshold = float(np.expm1(threshold) + np.min(raw))

    return {
        "labels": labels,
        "regions": regions,
        "threshold": threshold,
        "means": means,
        "weights": weights,
        "event_component": event_component,
        "bg_component": bg_component,
        "posterior": posterior,
        "raw_event_areas": raw_event_areas,
    }


def cluster_peaks_to_boxes(cluster_drifts, cluster_members, cluster_labels_unique,
                           edges_hz, win_starts, win_ends,
                           peak_k_mad, peak_min_sep_windows,
                           expand_windows=0, min_duration_windows=1):
    all_boxes = []
    peak_info = []

    print("\n=== POST-CLUSTER GMM STATE SEGMENTATION ===")

    for cd, members, c in zip(cluster_drifts, cluster_members, cluster_labels_unique):
        f0 = float(edges_hz[min(members)])
        f1 = float(edges_hz[max(members) + 1])

        res = fit_gmm_event_state(
            cd,
            min_event_posterior=GMM_MIN_EVENT_POSTERIOR,
            smooth_labels_k=GMM_SMOOTH_LABELS_K,
            merge_gap_windows=GMM_MERGE_GAP_WINDOWS,
            min_event_windows=max(GMM_MIN_EVENT_WINDOWS, min_duration_windows),
            min_event_area_q=GMM_MIN_EVENT_AREA_Q,
            use_log_drift=GMM_USE_LOG_DRIFT,
            random_state=GMM_RANDOM_STATE,
        )

        labels = res["labels"]
        regions = res["regions"]

        cluster_boxes = []
        peaks = []
        for st, en in regions:
            segment = cd[st:en + 1]
            peak_idx = int(st + np.argmax(segment))
            peaks.append(peak_idx)

            box = {
                "cluster_id": int(c),
                "t0": float(win_starts[st]),
                "t1": float(win_ends[en]),
                "f0": f0,
                "f1": f1,
                "peak_idx": int(peak_idx),
                "support_left": int(st),
                "support_right": int(en),
                "members": members
            }
            cluster_boxes.append(box)

        cluster_boxes = merge_overlapping_time_boxes(cluster_boxes)
        all_boxes.extend(cluster_boxes)

        means_str = np.array2string(res["means"], precision=3, separator=", ")
        weights_str = np.array2string(res["weights"], precision=3, separator=", ")
        event_fraction = float(np.mean(labels)) if len(labels) else 0.0
        print(
            f"Cluster {c} | {int(f0):4d}-{int(f1):5d} Hz | "
            f"mode=gmm_state | thr≈{res['threshold']:.3f} | "
            f"means={means_str} | weights={weights_str} | "
            f"event_fraction={event_fraction:.3f} | regions={len(regions)} | boxes={len(cluster_boxes)}"
        )

        peak_info.append({
            "cluster_id": int(c),
            "freq_lo": f0,
            "freq_hi": f1,
            "n_peaks": int(len(peaks)),
            "n_boxes": int(len(cluster_boxes)),
            "peaks": peaks,
            "z": labels.astype(float),
            "thr": res["threshold"],
            "labels": labels,
            "posterior": res["posterior"],
            "regions": regions,
            "means": res["means"],
            "weights": res["weights"],
        })

    return all_boxes, peak_info

def boxes_to_global_activity_curve(boxes, n_windows, win_starts, win_ends, smooth_k=1):
    activity = np.zeros(n_windows, dtype=float)

    for b in boxes:
        overlap = (win_starts < b["t1"]) & (win_ends > b["t0"])
        activity[overlap] = 1.0

    if smooth_k > 1:
        activity = moving_average(activity, smooth_k)

    return activity


# -----------------------------
# OPTIONAL GT LOADING + SIMPLE BOX METRICS
# -----------------------------

def try_load_ground_truth_boxes(path):
    if path is None:
        return []
    if not os.path.exists(path):
        print(f"GT file not found: {path}")
        return []

    sep = '\t' if path.lower().endswith('.txt') else None
    df = pd.read_csv(path, sep=sep, engine='python')
    cols = {c.lower().strip(): c for c in df.columns}

    def pick(names):
        for n in names:
            if n in cols:
                return cols[n]
        return None

    c_t0 = pick(["begin time (s)", "begin time", "start time", "t0"])
    c_t1 = pick(["end time (s)", "end time", "stop time", "t1"])
    c_f0 = pick(["low freq (hz)", "low freq", "f0", "low frequency"])
    c_f1 = pick(["high freq (hz)", "high freq", "f1", "high frequency"])

    if any(c is None for c in [c_t0, c_t1, c_f0, c_f1]):
        print("Could not identify GT columns. Expected time/frequency box columns.")
        return []

    boxes = []
    for _, row in df.iterrows():
        boxes.append({
            "t0": float(row[c_t0]),
            "t1": float(row[c_t1]),
            "f0": float(row[c_f0]),
            "f1": float(row[c_f1]),
        })
    return boxes


def interval_iou(a0, a1, b0, b1):
    inter = max(0.0, min(a1, b1) - max(a0, b0))
    union = max(a1, b1) - min(a0, b0)
    if union <= 0:
        return 0.0
    return inter / union


def box_iou(a, b):
    it = max(0.0, min(a["t1"], b["t1"]) - max(a["t0"], b["t0"]))
    ifr = max(0.0, min(a["f1"], b["f1"]) - max(a["f0"], b["f0"]))
    inter = it * ifr
    area_a = max(0.0, a["t1"] - a["t0"]) * max(0.0, a["f1"] - a["f0"])
    area_b = max(0.0, b["t1"] - b["t0"]) * max(0.0, b["f1"] - b["f0"])
    union = area_a + area_b - inter
    if union <= 0:
        return 0.0
    return inter / union


def evaluate_predicted_boxes(pred_boxes, gt_boxes, iou_thr=0.3):
    if len(gt_boxes) == 0:
        return None

    matched_gt = set()
    matched_pred = set()
    ious = []

    for i, pb in enumerate(pred_boxes):
        best_j = None
        best_iou = 0.0
        for j, gb in enumerate(gt_boxes):
            if j in matched_gt:
                continue
            iou = box_iou(pb, gb)
            if iou > best_iou:
                best_iou = iou
                best_j = j
        if best_j is not None and best_iou >= iou_thr:
            matched_pred.add(i)
            matched_gt.add(best_j)
            ious.append(best_iou)

    tp = len(matched_pred)
    fp = len(pred_boxes) - tp
    fn = len(gt_boxes) - tp
    precision = tp / (tp + fp + 1e-12)
    recall = tp / (tp + fn + 1e-12)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    mean_iou = float(np.mean(ious)) if len(ious) > 0 else 0.0

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mean_iou": mean_iou,
    }


# -----------------------------
# PLOTTING
# -----------------------------

def maybe_limit_time(y, sr, sec_limit):
    if sec_limit is None:
        return y
    n = int(round(sec_limit * sr))
    return y[:n]


def get_time_mask(win_starts, sec_limit):
    if sec_limit is None:
        return np.ones_like(win_starts, dtype=bool)
    return win_starts <= sec_limit


def plot_spectrogram(y, sr, sec_limit=None):
    y2 = maybe_limit_time(y, sr, sec_limit)

    S = librosa.stft(y2, n_fft=2048, hop_length=512)
    S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)

    fig, ax = plt.subplots(figsize=(14, 5), dpi=FIG_DPI)
    img = librosa.display.specshow(
        S_db,
        sr=sr,
        hop_length=512,
        x_axis="time",
        y_axis="hz",
        fmin=0,
        fmax=FMAX,
        cmap="magma",
        ax=ax
    )
    ax.set_title("Spectrogram")
    ax.set_ylim(0, FMAX)
    plt.colorbar(img, ax=ax, format="%+2.0f dB")
    plt.tight_layout()
    plt.show()


def plot_similarity_matrix(S, title="Similarity Matrix"):
    fig, ax = plt.subplots(figsize=(7, 6), dpi=FIG_DPI)
    im = ax.imshow(S, origin="lower", aspect="auto", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel("Band index")
    ax.set_ylabel("Band index")
    plt.colorbar(im, ax=ax, label="Similarity")
    plt.tight_layout()
    plt.show()


def plot_similarity_matrix_reordered(S, order, title="Reordered Similarity Matrix"):
    Sr = S[np.ix_(order, order)]
    fig, ax = plt.subplots(figsize=(7, 6), dpi=FIG_DPI)
    im = ax.imshow(Sr, origin="lower", aspect="auto", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel("Reordered band index")
    ax.set_ylabel("Reordered band index")
    plt.colorbar(im, ax=ax, label="Similarity")
    plt.tight_layout()
    plt.show()


def plot_dendrogram(Z):
    fig, ax = plt.subplots(figsize=(10, 4), dpi=FIG_DPI)
    dendrogram(Z, ax=ax)
    ax.set_title("Hierarchical Clustering of Bands")
    ax.set_xlabel("Band index")
    ax.set_ylabel("Distance")
    plt.tight_layout()
    plt.show()


def plot_cluster_bandmap(edges_hz, cluster_labels):
    uniq = sorted(np.unique(cluster_labels))
    fig, ax = plt.subplots(figsize=(10, 2.2), dpi=FIG_DPI)

    for bi in range(len(edges_hz) - 1):
        lo = edges_hz[bi]
        hi = edges_hz[bi + 1]
        c = cluster_labels[bi]
        ax.fill_between([lo, hi], 0, 1, alpha=0.7,
                        label=f"C{c}" if bi == np.where(cluster_labels == c)[0][0] else None)

    ax.set_xlim(edges_hz[0], edges_hz[-1])
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_xlabel("Frequency (Hz)")
    ax.set_title("Discovered Band Grouping")
    ax.legend(ncol=min(len(uniq), 8), loc="upper center", bbox_to_anchor=(0.5, -0.18))
    plt.tight_layout()
    plt.show()


def plot_cluster_drifts(cluster_drifts, cluster_members, cluster_labels_unique, win_starts, edges_hz, sec_limit=None):
    t = win_starts.copy()
    mask = get_time_mask(t, sec_limit)

    n = len(cluster_drifts)
    fig, axes = plt.subplots(n, 1, figsize=(14, max(2.5 * n, 6)), sharex=True, dpi=FIG_DPI)

    if n == 1:
        axes = [axes]

    for ax, cd, members, c in zip(axes, cluster_drifts, cluster_members, cluster_labels_unique):
        tt = t[mask]
        yy = cd[mask]

        lo = edges_hz[min(members)]
        hi = edges_hz[max(members) + 1]

        ax.plot(tt, yy, linewidth=1.4)
        ax.set_ylabel("Drift")
        ax.set_title(
            f"Cluster {c}: {int(lo)} - {int(hi)} Hz | members={members}"
        )
        ax.grid(alpha=0.25)

    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(f"Merged Cluster Drifts ({CLUSTER_AGG})", y=1.0)
    plt.tight_layout()
    plt.show()


def plot_all_cluster_drifts_overlay(cluster_drifts, cluster_members, cluster_labels_unique,
                                    win_starts, edges_hz, sec_limit=None,
                                    normalize=True, event_curve=None,
                                    activity_curve=None):
    t = win_starts.copy()
    mask = get_time_mask(t, sec_limit)
    tt = t[mask]

    fig, ax = plt.subplots(figsize=(14, 5), dpi=FIG_DPI)

    for cd, members, c in zip(cluster_drifts, cluster_members, cluster_labels_unique):
        lo = edges_hz[min(members)]
        hi = edges_hz[max(members) + 1]
        yy = cd[mask]
        if normalize:
            yy = minmax01(yy)
        ax.plot(tt, yy, linewidth=1.6, label=f"C{c}: {int(lo)}-{int(hi)} Hz")

    if event_curve is not None:
        ev = event_curve[mask]
        if normalize:
            ev = minmax01(ev)
        ax.plot(tt, ev, linewidth=2.5, linestyle="--", label="Spectrogram event curve")

    if activity_curve is not None:
        ac = activity_curve[mask]
        if normalize:
            ac = minmax01(ac)
        ax.plot(tt, ac, linewidth=2.0, linestyle=":", label="Box-derived activity")

    ax.set_title("Merged Cluster Drifts + Event Curve + Box Activity")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Normalized magnitude" if normalize else "Magnitude")
    ax.grid(alpha=0.25)
    ax.legend(loc="upper right", fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()


def plot_spectrogram_with_cluster_drifts(y, sr, win_starts, cluster_drifts, cluster_members,
                                         cluster_labels_unique, edges_hz,
                                         event_curve=None, activity_curve=None,
                                         sec_limit=None):
    y2 = maybe_limit_time(y, sr, sec_limit)
    S = librosa.stft(y2, n_fft=2048, hop_length=512)
    S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)

    t = win_starts.copy()
    mask = get_time_mask(t, sec_limit)
    tt = t[mask]

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                             gridspec_kw={"height_ratios": [3, 2]}, dpi=FIG_DPI)

    img = librosa.display.specshow(
        S_db,
        sr=sr,
        hop_length=512,
        x_axis="time",
        y_axis="hz",
        fmin=0,
        fmax=FMAX,
        cmap="magma",
        ax=axes[0]
    )
    axes[0].set_ylim(0, FMAX)
    axes[0].set_title("Spectrogram")
    plt.colorbar(img, ax=axes[0], format="%+2.0f dB")

    for cd, members, c in zip(cluster_drifts, cluster_members, cluster_labels_unique):
        lo = edges_hz[min(members)]
        hi = edges_hz[max(members) + 1]
        yy = minmax01(cd[mask])
        axes[1].plot(tt, yy, linewidth=1.5, label=f"C{c}: {int(lo)}-{int(hi)} Hz")

    if event_curve is not None:
        ev = minmax01(event_curve[mask])
        axes[1].plot(tt, ev, linewidth=2.5, linestyle="--", label="Spectrogram event curve")

    if activity_curve is not None:
        ac = minmax01(activity_curve[mask])
        axes[1].plot(tt, ac, linewidth=2.0, linestyle=":", label="Box-derived activity")

    axes[1].set_title("Merged Cluster Drifts vs Event Curve vs Box Activity")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Normalized magnitude")
    axes[1].grid(alpha=0.25)
    axes[1].legend(loc="upper right", fontsize=8, ncol=2)

    plt.tight_layout()
    plt.show()


def plot_spectrogram_with_discovered_boxes(y, sr, boxes, gt_boxes=None, sec_limit=None):
    y2 = maybe_limit_time(y, sr, sec_limit)
    S = librosa.stft(y2, n_fft=2048, hop_length=512)
    S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)

    fig, ax = plt.subplots(figsize=(14, 6), dpi=FIG_DPI)
    img = librosa.display.specshow(
        S_db,
        sr=sr,
        hop_length=512,
        x_axis="time",
        y_axis="hz",
        fmin=0,
        fmax=FMAX,
        cmap="magma",
        ax=ax
    )
    ax.set_ylim(0, FMAX)
    ax.set_title("Spectrogram with Discovered Peak-Derived Boxes")

    for b in boxes:
        if sec_limit is not None and b["t0"] > sec_limit:
            continue

        rect = plt.Rectangle(
            (b["t0"], b["f0"]),
            b["t1"] - b["t0"],
            b["f1"] - b["f0"],
            fill=False,
            edgecolor="cyan",
            linewidth=1.5
        )
        ax.add_patch(rect)

    if gt_boxes is not None:
        for b in gt_boxes:
            if sec_limit is not None and b["t0"] > sec_limit:
                continue
            rect = plt.Rectangle(
                (b["t0"], b["f0"]),
                b["t1"] - b["t0"],
                b["f1"] - b["f0"],
                fill=False,
                edgecolor="lime",
                linewidth=1.2,
                linestyle="--"
            )
            ax.add_patch(rect)

    plt.colorbar(img, ax=ax, format="%+2.0f dB")
    plt.tight_layout()
    plt.show()


def print_cluster_event_metrics(metrics, cluster_members, cluster_labels_unique, edges_hz):
    print("\n=== CLUSTER vs SPECTROGRAM EVENT METRICS ===")
    for m, members, c in zip(metrics, cluster_members, cluster_labels_unique):
        lo = edges_hz[min(members)]
        hi = edges_hz[max(members) + 1]

        print(
            f"Cluster {c} | {int(lo):4d}-{int(hi):5d} Hz | members={members}\n"
            f"  Pearson r       : {m['pearson_r']:.3f}\n"
            f"  Spearman rho    : {m['spearman_rho']:.3f}\n"
            f"  Max lag corr    : {m['max_lag_corr']:.3f}  (lag={m['best_lag_windows']} windows)\n"
            f"  Peak precision  : {m['peak_precision']:.3f}\n"
            f"  Peak recall     : {m['peak_recall']:.3f}\n"
            f"  Peak F1         : {m['peak_f1']:.3f}\n"
            f"  Cluster peaks   : {m['n_cluster_peaks']}\n"
            f"  Event peaks     : {m['n_event_peaks']}"
        )


def print_similarity_diagnostics(S_time, S_freq, S_adj, S_total):
    print("\n=== SIMILARITY MATRIX DIAGNOSTICS ===")
    for name, S in [("Time", S_time), ("Freq", S_freq), ("Adj", S_adj), ("Total", S_total)]:
        off = S[~np.eye(S.shape[0], dtype=bool)]
        print(f"{name:>5s} | min={off.min():.3f} mean={off.mean():.3f} max={off.max():.3f}")



# -----------------------------
# BATCH HELPERS
# -----------------------------

def normalise_folder_name(name):
    """Normalise folder names for flexible matching."""
    return name.lower().replace("-", "_").replace(" ", "_").strip()


def find_candidate_subdir(parent_dir, candidate_names):
    """
    Finds the first matching subfolder inside parent_dir using flexible,
    case-insensitive matching.
    """
    parent_dir = Path(parent_dir)
    if not parent_dir.exists():
        return None

    candidate_norms = {normalise_folder_name(c) for c in candidate_names}

    for child in parent_dir.iterdir():
        if child.is_dir() and normalise_folder_name(child.name) in candidate_norms:
            return child

    return None


def find_audio_files(audio_folder):
    """Find audio files inside a discovered recordings/audio folder."""
    audio_folder = Path(audio_folder)
    if not audio_folder.exists():
        return []

    paths = []
    for ext in AUDIO_EXTENSIONS:
        if RECURSIVE_AUDIO_SEARCH:
            paths.extend(audio_folder.rglob(ext))
        else:
            paths.extend(audio_folder.glob(ext))

    return sorted(set(paths))


def discover_dataset_folders(data_root):
    """
    Discover folders containing recordings and Raven Pro annotation folders.

    Expected structure:
        DATA_ROOT/
            dataset_or_site_1/
                recordings/
                annotations/
            dataset_or_site_2/
                audio/
                raven/

    The function also supports DATA_ROOT itself being a single dataset folder.
    """
    data_root = Path(data_root)
    if not data_root.exists():
        raise FileNotFoundError(f"DATA_ROOT does not exist: {data_root}")

    candidate_dataset_dirs = [data_root] + [p for p in data_root.iterdir() if p.is_dir()]
    datasets = []
    seen_audio_dirs = set()

    for dataset_dir in candidate_dataset_dirs:
        audio_dir = find_candidate_subdir(dataset_dir, RECORDING_SUBDIR_CANDIDATES)
        gt_dir = find_candidate_subdir(dataset_dir, ANNOTATION_SUBDIR_CANDIDATES)

        if audio_dir is None:
            continue

        audio_files = find_audio_files(audio_dir)
        if not audio_files:
            continue

        resolved_audio_dir = audio_dir.resolve()
        if resolved_audio_dir in seen_audio_dirs:
            continue
        seen_audio_dirs.add(resolved_audio_dir)

        datasets.append({
            "dataset_name": dataset_dir.name,
            "dataset_dir": dataset_dir,
            "audio_dir": audio_dir,
            "gt_dir": gt_dir,
            "audio_files": audio_files,
            "n_audio_files": len(audio_files),
        })

    return datasets


def find_matching_gt(wav_path, gt_folder):
    """
    Finds a Raven-style annotation file for a given audio file.

    Matching strategy:
    1. Exact same stem: recording.wav -> recording.txt/csv/tsv
    2. Common Raven suffixes: recording_SelectionTable.txt, etc.
    3. Loose contains match: if the audio stem appears inside the annotation name.
    """
    if gt_folder is None:
        return None

    gt_dir = Path(gt_folder)
    if not gt_dir.exists():
        return None

    stem = Path(wav_path).stem

    candidates = []
    for ext in GT_EXTENSIONS:
        candidates.extend([
            gt_dir / f"{stem}{ext}",
            gt_dir / f"{stem}_annotations{ext}",
            gt_dir / f"{stem}_annotation{ext}",
            gt_dir / f"{stem}_SelectionTable{ext}",
            gt_dir / f"{stem}_Selection Table{ext}",
            gt_dir / f"{stem}.SelectionTable{ext}",
        ])

    for c in candidates:
        if c.exists():
            return str(c)

    search_iter = gt_dir.rglob("*") if RECURSIVE_GT_SEARCH else gt_dir.glob("*")
    for p in search_iter:
        if p.is_file() and p.suffix.lower() in GT_EXTENSIONS:
            if stem.lower() in p.stem.lower():
                return str(p)

    return None

def save_boxes_csv(boxes, out_csv):
    rows = []
    for b in boxes:
        rows.append({
            "cluster_id": b.get("cluster_id"),
            "t0": b.get("t0"),
            "t1": b.get("t1"),
            "duration_s": b.get("t1", 0) - b.get("t0", 0),
            "f0": b.get("f0"),
            "f1": b.get("f1"),
            "bandwidth_hz": b.get("f1", 0) - b.get("f0", 0),
            "peak_idx": b.get("peak_idx"),
            "support_left": b.get("support_left"),
            "support_right": b.get("support_right"),
            "members": ",".join(map(str, b.get("members", []))),
        })
    pd.DataFrame(rows).to_csv(out_csv, index=False)


def save_spectrogram_with_boxes(y, sr, boxes, gt_boxes, out_png, title, sec_limit=None):
    y2 = maybe_limit_time(y, sr, sec_limit)
    S = librosa.stft(y2, n_fft=2048, hop_length=512)
    S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)

    fig, ax = plt.subplots(figsize=(14, 6), dpi=FIG_DPI)
    img = librosa.display.specshow(
        S_db, sr=sr, hop_length=512, x_axis="time", y_axis="hz",
        fmin=0, fmax=FMAX, cmap="magma", ax=ax
    )
    ax.set_ylim(0, FMAX)
    ax.set_title(title)

    for b in boxes:
        if sec_limit is not None and b["t0"] > sec_limit:
            continue
        ax.add_patch(plt.Rectangle(
            (b["t0"], b["f0"]), b["t1"] - b["t0"], b["f1"] - b["f0"],
            fill=False, edgecolor="cyan", linewidth=1.3
        ))

    if gt_boxes is not None:
        for b in gt_boxes:
            if sec_limit is not None and b["t0"] > sec_limit:
                continue
            ax.add_patch(plt.Rectangle(
                (b["t0"], b["f0"]), b["t1"] - b["t0"], b["f1"] - b["f0"],
                fill=False, edgecolor="lime", linewidth=1.1, linestyle="--"
            ))

    plt.colorbar(img, ax=ax, format="%+2.0f dB")
    plt.tight_layout()
    fig.savefig(out_png, bbox_inches="tight")
    plt.close(fig)


def save_cluster_drifts_plot(cluster_drifts, cluster_members, cluster_labels_unique,
                             win_starts, edges_hz, out_png, title, sec_limit=None):
    t = win_starts.copy()
    mask = get_time_mask(t, sec_limit)
    n = len(cluster_drifts)
    fig, axes = plt.subplots(n, 1, figsize=(14, max(2.3 * n, 6)), sharex=True, dpi=FIG_DPI)
    if n == 1:
        axes = [axes]

    for ax, cd, members, c in zip(axes, cluster_drifts, cluster_members, cluster_labels_unique):
        lo = edges_hz[min(members)]
        hi = edges_hz[max(members) + 1]
        ax.plot(t[mask], cd[mask], linewidth=1.2)
        ax.set_ylabel("Drift")
        ax.set_title(f"Cluster {c}: {int(lo)}-{int(hi)} Hz | members={members}")
        ax.grid(alpha=0.25)

    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(title, y=1.0)
    plt.tight_layout()
    fig.savefig(out_png, bbox_inches="tight")
    plt.close(fig)


def gmm_boxes_from_cluster_drifts(cluster_drifts, cluster_members, cluster_labels_unique,
                                  edges_hz, win_starts, win_ends,
                                  min_event_posterior=0.50,
                                  smooth_labels_k=3,
                                  merge_gap_windows=2,
                                  min_event_windows=2,
                                  min_event_area_q=None,
                                  use_log_drift=False,
                                  random_state=0,
                                  verbose=False):
    all_boxes = []
    peak_info = []

    for cd, members, c in zip(cluster_drifts, cluster_members, cluster_labels_unique):
        f0 = float(edges_hz[min(members)])
        f1 = float(edges_hz[max(members) + 1])

        res = fit_gmm_event_state(
            cd,
            min_event_posterior=min_event_posterior,
            smooth_labels_k=smooth_labels_k,
            merge_gap_windows=merge_gap_windows,
            min_event_windows=min_event_windows,
            min_event_area_q=min_event_area_q,
            use_log_drift=use_log_drift,
            random_state=random_state,
        )

        labels = res["labels"]
        regions = res["regions"]
        cluster_boxes = []
        peaks = []

        for st, en in regions:
            segment = cd[st:en + 1]
            peak_idx = int(st + np.argmax(segment))
            peaks.append(peak_idx)
            cluster_boxes.append({
                "cluster_id": int(c),
                "t0": float(win_starts[st]),
                "t1": float(win_ends[en]),
                "f0": f0,
                "f1": f1,
                "peak_idx": int(peak_idx),
                "support_left": int(st),
                "support_right": int(en),
                "members": members,
            })

        cluster_boxes = merge_overlapping_time_boxes(cluster_boxes)
        all_boxes.extend(cluster_boxes)

        if verbose:
            means_str = np.array2string(res["means"], precision=3, separator=", ")
            weights_str = np.array2string(res["weights"], precision=3, separator=", ")
            event_fraction = float(np.mean(labels)) if len(labels) else 0.0
            print(
                f"Cluster {c} | {int(f0):4d}-{int(f1):5d} Hz | "
                f"thr≈{res['threshold']:.3f} | means={means_str} | "
                f"weights={weights_str} | event_fraction={event_fraction:.3f} | "
                f"regions={len(regions)} | boxes={len(cluster_boxes)}"
            )

        peak_info.append({
            "cluster_id": int(c),
            "freq_lo": f0,
            "freq_hi": f1,
            "n_peaks": int(len(peaks)),
            "n_boxes": int(len(cluster_boxes)),
            "peaks": peaks,
            "labels": labels,
            "posterior": res["posterior"],
            "regions": regions,
            "means": res["means"],
            "weights": res["weights"],
            "threshold": res["threshold"],
        })

    return all_boxes, peak_info


def score_boxes_against_event_curve(boxes, event_curve, win_starts, win_ends):
    activity = boxes_to_global_activity_curve(
        boxes=boxes, n_windows=len(win_starts),
        win_starts=win_starts, win_ends=win_ends,
        smooth_k=BOX_SMOOTH_ACTIVITY_K,
    )
    corr = safe_pearson(robust_zscore(activity), robust_zscore(event_curve))

    _, _, event_peaks, _ = detect_peaks_from_drift(
        event_curve, k_mad=EVENT_PEAK_K_MAD,
        min_sep_windows=EVENT_PEAK_MIN_SEP_WINDOWS,
    )

    pred_peaks = []
    for b in boxes:
        centre_t = 0.5 * (b["t0"] + b["t1"])
        pred_peaks.append(int(np.argmin(np.abs(win_starts - centre_t))))

    peak_scores = match_peaks_with_tolerance(event_peaks, pred_peaks, tol=PEAK_MATCH_TOL_WINDOWS)
    box_density = len(boxes) / max(1, len(win_starts))
    proxy = (
        PROXY_CORR_WEIGHT * corr
        + PROXY_PEAK_F1_WEIGHT * peak_scores["f1"]
        - PROXY_BOX_DENSITY_PENALTY * box_density
    )
    return {
        "proxy_score": float(proxy),
        "event_corr": float(corr),
        "event_peak_f1": float(peak_scores["f1"]),
        "event_peak_precision": float(peak_scores["precision"]),
        "event_peak_recall": float(peak_scores["recall"]),
        "box_density": float(box_density),
    }


def run_gmm_grid_search(cluster_drifts, cluster_members, uniq_clusters, edges_hz,
                        win_starts, win_ends, event_curve, gt_boxes):
    rows = []
    best = None

    for posterior in GRID_GMM_MIN_EVENT_POSTERIOR:
        for smooth_k in GRID_GMM_SMOOTH_LABELS_K:
            for gap in GRID_GMM_MERGE_GAP_WINDOWS:
                for min_w in GRID_GMM_MIN_EVENT_WINDOWS:
                    for use_log in GRID_GMM_USE_LOG_DRIFT:
                        for area_q in GRID_GMM_MIN_EVENT_AREA_Q:
                            boxes, _ = gmm_boxes_from_cluster_drifts(
                                cluster_drifts, cluster_members, uniq_clusters,
                                edges_hz, win_starts, win_ends,
                                min_event_posterior=posterior,
                                smooth_labels_k=smooth_k,
                                merge_gap_windows=gap,
                                min_event_windows=min_w,
                                min_event_area_q=area_q,
                                use_log_drift=use_log,
                                random_state=GMM_RANDOM_STATE,
                                verbose=False,
                            )

                            proxy = score_boxes_against_event_curve(boxes, event_curve, win_starts, win_ends)
                            gt_metrics = evaluate_predicted_boxes(boxes, gt_boxes, iou_thr=GT_IOU_THRESHOLD) if len(gt_boxes) else None

                            if gt_metrics is not None:
                                score = 0.65 * gt_metrics["f1"] + 0.20 * gt_metrics["mean_iou"] + 0.15 * proxy["event_corr"]
                            else:
                                score = proxy["proxy_score"]

                            row = {
                                "score": float(score),
                                "posterior": posterior,
                                "smooth_k": smooth_k,
                                "merge_gap": gap,
                                "min_event_windows": min_w,
                                "use_log_drift": use_log,
                                "min_event_area_q": area_q,
                                "n_boxes": len(boxes),
                                **proxy,
                            }
                            if gt_metrics is not None:
                                row.update({f"gt_{k}": v for k, v in gt_metrics.items()})
                            rows.append(row)

                            if best is None or row["score"] > best["score"]:
                                best = row

    return best, pd.DataFrame(rows).sort_values("score", ascending=False)


def compute_pipeline_for_file(wav_path):
    print(f"\n=== Processing {Path(wav_path).name} ===")
    y, sr = librosa.load(wav_path, sr=TARGET_SR, mono=True)
    y = apply_waveform_preprocessing(y, sr)
    duration_s = len(y) / sr

    win_starts, win_ends = make_fixed_windows(duration_s, WINDOW_S, HOP_S)

    repr_vectors = compute_window_repr_vectors(
        y=y, sr=sr, win_starts=win_starts, win_ends=win_ends,
        n_mels=N_MELS_FULL, fmin=FMIN, fmax=FMAX, use_pcen=USE_PCEN,
    )

    if DENOISE_METHOD == "median_specsub":
        repr_vectors = subtract_stationary_band_floor(
            repr_vectors, strength=SPEC_FLOOR_STRENGTH, floor_mode=SPEC_FLOOR_MODE
        )

    edges_hz = band_edges_hz(FMIN, FMAX, N_FINE_BANDS, scale=BAND_SCALE)
    mel_freqs = librosa.mel_frequencies(n_mels=N_MELS_FULL, fmin=FMIN, fmax=FMAX)
    band_vectors = split_mel_vectors_into_bands(repr_vectors, mel_freqs, edges_hz)
    band_drifts = compute_direct_band_drifts(band_vectors, smooth_k=DRIFT_SMOOTH_K)

    band_drifts_z, band_evidences = [], []
    for d in band_drifts:
        dz, ev, _, _ = detect_peaks_from_drift(d, k_mad=PEAK_K_MAD, min_sep_windows=PEAK_MIN_SEP_WINDOWS)
        band_drifts_z.append(dz)
        band_evidences.append(ev)

    band_descs = compute_band_frequency_descriptors(
        band_vectors, k_mad_activity=FREQ_DESC_ACTIVITY_K_MAD,
        normalize_subband=FREQ_DESC_NORMALIZE_SUBBAND,
        use_activity=FREQ_DESC_USE_ACTIVITY,
    )

    S_time = build_time_similarity_matrix(
        drifts=band_drifts_z, evidences=band_evidences,
        mode=SIMILARITY_MODE, max_lag=MAX_LAG_WINDOWS,
    )
    S_freq = build_frequency_similarity_matrix(band_descs)
    S_adj = build_adjacency_similarity_matrix(N_FINE_BANDS, tau=ADJ_TAU)
    S_total = combine_similarity_matrices(
        S_time=S_time, S_freq=S_freq, S_adj=S_adj,
        alpha=ALPHA_TIME, beta=BETA_FREQ, gamma=GAMMA_ADJ,
    )

    _, cluster_labels = cluster_bands_from_similarity(
        S_total, linkage_method=LINKAGE_METHOD,
        n_groups=N_GROUPS, distance_threshold=DISTANCE_THRESHOLD,
    )

    uniq_clusters, cluster_drifts, cluster_members = aggregate_cluster_drifts(
        drifts=band_drifts_z, cluster_labels=cluster_labels, agg=CLUSTER_AGG,
    )

    global_repr_vectors = compute_global_repr_vectors(
        y=y, sr=sr, win_starts=win_starts, win_ends=win_ends,
        n_mels=N_MELS_FULL, fmin=FMIN, fmax=FMAX,
    )

    if DENOISE_METHOD == "median_specsub":
        global_repr_vectors = subtract_stationary_band_floor(
            global_repr_vectors, strength=SPEC_FLOOR_STRENGTH, floor_mode=SPEC_FLOOR_MODE
        )

    event_curve = compute_spectrogram_event_curve(global_repr_vectors, smooth_k=EVENT_SMOOTH_K)

    return {
        "y": y, "sr": sr, "duration_s": duration_s,
        "win_starts": win_starts, "win_ends": win_ends,
        "edges_hz": edges_hz, "cluster_labels": cluster_labels,
        "uniq_clusters": uniq_clusters, "cluster_drifts": cluster_drifts,
        "cluster_members": cluster_members, "event_curve": event_curve,
    }



def get_file_output_paths(wav_path, out_dir, dataset_name):
    """
    Centralises output paths for one audio file so skip checks and saving
    always use the same folder/file naming convention.
    """
    wav_path = Path(wav_path)
    stem = wav_path.stem
    file_out_dir = Path(out_dir) / dataset_name / stem

    return {
        "file_out_dir": file_out_dir,
        "predicted_boxes_csv": file_out_dir / f"{stem}_predicted_boxes.csv",
        "done_marker": file_out_dir / f"{stem}_done.txt",
        "per_file_summary_csv": file_out_dir / f"{stem}_summary.csv",
        "grid_results_csv": file_out_dir / f"{stem}_grid_results.csv",
        "spectrogram_boxes_png": file_out_dir / f"{stem}_spectrogram_boxes.png",
        "cluster_drifts_png": file_out_dir / f"{stem}_cluster_drifts.png",
    }


def file_is_completed(wav_path, out_dir, dataset_name):
    """
    Returns True when this file appears to have completed successfully before.

    By default, this checks for the predicted boxes CSV. If
    REQUIRE_DONE_MARKER_FOR_SKIP=True, it also requires the done marker.
    """
    paths = get_file_output_paths(wav_path, out_dir, dataset_name)
    predicted_exists = paths["predicted_boxes_csv"].exists()
    marker_exists = paths["done_marker"].exists()

    if REQUIRE_DONE_MARKER_FOR_SKIP:
        return predicted_exists and marker_exists

    return predicted_exists or marker_exists


def build_skipped_row(wav_path, out_dir, gt_folder=None, dataset_name=None):
    """
    Builds a lightweight summary row for a skipped file.
    If a per-file summary exists, load it so previous metrics are preserved
    in the new batch_summary.csv.
    """
    wav_path = Path(wav_path)
    if dataset_name is None:
        dataset_name = "single_dataset"

    paths = get_file_output_paths(wav_path, out_dir, dataset_name)
    summary_csv = paths["per_file_summary_csv"]

    if summary_csv.exists():
        try:
            old = pd.read_csv(summary_csv)
            if len(old):
                row = old.iloc[0].to_dict()
                row["skipped_existing"] = True
                row["skip_reason"] = "existing_per_file_summary"
                return row
        except Exception:
            pass

    gt_path = find_matching_gt(wav_path, gt_folder)
    return {
        "dataset": dataset_name,
        "file": wav_path.name,
        "wav_path": str(wav_path),
        "gt_path": gt_path,
        "skipped_existing": True,
        "skip_reason": "existing_completed_outputs",
        "output_dir": str(paths["file_out_dir"]),
    }

def process_one_file(wav_path, out_dir, gt_folder=None, dataset_name=None):
    wav_path = Path(wav_path)
    stem = wav_path.stem

    if dataset_name is None:
        dataset_name = "single_dataset"

    # Keep outputs separated by dataset/site/session.
    paths = get_file_output_paths(wav_path, out_dir, dataset_name)
    file_out_dir = paths["file_out_dir"]
    file_out_dir.mkdir(parents=True, exist_ok=True)

    predicted_boxes_csv = paths["predicted_boxes_csv"]
    summary_marker = paths["done_marker"]
    per_file_summary_csv = paths["per_file_summary_csv"]

    if SKIP_COMPLETED_FILES and file_is_completed(wav_path, out_dir, dataset_name):
        print(f"Skipping completed file: {wav_path}")
        return build_skipped_row(
            wav_path=wav_path,
            out_dir=out_dir,
            gt_folder=gt_folder,
            dataset_name=dataset_name,
        )

    gt_path = find_matching_gt(wav_path, gt_folder)
    gt_boxes = try_load_ground_truth_boxes(gt_path) if gt_path is not None else []

    pipe = compute_pipeline_for_file(wav_path)

    params_used = {
        "posterior": GMM_MIN_EVENT_POSTERIOR,
        "smooth_k": GMM_SMOOTH_LABELS_K,
        "merge_gap": GMM_MERGE_GAP_WINDOWS,
        "min_event_windows": GMM_MIN_EVENT_WINDOWS,
        "use_log_drift": GMM_USE_LOG_DRIFT,
        "min_event_area_q": GMM_MIN_EVENT_AREA_Q,
    }

    grid_csv = None
    if RUN_PER_FILE_GRID_SEARCH:
        best, grid_df = run_gmm_grid_search(
            pipe["cluster_drifts"], pipe["cluster_members"], pipe["uniq_clusters"],
            pipe["edges_hz"], pipe["win_starts"], pipe["win_ends"],
            pipe["event_curve"], gt_boxes,
        )
        grid_csv = paths["grid_results_csv"]
        grid_df.to_csv(grid_csv, index=False)
        params_used = {
            "posterior": best["posterior"],
            "smooth_k": int(best["smooth_k"]),
            "merge_gap": int(best["merge_gap"]),
            "min_event_windows": int(best["min_event_windows"]),
            "use_log_drift": bool(best["use_log_drift"]),
            "min_event_area_q": best["min_event_area_q"] if not pd.isna(best["min_event_area_q"]) else None,
        }

    print("Running final GMM segmentation with params:", params_used)
    boxes, _ = gmm_boxes_from_cluster_drifts(
        pipe["cluster_drifts"], pipe["cluster_members"], pipe["uniq_clusters"],
        pipe["edges_hz"], pipe["win_starts"], pipe["win_ends"],
        min_event_posterior=params_used["posterior"],
        smooth_labels_k=params_used["smooth_k"],
        merge_gap_windows=params_used["merge_gap"],
        min_event_windows=params_used["min_event_windows"],
        min_event_area_q=params_used["min_event_area_q"],
        use_log_drift=params_used["use_log_drift"],
        random_state=GMM_RANDOM_STATE,
        verbose=True,
    )

    if SAVE_PREDICTED_BOX_CSVS:
        save_boxes_csv(boxes, predicted_boxes_csv)

    proxy = score_boxes_against_event_curve(boxes, pipe["event_curve"], pipe["win_starts"], pipe["win_ends"])
    gt_metrics = evaluate_predicted_boxes(boxes, gt_boxes, iou_thr=GT_IOU_THRESHOLD) if len(gt_boxes) else None

    activity_curve = boxes_to_global_activity_curve(
        boxes=boxes, n_windows=len(pipe["win_starts"]),
        win_starts=pipe["win_starts"], win_ends=pipe["win_ends"],
        smooth_k=BOX_SMOOTH_ACTIVITY_K,
    )
    global_cluster_drift = np.median(np.stack(pipe["cluster_drifts"], axis=0), axis=0)
    global_corr = safe_pearson(robust_zscore(global_cluster_drift), robust_zscore(pipe["event_curve"]))
    post_corr = safe_pearson(robust_zscore(activity_curve), robust_zscore(pipe["event_curve"]))

    if SAVE_PER_FILE_PLOTS:
        save_spectrogram_with_boxes(
            pipe["y"], pipe["sr"], boxes, gt_boxes if len(gt_boxes) else None,
            paths["spectrogram_boxes_png"],
            title=f"{dataset_name} / {stem}: predicted boxes (cyan)" + (" + GT (green)" if len(gt_boxes) else ""),
            sec_limit=PLOT_SECONDS_LIMIT,
        )

    if SAVE_CLUSTER_DRIFT_PLOTS:
        save_cluster_drifts_plot(
            pipe["cluster_drifts"], pipe["cluster_members"], pipe["uniq_clusters"],
            pipe["win_starts"], pipe["edges_hz"],
            paths["cluster_drifts_png"],
            title=f"{dataset_name} / {stem}: merged cluster drifts",
            sec_limit=PLOT_SECONDS_LIMIT,
        )

    row = {
        "dataset": dataset_name,
        "file": wav_path.name,
        "wav_path": str(wav_path),
        "gt_path": gt_path,
        "duration_s": pipe["duration_s"],
        "n_windows": len(pipe["win_starts"]),
        "n_clusters": len(pipe["uniq_clusters"]),
        "n_pred_boxes": len(boxes),
        "n_gt_boxes": len(gt_boxes),
        "global_drift_event_corr": global_corr,
        "box_activity_event_corr": post_corr,
        **{f"proxy_{k}": v for k, v in proxy.items()},
        **{f"param_{k}": v for k, v in params_used.items()},
        "grid_results_csv": str(grid_csv) if grid_csv is not None else None,
        "skipped_existing": False,
        "output_dir": str(file_out_dir),
    }

    if gt_metrics is not None:
        row.update({f"gt_{k}": v for k, v in gt_metrics.items()})
    else:
        row.update({
            "gt_tp": None, "gt_fp": None, "gt_fn": None,
            "gt_precision": None, "gt_recall": None,
            "gt_f1": None, "gt_mean_iou": None,
        })

    if SAVE_PER_FILE_SUMMARY:
        pd.DataFrame([row]).to_csv(per_file_summary_csv, index=False)

    summary_marker.write_text("done\n", encoding="utf-8")
    return row


# -----------------------------
# MAIN: NESTED FOLDER SCAN
# -----------------------------

if __name__ == "__main__":
    out_dir = Path(OUTPUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)

    datasets = discover_dataset_folders(DATA_ROOT)

    # Fallback to single-file mode if no nested folders are found.
    if not datasets:
        if os.path.exists(WAV_PATH):
            fallback_gt_dir = Path(GT_PATH).parent if GT_PATH is not None else None
            datasets = [{
                "dataset_name": "single_file",
                "dataset_dir": Path(WAV_PATH).parent,
                "audio_dir": Path(WAV_PATH).parent,
                "gt_dir": fallback_gt_dir,
                "audio_files": [Path(WAV_PATH)],
                "n_audio_files": 1,
            }]
        else:
            raise FileNotFoundError(
                f"No nested dataset folders found in DATA_ROOT={DATA_ROOT}, "
                f"and WAV_PATH does not exist: {WAV_PATH}"
            )

    total_files = sum(len(d["audio_files"]) for d in datasets)

    print(f"Found {len(datasets)} dataset folders")
    print(f"Found {total_files} audio files total")
    print(f"Output directory: {out_dir.resolve()}")
    print(f"RUN_PER_FILE_GRID_SEARCH = {RUN_PER_FILE_GRID_SEARCH}")
    print(f"SKIP_COMPLETED_FILES = {SKIP_COMPLETED_FILES}")

    for dataset in datasets:
        print(
            f"  - {dataset['dataset_name']}: "
            f"{len(dataset['audio_files'])} audio files | "
            f"audio={dataset['audio_dir']} | "
            f"annotations={dataset['gt_dir']}"
        )

    summary_rows = []
    failures = []
    file_counter = 0

    for dataset in datasets:
        dataset_name = dataset["dataset_name"]
        audio_files = dataset["audio_files"]
        gt_dir = dataset["gt_dir"]

        print(f"\n=== Dataset: {dataset_name} ===")

        for wav_path in audio_files:
            file_counter += 1
            print(f"\n[{file_counter}/{total_files}] {wav_path}")

            try:
                row = process_one_file(
                    wav_path=wav_path,
                    out_dir=out_dir,
                    gt_folder=gt_dir,
                    dataset_name=dataset_name,
                )
                summary_rows.append(row)

                if row.get("skipped_existing"):
                    print(f"Skipped existing output: {row.get('output_dir')}")
                    continue

                if row.get("gt_f1") is not None and not pd.isna(row.get("gt_f1")):
                    print(
                        f"GT F1={row['gt_f1']:.3f} | P={row['gt_precision']:.3f} | "
                        f"R={row['gt_recall']:.3f} | IoU={row['gt_mean_iou']:.3f} | "
                        f"boxes={row['n_pred_boxes']}/{row['n_gt_boxes']}"
                    )
                else:
                    print(
                        f"No GT | proxy={row['proxy_proxy_score']:.3f} | "
                        f"event_corr={row['box_activity_event_corr']:.3f} | boxes={row['n_pred_boxes']}"
                    )

            except Exception as e:
                print(f"FAILED: {wav_path}: {e}")
                traceback.print_exc()
                failures.append({
                    "dataset": dataset_name,
                    "file": str(wav_path),
                    "error": repr(e),
                })

    summary_df = pd.DataFrame(summary_rows)
    summary_csv = out_dir / "batch_summary.csv"
    summary_df.to_csv(summary_csv, index=False)

    if len(summary_df) and "skipped_existing" in summary_df.columns:
        skipped_df = summary_df[summary_df["skipped_existing"] == True].copy()
        if len(skipped_df):
            skipped_csv = out_dir / "batch_skipped_existing.csv"
            skipped_df.to_csv(skipped_csv, index=False)
            print(f"\nSkipped existing files: {len(skipped_df)}. See {skipped_csv}")

    if len(summary_df):
        if "skipped_existing" in summary_df.columns:
            rankable_df = summary_df[summary_df["skipped_existing"] != True].copy()
        else:
            rankable_df = summary_df.copy()

        if len(rankable_df):
            if "gt_f1" in rankable_df.columns and rankable_df["gt_f1"].notna().any():
                ranked = rankable_df.sort_values(
                    ["gt_f1", "gt_mean_iou", "box_activity_event_corr"],
                    ascending=False,
                )
            else:
                ranked = rankable_df.sort_values(
                    ["box_activity_event_corr", "proxy_proxy_score"],
                    ascending=False,
                )

            ranked.to_csv(out_dir / "batch_summary_ranked.csv", index=False)

            print("\n=== TOP FILES ===")
            show_cols = [
                c for c in [
                    "dataset", "file", "gt_f1", "gt_recall", "gt_precision",
                    "gt_mean_iou", "box_activity_event_corr", "n_pred_boxes",
                    "n_gt_boxes", "output_dir"
                ] if c in ranked.columns
            ]
            print(ranked[show_cols].head(15).to_string(index=False))

    if failures:
        failures_csv = out_dir / "batch_failures.csv"
        pd.DataFrame(failures).to_csv(failures_csv, index=False)
        print(f"\nFailures: {len(failures)}. See {failures_csv}")

    print(f"\nDone. Summary: {summary_csv}")


<>:55: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<>:56: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<>:55: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<>:56: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
C:\Users\Dr. Tom\AppData\Local\Temp\ipykernel_27584\4154406399.py:55: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
  DATA_ROOT = "D:\WABAD"
C:\Users\Dr. Tom\AppData\Local\Temp\ipykernel_27584\4154406399.py:56: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will n

Found 9 dataset folders
Found 711 audio files total
Output directory: D:\WABAD\batch_gmm_outputs
RUN_PER_FILE_GRID_SEARCH = False
SKIP_COMPLETED_FILES = True
  - BAM (1): 118 audio files | audio=D:\WABAD\BAM (1)\Recordings | annotations=D:\WABAD\BAM (1)\Raven Pro annotations
  - BIAL: 100 audio files | audio=D:\WABAD\BIAL\Recordings | annotations=D:\WABAD\BIAL\Raven Pro annotations
  - CLH (1): 61 audio files | audio=D:\WABAD\CLH (1)\Recordings | annotations=D:\WABAD\CLH (1)\Raven Pro annotations
  - DONG: 60 audio files | audio=D:\WABAD\DONG\Recordings | annotations=D:\WABAD\DONG\Raven Pro annotations
  - FEU: 69 audio files | audio=D:\WABAD\FEU\Recordings | annotations=D:\WABAD\FEU\Raven Pro annotations
  - LIM: 92 audio files | audio=D:\WABAD\LIM\Recordings | annotations=D:\WABAD\LIM\Raven Pro annotations
  - MAPIMI: 64 audio files | audio=D:\WABAD\MAPIMI\Recordings | annotations=D:\WABAD\MAPIMI\Raven Pro annotations
  - PITI (1): 48 audio files | audio=D:\WABAD\PITI (1)\Recordings 

c:\Users\Dr. Tom\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Applying HPSS percussive emphasis...
Running final GMM segmentation with params: {'posterior': 0.5, 'smooth_k': 3, 'merge_gap': 2, 'min_event_windows': 2, 'use_log_drift': False, 'min_event_area_q': None}
Cluster 1 |  200-10000 Hz | thr≈1.661 | means=[-0.448,  3.771] | weights=[0.671, 0.329] | event_fraction=0.307 | regions=9 | boxes=9
GT F1=0.000 | P=0.000 | R=0.000 | IoU=0.000 | boxes=9/8

[620/711] D:\WABAD\ROTOK\Recordings\ROTOK_20170125_070901.wav

=== Processing ROTOK_20170125_070901.wav ===
Applying HPSS percussive emphasis...
Running final GMM segmentation with params: {'posterior': 0.5, 'smooth_k': 3, 'merge_gap': 2, 'min_event_windows': 2, 'use_log_drift': False, 'min_event_area_q': None}
Cluster 1 |  200-10000 Hz | thr≈4.646 | means=[-0.32 ,  9.612] | weights=[0.749, 0.251] | event_fraction=0.236 | regions=9 | boxes=9
GT F1=0.000 | P=0.000 | R=0.000 | IoU=0.000 | boxes=9/10

[621/711] D:\WABAD\ROTOK\Recordings\ROTOK_20170125_074801.wav

=== Processing ROTOK_20170125_074801.w

In [ ]:
# ============================================================
# DATA-DRIVEN MULTI-BAND DRIFT GROUPING + TIME/FREQUENCY HYBRID SIMILARITY
#
# This version keeps the original time-based drift peak logic, but changes
# the band merging stage so that clustering is driven by a combined similarity:
#
#   S_total = alpha * S_time + beta * S_freq + gamma * S_adjacency
#
# where:
#   - S_time comes from the existing drift-based similarity
#   - S_freq comes from stable per-band spectral descriptors
#   - S_adjacency mildly prefers merging nearby bands
#
# The rest of the downstream pipeline is kept close to the original baseline.
# ============================================================

import os
import glob
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from scipy.stats import pearsonr, spearmanr
from sklearn.mixture import GaussianMixture

# -----------------------------
# INPUTS
# -----------------------------
# Batch mode inputs
# DATA_ROOT should contain one folder per site/session/recording set.
# Each inner folder should contain one recordings/audio folder and one
# Raven Pro annotation/selection-table folder, for example:
#
# DATA_ROOT/
#   Site_1/
#     recordings/
#       file1.wav
#     annotations/
#       file1.txt
#   Site_2/
#     audio/
#       file2.wav
#     raven/
#       file2_SelectionTable.txt
#
# The folder names are flexible. See RECORDING_SUBDIR_CANDIDATES and
# ANNOTATION_SUBDIR_CANDIDATES below.
DATA_ROOT = "D:\WABAD"
OUTPUT_DIR = "D:\WABAD/batch_gmm_outputs"

# Single-file fallback. Used only if no nested datasets are found.
WAV_PATH = "ARD_20211027_072000.wav"
GT_PATH = None  # Optional Raven-style annotation txt/csv for confirmatory box metrics

# Inner folder name candidates. Add your exact folder names here if needed.
RECORDING_SUBDIR_CANDIDATES = (
    "Recordings",
    "recording",
    "audio",
    "audios",
    "wav",
    "wavs",
    "sounds",
)

ANNOTATION_SUBDIR_CANDIDATES = (
    "annotations",
    "annotation",
    "raven",
    "raven_annotations",
    "Raven Pro annotations",
    "raven_pro",
    "selection_tables",
    "selection tables",
    "gt",
    "ground_truth",
)

AUDIO_EXTENSIONS = ("*.wav", "*.WAV", "*.flac", "*.FLAC")
GT_EXTENSIONS = (".txt", ".csv", ".tsv")

# If True, audio and annotation files can be inside subfolders under the
# discovered recordings/annotations folders.
RECURSIVE_AUDIO_SEARCH = True
RECURSIVE_GT_SEARCH = True

# If True, skip files that already have completed output from a previous run.
# This makes the script restart-safe for large folders.
SKIP_COMPLETED_FILES = True

# Completion is checked using both the predicted boxes CSV and a small done marker.
# The done marker is written only after the file finishes successfully.
# If you have older outputs from before this marker existed, set this to False
# so the script can skip based on the predicted boxes CSV alone.
REQUIRE_DONE_MARKER_FOR_SKIP = False

# Write a per-file summary CSV inside each output folder. This helps preserve
# metrics for completed files even if the full batch later crashes.
SAVE_PER_FILE_SUMMARY = True

# If True, optimise GMM post-cluster parameters per file. Useful for exploration
# and picking promising examples, but it uses GT when available, so report fixed
# parameter results separately for generalisability claims.
RUN_PER_FILE_GRID_SEARCH = False
SAVE_PER_FILE_PLOTS = True
SAVE_CLUSTER_DRIFT_PLOTS = True
SAVE_PREDICTED_BOX_CSVS = True

# GT matching
GT_IOU_THRESHOLD = 0.30

# Evaluation controls.
# Strict metrics count unmatched predicted boxes as false positives.
# Coverage metrics focus on whether the GT annotations were found, which is
# useful when the GT is incomplete and the detector produces extra plausible boxes.
GT_TEMPORAL_HIT_IOU_THRESHOLD = 0.10
GT_FREQUENCY_HIT_IOU_THRESHOLD = 0.10
GT_ANY_OVERLAP_AS_HIT = True
TEMPORAL_EVAL_HOP_S = 0.125  # should normally match HOP_S below
RECOMPUTE_EVAL_FOR_SKIPPED_FILES = True
SAVE_EVALUATION_TABLES = True

# Grid-search values used only when RUN_PER_FILE_GRID_SEARCH=True
GRID_GMM_MIN_EVENT_POSTERIOR = [0.35, 0.40, 0.45, 0.50, 0.55]
GRID_GMM_SMOOTH_LABELS_K = [1, 3, 5]
GRID_GMM_MERGE_GAP_WINDOWS = [1, 2, 3, 5]
GRID_GMM_MIN_EVENT_WINDOWS = [1, 2]
GRID_GMM_USE_LOG_DRIFT = [False, True]
GRID_GMM_MIN_EVENT_AREA_Q = [None, 0.05, 0.10]

# When no GT exists, rank grid-search configurations using a proxy objective.
PROXY_CORR_WEIGHT = 0.80
PROXY_PEAK_F1_WEIGHT = 0.20
PROXY_BOX_DENSITY_PENALTY = 0.03

TARGET_SR = 22050

# Micro-windowing
WINDOW_S = 0.25
HOP_S = 0.125

# Spectral representation used for per-band drift
N_FFT = 1024
HOP_LENGTH = 256
N_MELS_FULL = 128
FMIN = 200.0
FMAX = 10000.0

# Fine band decomposition
N_FINE_BANDS = 20
BAND_SCALE = "mel"   # "mel" or "linear"

# Drift smoothing
DRIFT_SMOOTH_K = 7

# Peak / boundary evidence extraction
PEAK_K_MAD = 2.5
PEAK_MIN_SEP_WINDOWS = 2

# Time similarity
SIMILARITY_MODE = "lagged_corr"   # "jaccard_binary" or "lagged_corr"
MAX_LAG_WINDOWS = 2

# New hybrid similarity weights
ALPHA_TIME = 0.70
BETA_FREQ = 0.20
GAMMA_ADJ = 0.10
ADJ_TAU = 2.5

# Frequency descriptor controls
FREQ_DESC_USE_ACTIVITY = True
FREQ_DESC_ACTIVITY_K_MAD = 2.0
FREQ_DESC_NORMALIZE_SUBBAND = True

# Hierarchical clustering
LINKAGE_METHOD = "average"
N_GROUPS = None
DISTANCE_THRESHOLD = 0.55

# Merged cluster drift aggregation
CLUSTER_AGG = "median"   # "median", "mean", "max"

# Plotting
PLOT_SECONDS_LIMIT = None
FIG_DPI = 120

# Spectrogram-event comparison
EVENT_SMOOTH_K = 7
EVENT_PEAK_K_MAD = 2.0
EVENT_PEAK_MIN_SEP_WINDOWS = 2
PEAK_MATCH_TOL_WINDOWS = 2

# Peak -> box controls
BOX_EXPAND_WINDOWS = 0
BOX_MIN_DURATION_WINDOWS = 1
BOX_SMOOTH_ACTIVITY_K = 5

# Post-cluster state segmentation controls
# This replaces threshold/peak-only detection for final boxes.
# Each merged cluster drift is treated as a 2-state problem:
# background vs event. A 2-component GMM is fitted per merged cluster.
POST_CLUSTER_DETECTOR = "gmm_state"
GMM_RANDOM_STATE = 0
GMM_MIN_EVENT_POSTERIOR = 0.50      # 0.50 = standard GMM decision boundary; raise to 0.60-0.70 for stricter boxes
GMM_SMOOTH_LABELS_K = 3             # odd integer; set 1 to disable short flicker smoothing
GMM_MERGE_GAP_WINDOWS = 2           # merge events separated by <= this many windows
GMM_MIN_EVENT_WINDOWS = 2           # remove very short events
GMM_MIN_EVENT_AREA_Q = None         # e.g. 0.10 to remove weakest 10% of event regions, None disables
GMM_USE_LOG_DRIFT = False           # True can help when one cluster has extreme spikes

# Optional preprocessing / denoising controls
DENOISE_METHOD = "hpss_percussive"
# options:
# "none"
# "spectral_gate"
# "hpss_percussive"
# "hpss_residual"
# "median_specsub"

USE_PCEN = True

# spectral gate params
NOISE_PROFILE_SEC = 2.0
SPEC_SUB_FLOOR = 0.1

# HPSS params
HPSS_KERNEL_SIZE = 31
HPSS_POWER = 2.0
HPSS_MARGIN = (1.0, 5.0)

# median spectral subtraction params
SPEC_FLOOR_MODE = "median"
SPEC_FLOOR_STRENGTH = 1.0

# -----------------------------
# UTILS
# -----------------------------

def moving_average(x: np.ndarray, k: int) -> np.ndarray:
    if k <= 1:
        return x.copy()
    pad = k // 2
    xp = np.pad(x, (pad, pad), mode="edge")
    w = np.ones(k, dtype=float) / k
    return np.convolve(xp, w, mode="valid")


def robust_mad(x: np.ndarray) -> float:
    med = np.median(x)
    return float(np.median(np.abs(x - med)) + 1e-12)


def robust_zscore(x: np.ndarray) -> np.ndarray:
    med = np.median(x)
    mad = robust_mad(x)
    return (x - med) / mad


def minmax01(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    mn = np.min(x)
    mx = np.max(x)
    if mx - mn < 1e-12:
        return np.zeros_like(x)
    return (x - mn) / (mx - mn)


def safe_pearson(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if np.std(x) < 1e-12 or np.std(y) < 1e-12:
        return 0.0
    r = pearsonr(x, y)[0]
    if np.isnan(r):
        return 0.0
    return float(r)


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na < 1e-12 or nb < 1e-12:
        return 0.0
    s = float(np.dot(a, b) / (na * nb))
    return max(0.0, min(1.0, 0.5 * (s + 1.0)))


def make_fixed_windows(duration_s: float, window_s: float, hop_s: float):
    starts = []
    ends = []
    t = 0.0
    while t + window_s <= duration_s + 1e-12:
        starts.append(t)
        ends.append(t + window_s)
        t += hop_s
    return np.array(starts, dtype=float), np.array(ends, dtype=float)


def nonmax_suppress_peaks(peaks, scores, radius):
    if radius <= 0 or len(peaks) <= 1:
        return sorted(peaks)
    peaks_sorted = sorted(peaks, key=lambda i: scores[i], reverse=True)
    blocked = np.zeros(len(scores), dtype=bool)
    kept = []
    for p in peaks_sorted:
        if blocked[p]:
            continue
        kept.append(p)
        lo = max(0, p - radius)
        hi = min(len(scores), p + radius + 1)
        blocked[lo:hi] = True
    return sorted(kept)


def detect_peaks_from_drift(drift: np.ndarray, k_mad: float, min_sep_windows: int):
    z = robust_zscore(drift)
    thr = np.median(z) + k_mad * robust_mad(z)

    peaks = []
    for i in range(1, len(z) - 1):
        if z[i] >= thr and z[i] >= z[i - 1] and z[i] >= z[i + 1]:
            peaks.append(i)

    peaks = nonmax_suppress_peaks(peaks, z, radius=min_sep_windows)

    evidence = np.zeros(len(z), dtype=int)
    evidence[peaks] = 1
    return z, evidence, peaks, thr


def band_edges_hz(fmin: float, fmax: float, n_bands: int, scale: str = "mel") -> np.ndarray:
    if scale == "mel":
        m0 = librosa.hz_to_mel(fmin)
        m1 = librosa.hz_to_mel(fmax)
        m_edges = np.linspace(m0, m1, n_bands + 1)
        return librosa.mel_to_hz(m_edges)
    elif scale == "linear":
        return np.linspace(fmin, fmax, n_bands + 1)
    else:
        raise ValueError("scale must be 'mel' or 'linear'")


# -----------------------------
# PREPROCESSING / DENOISING
# -----------------------------

def denoise_spectral_gate(y, sr, n_fft=1024, hop_length=256,
                          noise_profile_sec=2.0, floor=0.1):
    S = librosa.stft(y, n_fft=n_fft, hop_length=hop_length)
    mag = np.abs(S)
    phase = np.angle(S)

    noise_frames = max(1, int(round((noise_profile_sec * sr) / hop_length)))
    noise_frames = min(noise_frames, mag.shape[1])

    noise_mag = np.median(mag[:, :noise_frames], axis=1, keepdims=True)
    mag_denoised = mag - noise_mag
    mag_denoised = np.maximum(mag_denoised, floor * mag)

    S_denoised = mag_denoised * np.exp(1j * phase)
    y_denoised = librosa.istft(S_denoised, hop_length=hop_length, length=len(y))
    return y_denoised


def denoise_hpss_percussive(y, kernel_size=31, power=2.0, margin=(1.0, 5.0)):
    y_h, y_p = librosa.effects.hpss(
        y,
        kernel_size=kernel_size,
        power=power,
        margin=margin
    )
    return y_p


def denoise_hpss_residual(y, kernel_size=31, power=2.0, margin=(1.0, 5.0)):
    y_h, y_p = librosa.effects.hpss(
        y,
        kernel_size=kernel_size,
        power=power,
        margin=margin
    )
    residual = y - y_h
    return residual


def subtract_stationary_band_floor(X, strength=1.0, floor_mode="median"):
    X = np.asarray(X, dtype=float)

    if floor_mode == "median":
        bg = np.median(X, axis=0, keepdims=True)
    elif floor_mode == "mean":
        bg = np.mean(X, axis=0, keepdims=True)
    else:
        raise ValueError("floor_mode must be 'median' or 'mean'")

    out = X - strength * bg
    return np.maximum(out, 0.0)


def apply_waveform_preprocessing(y, sr):
    if DENOISE_METHOD == "none":
        return y

    if DENOISE_METHOD == "spectral_gate":
        print("Applying spectral gate denoising...")
        return denoise_spectral_gate(
            y=y,
            sr=sr,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            noise_profile_sec=NOISE_PROFILE_SEC,
            floor=SPEC_SUB_FLOOR
        )

    if DENOISE_METHOD == "hpss_percussive":
        print("Applying HPSS percussive emphasis...")
        return denoise_hpss_percussive(
            y,
            kernel_size=HPSS_KERNEL_SIZE,
            power=HPSS_POWER,
            margin=HPSS_MARGIN
        )

    if DENOISE_METHOD == "hpss_residual":
        print("Applying HPSS residual emphasis...")
        return denoise_hpss_residual(
            y,
            kernel_size=HPSS_KERNEL_SIZE,
            power=HPSS_POWER,
            margin=HPSS_MARGIN
        )

    if DENOISE_METHOD == "median_specsub":
        return y

    raise ValueError(f"Unsupported DENOISE_METHOD: {DENOISE_METHOD}")


# -----------------------------
# FEATURE EXTRACTION
# -----------------------------

def compute_window_repr_vectors(y, sr, win_starts, win_ends, n_mels, fmin, fmax,
                                use_pcen=False):
    out = []

    for t0, t1 in zip(win_starts, win_ends):
        s0 = int(round(t0 * sr))
        s1 = int(round(t1 * sr))
        seg = y[s0:s1]

        if len(seg) < 16:
            out.append(np.zeros(n_mels, dtype=np.float32))
            continue

        n_fft_eff = min(N_FFT, 2 ** int(np.floor(np.log2(len(seg)))))
        n_fft_eff = max(n_fft_eff, 128)
        hop_eff = min(HOP_LENGTH, max(32, n_fft_eff // 4))

        S = librosa.feature.melspectrogram(
            y=seg,
            sr=sr,
            n_fft=n_fft_eff,
            hop_length=hop_eff,
            n_mels=n_mels,
            fmin=fmin,
            fmax=fmax,
            power=2.0,
        )

        if use_pcen:
            rep = librosa.pcen(S * (2.0 ** 16), sr=sr, hop_length=hop_eff)
        else:
            rep = librosa.power_to_db(S, ref=np.max)

        vec = rep.mean(axis=1)
        out.append(vec.astype(np.float32))

    return np.stack(out, axis=0)


def split_mel_vectors_into_bands(logmel_vectors, mel_freqs, edges_hz):
    band_vectors = []
    for bi in range(len(edges_hz) - 1):
        lo, hi = edges_hz[bi], edges_hz[bi + 1]
        if bi < len(edges_hz) - 2:
            mask = (mel_freqs >= lo) & (mel_freqs < hi)
        else:
            mask = (mel_freqs >= lo) & (mel_freqs <= hi)

        if np.sum(mask) == 0:
            band_vectors.append(np.zeros((logmel_vectors.shape[0], 1), dtype=np.float32))
        else:
            band_vectors.append(logmel_vectors[:, mask])
    return band_vectors


def compute_direct_band_drifts(band_vectors, smooth_k):
    drifts = []
    for Xb in band_vectors:
        d = np.zeros(Xb.shape[0], dtype=float)
        for t in range(1, Xb.shape[0]):
            d[t] = np.linalg.norm(Xb[t] - Xb[t - 1])
        ds = moving_average(d, smooth_k)
        drifts.append(ds)
    return drifts


def compute_band_frequency_descriptors(band_vectors, k_mad_activity=2.0,
                                       normalize_subband=True,
                                       use_activity=True):
    """
    Stable per-band descriptor.

    Each band gets a compact summary vector built from:
    - median energy per sub-bin
    - variability per sub-bin (MAD)
    - global stats: median, MAD, max, flatness proxy, activity proportion,
      burstiness, temporal sparsity, mean pairwise sub-bin correlation
    """
    descs = []

    for Xb in band_vectors:
        X = np.asarray(Xb, dtype=float)
        if X.ndim == 1:
            X = X[:, None]

        if normalize_subband:
            Xn = np.zeros_like(X)
            for j in range(X.shape[1]):
                col = X[:, j]
                Xn[:, j] = robust_zscore(col)
            X_work = Xn
        else:
            X_work = X

        med_sub = np.median(X_work, axis=0)
        mad_sub = np.median(np.abs(X_work - med_sub[None, :]), axis=0)

        collapsed = np.mean(X_work, axis=1)
        collapsed_z = robust_zscore(collapsed)
        thr = np.median(collapsed_z) + k_mad_activity * robust_mad(collapsed_z)
        active = (collapsed_z >= thr).astype(float)

        if use_activity:
            activity_prop = float(np.mean(active))
            runs = np.diff(np.pad(active.astype(int), (1, 1), mode='constant'))
            starts = np.where(runs == 1)[0]
            ends = np.where(runs == -1)[0]
            durations = ends - starts
            mean_run = float(np.mean(durations)) if len(durations) > 0 else 0.0
            n_runs = float(len(durations))
        else:
            activity_prop = 0.0
            mean_run = 0.0
            n_runs = 0.0

        global_median = float(np.median(collapsed))
        global_mad = float(robust_mad(collapsed))
        global_max = float(np.max(collapsed))
        global_mean = float(np.mean(collapsed))
        global_std = float(np.std(collapsed))
        burstiness = float(global_std / (abs(global_mean) + 1e-12))
        sparsity = float(np.mean(np.abs(collapsed_z) < 0.5))

        # Flatness-like ratio over positive-shifted data
        xp = collapsed - np.min(collapsed) + 1e-6
        gmean = float(np.exp(np.mean(np.log(xp))))
        amean = float(np.mean(xp))
        flatness = float(gmean / (amean + 1e-12))

        # Correlation between sub-bins inside the band
        if X_work.shape[1] >= 2:
            C = np.corrcoef(X_work.T)
            vals = C[np.triu_indices_from(C, k=1)]
            vals = vals[np.isfinite(vals)]
            mean_subcorr = float(np.mean(vals)) if len(vals) > 0 else 0.0
        else:
            mean_subcorr = 1.0

        desc = np.concatenate([
            med_sub,
            mad_sub,
            np.array([
                global_median,
                global_mad,
                global_max,
                burstiness,
                sparsity,
                flatness,
                activity_prop,
                mean_run,
                n_runs,
                mean_subcorr,
            ], dtype=float)
        ])
        descs.append(desc)

    max_len = max(len(d) for d in descs)
    out = []
    for d in descs:
        if len(d) < max_len:
            pad = np.zeros(max_len - len(d), dtype=float)
            d = np.concatenate([d, pad])
        out.append(d)
    return np.stack(out, axis=0)


# -----------------------------
# SIMILARITY / CLUSTERING
# -----------------------------

def jaccard_similarity_binary(a: np.ndarray, b: np.ndarray) -> float:
    inter = np.sum((a > 0) & (b > 0))
    union = np.sum((a > 0) | (b > 0))
    if union == 0:
        return 0.0
    return float(inter / union)


def lagged_corr_similarity(a: np.ndarray, b: np.ndarray, max_lag: int = 2) -> float:
    a = robust_zscore(a)
    b = robust_zscore(b)
    best = -1.0
    for lag in range(-max_lag, max_lag + 1):
        if lag < 0:
            aa = a[-lag:]
            bb = b[:len(aa)]
        elif lag > 0:
            aa = a[:-lag]
            bb = b[lag:]
        else:
            aa = a
            bb = b

        if len(aa) < 3:
            continue

        c = np.corrcoef(aa, bb)[0, 1]
        if np.isnan(c):
            c = 0.0
        best = max(best, c)

    return float(max(best, 0.0))


def build_time_similarity_matrix(drifts, evidences, mode="jaccard_binary", max_lag=2):
    n = len(drifts)
    S = np.eye(n, dtype=float)

    for i in range(n):
        for j in range(i + 1, n):
            if mode == "jaccard_binary":
                sim = jaccard_similarity_binary(evidences[i], evidences[j])
            elif mode == "lagged_corr":
                sim = lagged_corr_similarity(drifts[i], drifts[j], max_lag=max_lag)
            else:
                raise ValueError("Unsupported similarity mode")

            S[i, j] = sim
            S[j, i] = sim

    return S


def build_frequency_similarity_matrix(band_descs):
    n = band_descs.shape[0]
    S = np.eye(n, dtype=float)

    # robust standardization across bands
    X = np.asarray(band_descs, dtype=float)
    Xs = np.zeros_like(X)
    for j in range(X.shape[1]):
        col = X[:, j]
        Xs[:, j] = robust_zscore(col)

    for i in range(n):
        for j in range(i + 1, n):
            sim = cosine_sim(Xs[i], Xs[j])
            S[i, j] = sim
            S[j, i] = sim
    return S


def build_adjacency_similarity_matrix(n_bands, tau=2.5):
    S = np.eye(n_bands, dtype=float)
    for i in range(n_bands):
        for j in range(i + 1, n_bands):
            d = abs(i - j)
            sim = float(np.exp(-d / tau))
            S[i, j] = sim
            S[j, i] = sim
    return S


def combine_similarity_matrices(S_time, S_freq, S_adj,
                                alpha=0.7, beta=0.2, gamma=0.1):
    wsum = alpha + beta + gamma
    if wsum <= 0:
        raise ValueError("alpha + beta + gamma must be > 0")
    St = alpha / wsum
    Sf = beta / wsum
    Sa = gamma / wsum
    S = St * S_time + Sf * S_freq + Sa * S_adj
    S = np.clip(S, 0.0, 1.0)
    np.fill_diagonal(S, 1.0)
    return S


def cluster_bands_from_similarity(S, linkage_method="average", n_groups=None, distance_threshold=0.55):
    D = 1.0 - S
    np.fill_diagonal(D, 0.0)
    condensed = squareform(D, checks=False)
    Z = linkage(condensed, method=linkage_method)

    if n_groups is not None:
        labels = fcluster(Z, n_groups, criterion="maxclust")
    else:
        labels = fcluster(Z, distance_threshold, criterion="distance")

    return Z, labels


def aggregate_cluster_drifts(drifts, cluster_labels, agg="median"):
    uniq = sorted(np.unique(cluster_labels))
    cluster_drifts = []
    cluster_members = []

    for c in uniq:
        idx = np.where(cluster_labels == c)[0]
        M = np.stack([drifts[i] for i in idx], axis=0)

        if agg == "median":
            cd = np.median(M, axis=0)
        elif agg == "mean":
            cd = np.mean(M, axis=0)
        elif agg == "max":
            cd = np.max(M, axis=0)
        else:
            raise ValueError("agg must be median, mean, or max")

        cluster_drifts.append(cd)
        cluster_members.append(idx.tolist())

    return uniq, cluster_drifts, cluster_members


def reorder_by_dendrogram(Z):
    dend = dendrogram(Z, no_plot=True)
    return dend["leaves"]


def cluster_band_ranges(edges_hz, cluster_labels):
    uniq = sorted(np.unique(cluster_labels))
    out = {}
    for c in uniq:
        idx = np.where(cluster_labels == c)[0]
        lo = edges_hz[np.min(idx)]
        hi = edges_hz[np.max(idx) + 1]
        out[c] = (lo, hi, idx.tolist())
    return out


# -----------------------------
# EVENT CURVE FROM SPECTROGRAM
# -----------------------------

def compute_global_repr_vectors(y, sr, win_starts, win_ends, n_mels, fmin, fmax):
    return compute_window_repr_vectors(
        y=y,
        sr=sr,
        win_starts=win_starts,
        win_ends=win_ends,
        n_mels=n_mels,
        fmin=fmin,
        fmax=fmax,
        use_pcen=USE_PCEN
    )


def compute_spectrogram_event_curve(global_repr_vectors, smooth_k=7):
    d = np.zeros(global_repr_vectors.shape[0], dtype=float)
    for t in range(1, global_repr_vectors.shape[0]):
        diff = global_repr_vectors[t] - global_repr_vectors[t - 1]
        d[t] = np.linalg.norm(diff)
    d = moving_average(d, smooth_k)
    return d


def match_peaks_with_tolerance(ref_peaks, test_peaks, tol=2):
    ref_peaks = sorted(ref_peaks)
    test_peaks = sorted(test_peaks)

    matched_ref = set()
    matched_test = set()

    for i, rp in enumerate(ref_peaks):
        best_j = None
        best_dist = None
        for j, tp in enumerate(test_peaks):
            if j in matched_test:
                continue
            dist = abs(rp - tp)
            if dist <= tol and (best_dist is None or dist < best_dist):
                best_j = j
                best_dist = dist
        if best_j is not None:
            matched_ref.add(i)
            matched_test.add(best_j)

    tp = len(matched_test)
    fp = len(test_peaks) - tp
    fn = len(ref_peaks) - tp

    precision = tp / (tp + fp + 1e-12)
    recall = tp / (tp + fn + 1e-12)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def best_lagged_corr(a, b, max_lag=2):
    a = robust_zscore(a)
    b = robust_zscore(b)

    best_corr = -1.0
    best_lag = 0

    for lag in range(-max_lag, max_lag + 1):
        if lag < 0:
            aa = a[-lag:]
            bb = b[:len(aa)]
        elif lag > 0:
            aa = a[:-lag]
            bb = b[lag:]
        else:
            aa = a
            bb = b

        if len(aa) < 3:
            continue

        c = np.corrcoef(aa, bb)[0, 1]
        if np.isnan(c):
            c = 0.0

        if c > best_corr:
            best_corr = c
            best_lag = lag

    return float(best_corr), int(best_lag)


def compute_cluster_vs_event_metrics(cluster_drifts, event_curve, max_lag=2,
                                     drift_peak_k_mad=2.5, drift_peak_sep=2,
                                     event_peak_k_mad=2.0, event_peak_sep=2,
                                     peak_match_tol=2):
    metrics = []

    ev_z, ev_bin, ev_peaks, ev_thr = detect_peaks_from_drift(
        event_curve,
        k_mad=event_peak_k_mad,
        min_sep_windows=event_peak_sep
    )

    for i, cd in enumerate(cluster_drifts):
        x = robust_zscore(cd)
        y = robust_zscore(event_curve)

        if np.std(x) < 1e-12 or np.std(y) < 1e-12:
            pear = 0.0
            spear = 0.0
        else:
            pear = pearsonr(x, y)[0]
            spear = spearmanr(x, y).statistic
            if np.isnan(pear):
                pear = 0.0
            if np.isnan(spear):
                spear = 0.0

        lagcorr, bestlag = best_lagged_corr(cd, event_curve, max_lag=max_lag)

        cd_z, cd_bin, cd_peaks, cd_thr = detect_peaks_from_drift(
            cd,
            k_mad=drift_peak_k_mad,
            min_sep_windows=drift_peak_sep
        )

        peak_scores = match_peaks_with_tolerance(
            ref_peaks=ev_peaks,
            test_peaks=cd_peaks,
            tol=peak_match_tol
        )

        metrics.append({
            "cluster_index": i,
            "pearson_r": float(pear),
            "spearman_rho": float(spear),
            "max_lag_corr": float(lagcorr),
            "best_lag_windows": int(bestlag),
            "n_cluster_peaks": int(len(cd_peaks)),
            "n_event_peaks": int(len(ev_peaks)),
            "peak_precision": float(peak_scores["precision"]),
            "peak_recall": float(peak_scores["recall"]),
            "peak_f1": float(peak_scores["f1"]),
            "cluster_peaks": cd_peaks,
            "event_peaks": ev_peaks,
            "cluster_drift_z": cd_z,
            "event_curve_z": ev_z,
        })

    return metrics


# -----------------------------
# PEAK -> BOX LOGIC
# -----------------------------

def find_peak_support(z, peak_idx, thr, expand_windows=0):
    n = len(z)

    left = peak_idx
    while left > 0 and z[left] >= thr:
        left -= 1
    if z[left] < thr and left < peak_idx:
        left += 1

    right = peak_idx
    while right < n - 1 and z[right] >= thr:
        right += 1
    if z[right] < thr and right > peak_idx:
        right -= 1

    left = max(0, left - expand_windows)
    right = min(n - 1, right + expand_windows)

    if right < left:
        left = max(0, peak_idx - 1)
        right = min(n - 1, peak_idx + 1)

    return left, right


def merge_overlapping_time_boxes(boxes):
    if len(boxes) == 0:
        return []

    boxes = sorted(boxes, key=lambda b: (b["cluster_id"], b["t0"], b["t1"]))
    merged = [boxes[0].copy()]

    for b in boxes[1:]:
        prev = merged[-1]
        same_cluster = (prev["cluster_id"] == b["cluster_id"])
        overlapping = b["t0"] <= prev["t1"]

        if same_cluster and overlapping:
            prev["t1"] = max(prev["t1"], b["t1"])
        else:
            merged.append(b.copy())

    return merged



def smooth_binary_labels(labels, k=3):
    """Simple majority filter for binary labels."""
    labels = np.asarray(labels, dtype=int)
    if k <= 1:
        return labels.copy()
    if k % 2 == 0:
        k += 1
    pad = k // 2
    padded = np.pad(labels, (pad, pad), mode="edge")
    out = np.zeros_like(labels)
    for i in range(len(labels)):
        window = padded[i:i + k]
        out[i] = 1 if np.mean(window) >= 0.5 else 0
    return out


def binary_labels_to_regions(labels, merge_gap_windows=2, min_event_windows=2):
    """Convert 0/1 labels into contiguous event regions."""
    labels = np.asarray(labels, dtype=int)
    regions = []
    in_event = False
    start = None

    for i, lab in enumerate(labels):
        if lab == 1 and not in_event:
            in_event = True
            start = i
        elif lab == 0 and in_event:
            regions.append([start, i - 1])
            in_event = False

    if in_event:
        regions.append([start, len(labels) - 1])

    # Merge nearby regions
    if merge_gap_windows is not None and merge_gap_windows >= 0 and len(regions) > 1:
        merged = [regions[0]]
        for st, en in regions[1:]:
            prev = merged[-1]
            gap = st - prev[1] - 1
            if gap <= merge_gap_windows:
                prev[1] = en
            else:
                merged.append([st, en])
        regions = merged

    # Drop very short regions
    regions = [[st, en] for st, en in regions if (en - st + 1) >= min_event_windows]
    return regions


def fit_gmm_event_state(drift,
                        min_event_posterior=0.50,
                        smooth_labels_k=3,
                        merge_gap_windows=2,
                        min_event_windows=2,
                        min_event_area_q=None,
                        use_log_drift=False,
                        random_state=0):
    """
    Per-cluster 2-state segmentation using a Gaussian mixture model.

    The lower-mean component is interpreted as background and the higher-mean
    component as event. This estimates the non-event/event state from each
    post-cluster drift curve rather than using a manually chosen drift threshold.
    """
    raw = np.asarray(drift, dtype=float)

    # Optional compression for highly spiky clusters.
    if use_log_drift:
        shift = np.min(raw)
        x = np.log1p(raw - shift)
    else:
        x = raw.copy()

    if len(x) < 5 or np.std(x) < 1e-12:
        labels = np.zeros(len(x), dtype=int)
        return {
            "labels": labels,
            "regions": [],
            "threshold": float(np.inf),
            "means": np.array([float(np.mean(x)), float(np.mean(x))]),
            "weights": np.array([1.0, 0.0]),
            "event_component": 1,
            "posterior": np.zeros(len(x), dtype=float),
            "raw_event_areas": [],
        }

    X = x.reshape(-1, 1)
    gmm = GaussianMixture(
        n_components=2,
        covariance_type="full",
        random_state=random_state,
        reg_covar=1e-6,
        n_init=5,
    )
    gmm.fit(X)

    means = gmm.means_.flatten()
    weights = gmm.weights_.flatten()
    event_component = int(np.argmax(means))
    bg_component = 1 - event_component

    posterior = gmm.predict_proba(X)[:, event_component]
    labels = (posterior >= min_event_posterior).astype(int)
    labels = smooth_binary_labels(labels, k=smooth_labels_k)

    regions = binary_labels_to_regions(
        labels,
        merge_gap_windows=merge_gap_windows,
        min_event_windows=min_event_windows,
    )

    # Optional region-level area filtering. This is intentionally weak by default.
    raw_event_areas = []
    for st, en in regions:
        local_base = min(raw[st], raw[en]) if en > st else raw[st]
        area = float(np.sum(np.maximum(raw[st:en + 1] - local_base, 0.0)))
        raw_event_areas.append(area)

    if min_event_area_q is not None and len(regions) > 0:
        area_gate = float(np.quantile(raw_event_areas, min_event_area_q))
        kept_regions = []
        kept_areas = []
        for region, area in zip(regions, raw_event_areas):
            if area >= area_gate:
                kept_regions.append(region)
                kept_areas.append(area)
        regions = kept_regions
        raw_event_areas = kept_areas

        labels2 = np.zeros_like(labels)
        for st, en in regions:
            labels2[st:en + 1] = 1
        labels = labels2

    # Approximate threshold in the same space used by the GMM.
    # This is diagnostic only; boxes are built from posterior labels/regions.
    threshold = float(np.mean(means))
    if use_log_drift:
        threshold = float(np.expm1(threshold) + np.min(raw))

    return {
        "labels": labels,
        "regions": regions,
        "threshold": threshold,
        "means": means,
        "weights": weights,
        "event_component": event_component,
        "bg_component": bg_component,
        "posterior": posterior,
        "raw_event_areas": raw_event_areas,
    }


def cluster_peaks_to_boxes(cluster_drifts, cluster_members, cluster_labels_unique,
                           edges_hz, win_starts, win_ends,
                           peak_k_mad, peak_min_sep_windows,
                           expand_windows=0, min_duration_windows=1):
    all_boxes = []
    peak_info = []

    print("\n=== POST-CLUSTER GMM STATE SEGMENTATION ===")

    for cd, members, c in zip(cluster_drifts, cluster_members, cluster_labels_unique):
        f0 = float(edges_hz[min(members)])
        f1 = float(edges_hz[max(members) + 1])

        res = fit_gmm_event_state(
            cd,
            min_event_posterior=GMM_MIN_EVENT_POSTERIOR,
            smooth_labels_k=GMM_SMOOTH_LABELS_K,
            merge_gap_windows=GMM_MERGE_GAP_WINDOWS,
            min_event_windows=max(GMM_MIN_EVENT_WINDOWS, min_duration_windows),
            min_event_area_q=GMM_MIN_EVENT_AREA_Q,
            use_log_drift=GMM_USE_LOG_DRIFT,
            random_state=GMM_RANDOM_STATE,
        )

        labels = res["labels"]
        regions = res["regions"]

        cluster_boxes = []
        peaks = []
        for st, en in regions:
            segment = cd[st:en + 1]
            peak_idx = int(st + np.argmax(segment))
            peaks.append(peak_idx)

            box = {
                "cluster_id": int(c),
                "t0": float(win_starts[st]),
                "t1": float(win_ends[en]),
                "f0": f0,
                "f1": f1,
                "peak_idx": int(peak_idx),
                "support_left": int(st),
                "support_right": int(en),
                "members": members
            }
            cluster_boxes.append(box)

        cluster_boxes = merge_overlapping_time_boxes(cluster_boxes)
        all_boxes.extend(cluster_boxes)

        means_str = np.array2string(res["means"], precision=3, separator=", ")
        weights_str = np.array2string(res["weights"], precision=3, separator=", ")
        event_fraction = float(np.mean(labels)) if len(labels) else 0.0
        print(
            f"Cluster {c} | {int(f0):4d}-{int(f1):5d} Hz | "
            f"mode=gmm_state | thr≈{res['threshold']:.3f} | "
            f"means={means_str} | weights={weights_str} | "
            f"event_fraction={event_fraction:.3f} | regions={len(regions)} | boxes={len(cluster_boxes)}"
        )

        peak_info.append({
            "cluster_id": int(c),
            "freq_lo": f0,
            "freq_hi": f1,
            "n_peaks": int(len(peaks)),
            "n_boxes": int(len(cluster_boxes)),
            "peaks": peaks,
            "z": labels.astype(float),
            "thr": res["threshold"],
            "labels": labels,
            "posterior": res["posterior"],
            "regions": regions,
            "means": res["means"],
            "weights": res["weights"],
        })

    return all_boxes, peak_info

def boxes_to_global_activity_curve(boxes, n_windows, win_starts, win_ends, smooth_k=1):
    activity = np.zeros(n_windows, dtype=float)

    for b in boxes:
        overlap = (win_starts < b["t1"]) & (win_ends > b["t0"])
        activity[overlap] = 1.0

    if smooth_k > 1:
        activity = moving_average(activity, smooth_k)

    return activity


# -----------------------------
# OPTIONAL GT LOADING + SIMPLE BOX METRICS
# -----------------------------

def try_load_ground_truth_boxes(path):
    if path is None:
        return []
    if not os.path.exists(path):
        print(f"GT file not found: {path}")
        return []

    sep = '\t' if path.lower().endswith('.txt') else None
    df = pd.read_csv(path, sep=sep, engine='python')
    cols = {c.lower().strip(): c for c in df.columns}

    def pick(names):
        for n in names:
            if n in cols:
                return cols[n]
        return None

    c_t0 = pick(["begin time (s)", "begin time", "start time", "t0"])
    c_t1 = pick(["end time (s)", "end time", "stop time", "t1"])
    c_f0 = pick(["low freq (hz)", "low freq", "f0", "low frequency"])
    c_f1 = pick(["high freq (hz)", "high freq", "f1", "high frequency"])

    if any(c is None for c in [c_t0, c_t1, c_f0, c_f1]):
        print("Could not identify GT columns. Expected time/frequency box columns.")
        return []

    boxes = []
    for _, row in df.iterrows():
        boxes.append({
            "t0": float(row[c_t0]),
            "t1": float(row[c_t1]),
            "f0": float(row[c_f0]),
            "f1": float(row[c_f1]),
        })
    return boxes


def interval_iou(a0, a1, b0, b1):
    inter = max(0.0, min(a1, b1) - max(a0, b0))
    union = max(a1, b1) - min(a0, b0)
    if union <= 0:
        return 0.0
    return inter / union


def box_iou(a, b):
    it = max(0.0, min(a["t1"], b["t1"]) - max(a["t0"], b["t0"]))
    ifr = max(0.0, min(a["f1"], b["f1"]) - max(a["f0"], b["f0"]))
    inter = it * ifr
    area_a = max(0.0, a["t1"] - a["t0"]) * max(0.0, a["f1"] - a["f0"])
    area_b = max(0.0, b["t1"] - b["t0"]) * max(0.0, b["f1"] - b["f0"])
    union = area_a + area_b - inter
    if union <= 0:
        return 0.0
    return inter / union


def _safe_div(n, d):
    return float(n / d) if d not in (0, 0.0, None) and not pd.isna(d) else 0.0


def _box_duration(b):
    return max(0.0, float(b.get("t1", 0.0)) - float(b.get("t0", 0.0)))


def _box_bandwidth(b):
    return max(0.0, float(b.get("f1", 0.0)) - float(b.get("f0", 0.0)))


def _box_area(b):
    return _box_duration(b) * _box_bandwidth(b)


def _intersection_length(a0, a1, b0, b1):
    return max(0.0, min(float(a1), float(b1)) - max(float(a0), float(b0)))


def _union_length_intervals(intervals):
    """Union length of [start, end] intervals."""
    clean = []
    for lo, hi in intervals:
        lo = float(lo)
        hi = float(hi)
        if hi > lo:
            clean.append((lo, hi))
    if not clean:
        return 0.0
    clean.sort()
    total = 0.0
    cur_lo, cur_hi = clean[0]
    for lo, hi in clean[1:]:
        if lo <= cur_hi:
            cur_hi = max(cur_hi, hi)
        else:
            total += cur_hi - cur_lo
            cur_lo, cur_hi = lo, hi
    total += cur_hi - cur_lo
    return float(total)


def _pair_alignment_metrics(pred_box, gt_box):
    """Returns time, frequency, and 2D box alignment for one prediction/GT pair."""
    t_inter = _intersection_length(pred_box["t0"], pred_box["t1"], gt_box["t0"], gt_box["t1"])
    f_inter = _intersection_length(pred_box["f0"], pred_box["f1"], gt_box["f0"], gt_box["f1"])

    pred_t = _box_duration(pred_box)
    gt_t = _box_duration(gt_box)
    pred_f = _box_bandwidth(pred_box)
    gt_f = _box_bandwidth(gt_box)

    t_union = pred_t + gt_t - t_inter
    f_union = pred_f + gt_f - f_inter

    inter_area = t_inter * f_inter
    pred_area = _box_area(pred_box)
    gt_area = _box_area(gt_box)
    union_area = pred_area + gt_area - inter_area

    pred_centre_t = 0.5 * (float(pred_box["t0"]) + float(pred_box["t1"]))
    gt_centre_t = 0.5 * (float(gt_box["t0"]) + float(gt_box["t1"]))
    pred_centre_f = 0.5 * (float(pred_box["f0"]) + float(pred_box["f1"]))
    gt_centre_f = 0.5 * (float(gt_box["f0"]) + float(gt_box["f1"]))

    return {
        "temporal_iou": _safe_div(t_inter, t_union),
        "frequency_iou": _safe_div(f_inter, f_union),
        "iou": _safe_div(inter_area, union_area),
        "temporal_intersection_s": float(t_inter),
        "frequency_intersection_hz": float(f_inter),
        "temporal_recall_coverage": _safe_div(t_inter, gt_t),
        "frequency_recall_coverage": _safe_div(f_inter, gt_f),
        "area_recall_coverage": _safe_div(inter_area, gt_area),
        "temporal_centre_error_s": abs(pred_centre_t - gt_centre_t),
        "frequency_centre_error_hz": abs(pred_centre_f - gt_centre_f),
    }


def _make_temporal_activity_curve(boxes, win_starts=None, win_ends=None, duration_s=None, hop_s=None):
    """
    Converts boxes to a 0/1 temporal activity curve.
    This is used for time-only accuracy/precision/recall. It ignores frequency.
    """
    if win_starts is None or win_ends is None:
        if duration_s is None or duration_s <= 0:
            max_t = 0.0
            for b in boxes:
                max_t = max(max_t, float(b.get("t1", 0.0)))
            duration_s = max_t
        hop_s = hop_s or TEMPORAL_EVAL_HOP_S
        if duration_s <= 0 or hop_s <= 0:
            return np.zeros(0, dtype=int)
        win_starts = np.arange(0.0, duration_s + 1e-9, hop_s)
        win_ends = win_starts + hop_s

    activity = np.zeros(len(win_starts), dtype=int)
    for b in boxes:
        overlap = (win_starts < float(b["t1"])) & (win_ends > float(b["t0"]))
        activity[overlap] = 1
    return activity


def _binary_curve_metrics(pred_curve, gt_curve):
    n = int(min(len(pred_curve), len(gt_curve)))
    if n <= 0:
        return {
            "temporal_n_windows": 0,
            "temporal_tp_windows": 0,
            "temporal_fp_windows": 0,
            "temporal_fn_windows": 0,
            "temporal_tn_windows": 0,
            "temporal_accuracy": None,
            "temporal_precision": None,
            "temporal_recall": None,
            "temporal_hit_rate": None,
            "temporal_f1": None,
            "temporal_iou": None,
            "temporal_balanced_accuracy": None,
        }

    pred = np.asarray(pred_curve[:n], dtype=bool)
    gt = np.asarray(gt_curve[:n], dtype=bool)

    tp = int(np.sum(pred & gt))
    fp = int(np.sum(pred & ~gt))
    fn = int(np.sum(~pred & gt))
    tn = int(np.sum(~pred & ~gt))

    precision = _safe_div(tp, tp + fp)
    recall = _safe_div(tp, tp + fn)
    specificity = _safe_div(tn, tn + fp)
    f1 = _safe_div(2 * precision * recall, precision + recall)
    jaccard = _safe_div(tp, tp + fp + fn)
    acc = _safe_div(tp + tn, n)
    bal_acc = 0.5 * (recall + specificity)

    return {
        "temporal_n_windows": n,
        "temporal_tp_windows": tp,
        "temporal_fp_windows": fp,
        "temporal_fn_windows": fn,
        "temporal_tn_windows": tn,
        "temporal_accuracy": acc,
        "temporal_precision": precision,
        "temporal_recall": recall,
        "temporal_hit_rate": recall,
        "temporal_f1": f1,
        "temporal_iou": jaccard,
        "temporal_balanced_accuracy": bal_acc,
    }


def evaluate_predicted_boxes(pred_boxes, gt_boxes, iou_thr=0.3,
                             win_starts=None, win_ends=None,
                             duration_s=None, temporal_eval_hop_s=None,
                             temporal_hit_iou_thr=None,
                             frequency_hit_iou_thr=None):
    """
    Evaluates predicted time-frequency boxes against Raven GT boxes.

    Important interpretation:
    - strict precision/recall/F1 use one-to-one IoU matching and count unmatched
      predictions as false positives.
    - GT coverage/hit metrics ask whether annotated GT boxes were found and are
      less punitive when the detector produces additional boxes not present in GT.
    - temporal_* metrics are time-only binary activity metrics over analysis windows.
      They are useful for temporal alignment but can be biased by incomplete GT.
    """
    n_pred = len(pred_boxes)
    n_gt = len(gt_boxes)

    if n_gt == 0:
        return None

    temporal_hit_iou_thr = GT_TEMPORAL_HIT_IOU_THRESHOLD if temporal_hit_iou_thr is None else temporal_hit_iou_thr
    frequency_hit_iou_thr = GT_FREQUENCY_HIT_IOU_THRESHOLD if frequency_hit_iou_thr is None else frequency_hit_iou_thr
    temporal_eval_hop_s = TEMPORAL_EVAL_HOP_S if temporal_eval_hop_s is None else temporal_eval_hop_s

    # Pairwise metrics for every prediction/GT pair.
    pair_rows = []
    for pi, pb in enumerate(pred_boxes):
        for gi, gb in enumerate(gt_boxes):
            m = _pair_alignment_metrics(pb, gb)
            pair_rows.append({"pred_idx": pi, "gt_idx": gi, **m})

    # Strict one-to-one matching by descending 2D IoU.
    matched_pred = set()
    matched_gt = set()
    matched_rows = []
    for r in sorted(pair_rows, key=lambda x: x["iou"], reverse=True):
        if r["iou"] < iou_thr:
            break
        if r["pred_idx"] in matched_pred or r["gt_idx"] in matched_gt:
            continue
        matched_pred.add(r["pred_idx"])
        matched_gt.add(r["gt_idx"])
        matched_rows.append(r)

    tp = len(matched_rows)
    fp = n_pred - tp
    fn = n_gt - tp

    precision = _safe_div(tp, tp + fp)
    recall = _safe_div(tp, tp + fn)
    f1 = _safe_div(2 * precision * recall, precision + recall)

    def mean_from(rows, key):
        vals = [float(r[key]) for r in rows if r.get(key) is not None and not pd.isna(r.get(key))]
        return float(np.mean(vals)) if vals else 0.0

    def median_from(rows, key):
        vals = [float(r[key]) for r in rows if r.get(key) is not None and not pd.isna(r.get(key))]
        return float(np.median(vals)) if vals else 0.0

    # Best prediction per GT. This is deliberately not one-to-one, because many
    # predicted boxes may map to the same GT or split one GT into smaller regions.
    best_by_gt = []
    for gi, gb in enumerate(gt_boxes):
        rows = [r for r in pair_rows if r["gt_idx"] == gi]
        if rows:
            best_iou_row = max(rows, key=lambda x: x["iou"])
            best_temporal_row = max(rows, key=lambda x: x["temporal_iou"])
            best_frequency_row = max(rows, key=lambda x: x["frequency_iou"])
        else:
            best_iou_row = {
                "iou": 0.0, "temporal_iou": 0.0, "frequency_iou": 0.0,
                "area_recall_coverage": 0.0, "temporal_centre_error_s": None,
                "frequency_centre_error_hz": None,
            }
            best_temporal_row = {"temporal_iou": 0.0}
            best_frequency_row = {"frequency_iou": 0.0}

        # Union-based GT coverage: this is better when one GT is split into
        # several predicted boxes.
        t_intervals = []
        f_intervals = []
        for pb in pred_boxes:
            t0 = max(float(pb["t0"]), float(gb["t0"]))
            t1 = min(float(pb["t1"]), float(gb["t1"]))
            f0 = max(float(pb["f0"]), float(gb["f0"]))
            f1 = min(float(pb["f1"]), float(gb["f1"]))
            if t1 > t0:
                t_intervals.append((t0, t1))
            if t1 > t0 and f1 > f0:
                f_intervals.append((f0, f1))

        temporal_coverage = _safe_div(_union_length_intervals(t_intervals), _box_duration(gb))
        frequency_coverage = _safe_div(_union_length_intervals(f_intervals), _box_bandwidth(gb))

        any_overlap_hit = any(
            (r["temporal_intersection_s"] > 0 and r["frequency_intersection_hz"] > 0)
            for r in rows
        )

        best_by_gt.append({
            "gt_idx": gi,
            "best_iou": float(best_iou_row.get("iou", 0.0)),
            "best_temporal_iou": float(best_temporal_row.get("temporal_iou", 0.0)),
            "best_frequency_iou": float(best_frequency_row.get("frequency_iou", 0.0)),
            "best_area_recall_coverage": float(best_iou_row.get("area_recall_coverage", 0.0)),
            "temporal_coverage": float(temporal_coverage),
            "frequency_coverage": float(frequency_coverage),
            "temporal_centre_error_s": best_iou_row.get("temporal_centre_error_s"),
            "frequency_centre_error_hz": best_iou_row.get("frequency_centre_error_hz"),
            "hit_iou": float(best_iou_row.get("iou", 0.0)) >= iou_thr,
            "hit_temporal": float(best_temporal_row.get("temporal_iou", 0.0)) >= temporal_hit_iou_thr,
            "hit_frequency": float(best_frequency_row.get("frequency_iou", 0.0)) >= frequency_hit_iou_thr,
            "hit_any_overlap": bool(any_overlap_hit),
        })

    pred_curve = _make_temporal_activity_curve(
        pred_boxes, win_starts=win_starts, win_ends=win_ends,
        duration_s=duration_s, hop_s=temporal_eval_hop_s,
    )
    gt_curve = _make_temporal_activity_curve(
        gt_boxes, win_starts=win_starts, win_ends=win_ends,
        duration_s=duration_s, hop_s=temporal_eval_hop_s,
    )
    temporal_metrics = _binary_curve_metrics(pred_curve, gt_curve)

    # Conservative precision penalises all extra predicted boxes. This is often
    # useful, but can be harsh when GT is incomplete.
    strict_precision = precision

    out = {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "hit_rate": recall,
        "f1": f1,
        "mean_iou": mean_from(matched_rows, "iou"),
        "median_iou": median_from(matched_rows, "iou"),
        "mean_temporal_iou": mean_from(matched_rows, "temporal_iou"),
        "mean_frequency_iou": mean_from(matched_rows, "frequency_iou"),
        "mean_temporal_centre_error_s": mean_from(matched_rows, "temporal_centre_error_s"),
        "mean_frequency_centre_error_hz": mean_from(matched_rows, "frequency_centre_error_hz"),
        "strict_precision": strict_precision,
        "pred_to_gt_box_ratio": _safe_div(n_pred, n_gt),
        "n_unmatched_predictions": fp,
        "n_unmatched_gt": fn,
        "best_iou_mean": mean_from(best_by_gt, "best_iou"),
        "best_temporal_iou_mean": mean_from(best_by_gt, "best_temporal_iou"),
        "best_frequency_iou_mean": mean_from(best_by_gt, "best_frequency_iou"),
        "temporal_coverage_mean": mean_from(best_by_gt, "temporal_coverage"),
        "frequency_coverage_mean": mean_from(best_by_gt, "frequency_coverage"),
        "area_recall_coverage_mean": mean_from(best_by_gt, "best_area_recall_coverage"),
        "iou_hit_rate": _safe_div(sum(bool(x["hit_iou"]) for x in best_by_gt), n_gt),
        "box_temporal_hit_rate": _safe_div(sum(bool(x["hit_temporal"]) for x in best_by_gt), n_gt),
        "box_frequency_hit_rate": _safe_div(sum(bool(x["hit_frequency"]) for x in best_by_gt), n_gt),
        "any_overlap_hit_rate": _safe_div(sum(bool(x["hit_any_overlap"]) for x in best_by_gt), n_gt),
        **temporal_metrics,
    }

    return out


def load_boxes_csv_as_boxes(path):
    """Loads a predicted-box CSV written by save_boxes_csv()."""
    path = Path(path)
    if not path.exists():
        return []
    df = pd.read_csv(path)
    required = {"t0", "t1", "f0", "f1"}
    if not required.issubset(df.columns):
        return []
    boxes = []
    for _, row in df.iterrows():
        try:
            boxes.append({
                "t0": float(row["t0"]),
                "t1": float(row["t1"]),
                "f0": float(row["f0"]),
                "f1": float(row["f1"]),
            })
        except Exception:
            continue
    return boxes


def add_or_update_eval_metrics_for_existing_row(row, wav_path, out_dir, gt_folder, dataset_name):
    """
    Recomputes evaluation metrics for skipped files using existing predicted boxes.
    This avoids rerunning the detector just to build the evaluation tables.
    """
    if not RECOMPUTE_EVAL_FOR_SKIPPED_FILES:
        return row

    paths = get_file_output_paths(wav_path, out_dir, dataset_name)
    pred_boxes = load_boxes_csv_as_boxes(paths["predicted_boxes_csv"])
    gt_path = find_matching_gt(wav_path, gt_folder)
    gt_boxes = try_load_ground_truth_boxes(gt_path) if gt_path is not None else []

    if not gt_boxes:
        return row

    duration_s = row.get("duration_s")
    try:
        duration_s = float(duration_s)
    except Exception:
        duration_s = None
    if duration_s is None or pd.isna(duration_s) or duration_s <= 0:
        duration_s = max(
            [0.0]
            + [float(b.get("t1", 0.0)) for b in pred_boxes]
            + [float(b.get("t1", 0.0)) for b in gt_boxes]
        )

    gt_metrics = evaluate_predicted_boxes(
        pred_boxes,
        gt_boxes,
        iou_thr=GT_IOU_THRESHOLD,
        duration_s=duration_s,
        temporal_eval_hop_s=TEMPORAL_EVAL_HOP_S,
    )

    row.update({
        "gt_path": gt_path,
        "duration_s": duration_s,
        "n_pred_boxes": len(pred_boxes),
        "n_gt_boxes": len(gt_boxes),
    })
    if gt_metrics is not None:
        row.update({f"gt_{k}": v for k, v in gt_metrics.items()})
    return row


# -----------------------------
# PLOTTING
# -----------------------------

def maybe_limit_time(y, sr, sec_limit):
    if sec_limit is None:
        return y
    n = int(round(sec_limit * sr))
    return y[:n]


def get_time_mask(win_starts, sec_limit):
    if sec_limit is None:
        return np.ones_like(win_starts, dtype=bool)
    return win_starts <= sec_limit


def plot_spectrogram(y, sr, sec_limit=None):
    y2 = maybe_limit_time(y, sr, sec_limit)

    S = librosa.stft(y2, n_fft=2048, hop_length=512)
    S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)

    fig, ax = plt.subplots(figsize=(14, 5), dpi=FIG_DPI)
    img = librosa.display.specshow(
        S_db,
        sr=sr,
        hop_length=512,
        x_axis="time",
        y_axis="hz",
        fmin=0,
        fmax=FMAX,
        cmap="magma",
        ax=ax
    )
    ax.set_title("Spectrogram")
    ax.set_ylim(0, FMAX)
    plt.colorbar(img, ax=ax, format="%+2.0f dB")
    plt.tight_layout()
    plt.show()


def plot_similarity_matrix(S, title="Similarity Matrix"):
    fig, ax = plt.subplots(figsize=(7, 6), dpi=FIG_DPI)
    im = ax.imshow(S, origin="lower", aspect="auto", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel("Band index")
    ax.set_ylabel("Band index")
    plt.colorbar(im, ax=ax, label="Similarity")
    plt.tight_layout()
    plt.show()


def plot_similarity_matrix_reordered(S, order, title="Reordered Similarity Matrix"):
    Sr = S[np.ix_(order, order)]
    fig, ax = plt.subplots(figsize=(7, 6), dpi=FIG_DPI)
    im = ax.imshow(Sr, origin="lower", aspect="auto", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel("Reordered band index")
    ax.set_ylabel("Reordered band index")
    plt.colorbar(im, ax=ax, label="Similarity")
    plt.tight_layout()
    plt.show()


def plot_dendrogram(Z):
    fig, ax = plt.subplots(figsize=(10, 4), dpi=FIG_DPI)
    dendrogram(Z, ax=ax)
    ax.set_title("Hierarchical Clustering of Bands")
    ax.set_xlabel("Band index")
    ax.set_ylabel("Distance")
    plt.tight_layout()
    plt.show()


def plot_cluster_bandmap(edges_hz, cluster_labels):
    uniq = sorted(np.unique(cluster_labels))
    fig, ax = plt.subplots(figsize=(10, 2.2), dpi=FIG_DPI)

    for bi in range(len(edges_hz) - 1):
        lo = edges_hz[bi]
        hi = edges_hz[bi + 1]
        c = cluster_labels[bi]
        ax.fill_between([lo, hi], 0, 1, alpha=0.7,
                        label=f"C{c}" if bi == np.where(cluster_labels == c)[0][0] else None)

    ax.set_xlim(edges_hz[0], edges_hz[-1])
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_xlabel("Frequency (Hz)")
    ax.set_title("Discovered Band Grouping")
    ax.legend(ncol=min(len(uniq), 8), loc="upper center", bbox_to_anchor=(0.5, -0.18))
    plt.tight_layout()
    plt.show()


def plot_cluster_drifts(cluster_drifts, cluster_members, cluster_labels_unique, win_starts, edges_hz, sec_limit=None):
    t = win_starts.copy()
    mask = get_time_mask(t, sec_limit)

    n = len(cluster_drifts)
    fig, axes = plt.subplots(n, 1, figsize=(14, max(2.5 * n, 6)), sharex=True, dpi=FIG_DPI)

    if n == 1:
        axes = [axes]

    for ax, cd, members, c in zip(axes, cluster_drifts, cluster_members, cluster_labels_unique):
        tt = t[mask]
        yy = cd[mask]

        lo = edges_hz[min(members)]
        hi = edges_hz[max(members) + 1]

        ax.plot(tt, yy, linewidth=1.4)
        ax.set_ylabel("Drift")
        ax.set_title(
            f"Cluster {c}: {int(lo)} - {int(hi)} Hz | members={members}"
        )
        ax.grid(alpha=0.25)

    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(f"Merged Cluster Drifts ({CLUSTER_AGG})", y=1.0)
    plt.tight_layout()
    plt.show()


def plot_all_cluster_drifts_overlay(cluster_drifts, cluster_members, cluster_labels_unique,
                                    win_starts, edges_hz, sec_limit=None,
                                    normalize=True, event_curve=None,
                                    activity_curve=None):
    t = win_starts.copy()
    mask = get_time_mask(t, sec_limit)
    tt = t[mask]

    fig, ax = plt.subplots(figsize=(14, 5), dpi=FIG_DPI)

    for cd, members, c in zip(cluster_drifts, cluster_members, cluster_labels_unique):
        lo = edges_hz[min(members)]
        hi = edges_hz[max(members) + 1]
        yy = cd[mask]
        if normalize:
            yy = minmax01(yy)
        ax.plot(tt, yy, linewidth=1.6, label=f"C{c}: {int(lo)}-{int(hi)} Hz")

    if event_curve is not None:
        ev = event_curve[mask]
        if normalize:
            ev = minmax01(ev)
        ax.plot(tt, ev, linewidth=2.5, linestyle="--", label="Spectrogram event curve")

    if activity_curve is not None:
        ac = activity_curve[mask]
        if normalize:
            ac = minmax01(ac)
        ax.plot(tt, ac, linewidth=2.0, linestyle=":", label="Box-derived activity")

    ax.set_title("Merged Cluster Drifts + Event Curve + Box Activity")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Normalized magnitude" if normalize else "Magnitude")
    ax.grid(alpha=0.25)
    ax.legend(loc="upper right", fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()


def plot_spectrogram_with_cluster_drifts(y, sr, win_starts, cluster_drifts, cluster_members,
                                         cluster_labels_unique, edges_hz,
                                         event_curve=None, activity_curve=None,
                                         sec_limit=None):
    y2 = maybe_limit_time(y, sr, sec_limit)
    S = librosa.stft(y2, n_fft=2048, hop_length=512)
    S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)

    t = win_starts.copy()
    mask = get_time_mask(t, sec_limit)
    tt = t[mask]

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                             gridspec_kw={"height_ratios": [3, 2]}, dpi=FIG_DPI)

    img = librosa.display.specshow(
        S_db,
        sr=sr,
        hop_length=512,
        x_axis="time",
        y_axis="hz",
        fmin=0,
        fmax=FMAX,
        cmap="magma",
        ax=axes[0]
    )
    axes[0].set_ylim(0, FMAX)
    axes[0].set_title("Spectrogram")
    plt.colorbar(img, ax=axes[0], format="%+2.0f dB")

    for cd, members, c in zip(cluster_drifts, cluster_members, cluster_labels_unique):
        lo = edges_hz[min(members)]
        hi = edges_hz[max(members) + 1]
        yy = minmax01(cd[mask])
        axes[1].plot(tt, yy, linewidth=1.5, label=f"C{c}: {int(lo)}-{int(hi)} Hz")

    if event_curve is not None:
        ev = minmax01(event_curve[mask])
        axes[1].plot(tt, ev, linewidth=2.5, linestyle="--", label="Spectrogram event curve")

    if activity_curve is not None:
        ac = minmax01(activity_curve[mask])
        axes[1].plot(tt, ac, linewidth=2.0, linestyle=":", label="Box-derived activity")

    axes[1].set_title("Merged Cluster Drifts vs Event Curve vs Box Activity")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Normalized magnitude")
    axes[1].grid(alpha=0.25)
    axes[1].legend(loc="upper right", fontsize=8, ncol=2)

    plt.tight_layout()
    plt.show()


def plot_spectrogram_with_discovered_boxes(y, sr, boxes, gt_boxes=None, sec_limit=None):
    y2 = maybe_limit_time(y, sr, sec_limit)
    S = librosa.stft(y2, n_fft=2048, hop_length=512)
    S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)

    fig, ax = plt.subplots(figsize=(14, 6), dpi=FIG_DPI)
    img = librosa.display.specshow(
        S_db,
        sr=sr,
        hop_length=512,
        x_axis="time",
        y_axis="hz",
        fmin=0,
        fmax=FMAX,
        cmap="magma",
        ax=ax
    )
    ax.set_ylim(0, FMAX)
    ax.set_title("Spectrogram with Discovered Peak-Derived Boxes")

    for b in boxes:
        if sec_limit is not None and b["t0"] > sec_limit:
            continue

        rect = plt.Rectangle(
            (b["t0"], b["f0"]),
            b["t1"] - b["t0"],
            b["f1"] - b["f0"],
            fill=False,
            edgecolor="cyan",
            linewidth=1.5
        )
        ax.add_patch(rect)

    if gt_boxes is not None:
        for b in gt_boxes:
            if sec_limit is not None and b["t0"] > sec_limit:
                continue
            rect = plt.Rectangle(
                (b["t0"], b["f0"]),
                b["t1"] - b["t0"],
                b["f1"] - b["f0"],
                fill=False,
                edgecolor="lime",
                linewidth=1.2,
                linestyle="--"
            )
            ax.add_patch(rect)

    plt.colorbar(img, ax=ax, format="%+2.0f dB")
    plt.tight_layout()
    plt.show()


def print_cluster_event_metrics(metrics, cluster_members, cluster_labels_unique, edges_hz):
    print("\n=== CLUSTER vs SPECTROGRAM EVENT METRICS ===")
    for m, members, c in zip(metrics, cluster_members, cluster_labels_unique):
        lo = edges_hz[min(members)]
        hi = edges_hz[max(members) + 1]

        print(
            f"Cluster {c} | {int(lo):4d}-{int(hi):5d} Hz | members={members}\n"
            f"  Pearson r       : {m['pearson_r']:.3f}\n"
            f"  Spearman rho    : {m['spearman_rho']:.3f}\n"
            f"  Max lag corr    : {m['max_lag_corr']:.3f}  (lag={m['best_lag_windows']} windows)\n"
            f"  Peak precision  : {m['peak_precision']:.3f}\n"
            f"  Peak recall     : {m['peak_recall']:.3f}\n"
            f"  Peak F1         : {m['peak_f1']:.3f}\n"
            f"  Cluster peaks   : {m['n_cluster_peaks']}\n"
            f"  Event peaks     : {m['n_event_peaks']}"
        )


def print_similarity_diagnostics(S_time, S_freq, S_adj, S_total):
    print("\n=== SIMILARITY MATRIX DIAGNOSTICS ===")
    for name, S in [("Time", S_time), ("Freq", S_freq), ("Adj", S_adj), ("Total", S_total)]:
        off = S[~np.eye(S.shape[0], dtype=bool)]
        print(f"{name:>5s} | min={off.min():.3f} mean={off.mean():.3f} max={off.max():.3f}")



# -----------------------------
# BATCH HELPERS
# -----------------------------

def normalise_folder_name(name):
    """Normalise folder names for flexible matching."""
    return name.lower().replace("-", "_").replace(" ", "_").strip()


def find_candidate_subdir(parent_dir, candidate_names):
    """
    Finds the first matching subfolder inside parent_dir using flexible,
    case-insensitive matching.
    """
    parent_dir = Path(parent_dir)
    if not parent_dir.exists():
        return None

    candidate_norms = {normalise_folder_name(c) for c in candidate_names}

    for child in parent_dir.iterdir():
        if child.is_dir() and normalise_folder_name(child.name) in candidate_norms:
            return child

    return None


def find_audio_files(audio_folder):
    """Find audio files inside a discovered recordings/audio folder."""
    audio_folder = Path(audio_folder)
    if not audio_folder.exists():
        return []

    paths = []
    for ext in AUDIO_EXTENSIONS:
        if RECURSIVE_AUDIO_SEARCH:
            paths.extend(audio_folder.rglob(ext))
        else:
            paths.extend(audio_folder.glob(ext))

    return sorted(set(paths))


def discover_dataset_folders(data_root):
    """
    Discover folders containing recordings and Raven Pro annotation folders.

    Expected structure:
        DATA_ROOT/
            dataset_or_site_1/
                recordings/
                annotations/
            dataset_or_site_2/
                audio/
                raven/

    The function also supports DATA_ROOT itself being a single dataset folder.
    """
    data_root = Path(data_root)
    if not data_root.exists():
        raise FileNotFoundError(f"DATA_ROOT does not exist: {data_root}")

    candidate_dataset_dirs = [data_root] + [p for p in data_root.iterdir() if p.is_dir()]
    datasets = []
    seen_audio_dirs = set()

    for dataset_dir in candidate_dataset_dirs:
        audio_dir = find_candidate_subdir(dataset_dir, RECORDING_SUBDIR_CANDIDATES)
        gt_dir = find_candidate_subdir(dataset_dir, ANNOTATION_SUBDIR_CANDIDATES)

        if audio_dir is None:
            continue

        audio_files = find_audio_files(audio_dir)
        if not audio_files:
            continue

        resolved_audio_dir = audio_dir.resolve()
        if resolved_audio_dir in seen_audio_dirs:
            continue
        seen_audio_dirs.add(resolved_audio_dir)

        datasets.append({
            "dataset_name": dataset_dir.name,
            "dataset_dir": dataset_dir,
            "audio_dir": audio_dir,
            "gt_dir": gt_dir,
            "audio_files": audio_files,
            "n_audio_files": len(audio_files),
        })

    return datasets


def find_matching_gt(wav_path, gt_folder):
    """
    Finds a Raven-style annotation file for a given audio file.

    Matching strategy:
    1. Exact same stem: recording.wav -> recording.txt/csv/tsv
    2. Common Raven suffixes: recording_SelectionTable.txt, etc.
    3. Loose contains match: if the audio stem appears inside the annotation name.
    """
    if gt_folder is None:
        return None

    gt_dir = Path(gt_folder)
    if not gt_dir.exists():
        return None

    stem = Path(wav_path).stem

    candidates = []
    for ext in GT_EXTENSIONS:
        candidates.extend([
            gt_dir / f"{stem}{ext}",
            gt_dir / f"{stem}_annotations{ext}",
            gt_dir / f"{stem}_annotation{ext}",
            gt_dir / f"{stem}_SelectionTable{ext}",
            gt_dir / f"{stem}_Selection Table{ext}",
            gt_dir / f"{stem}.SelectionTable{ext}",
        ])

    for c in candidates:
        if c.exists():
            return str(c)

    search_iter = gt_dir.rglob("*") if RECURSIVE_GT_SEARCH else gt_dir.glob("*")
    for p in search_iter:
        if p.is_file() and p.suffix.lower() in GT_EXTENSIONS:
            if stem.lower() in p.stem.lower():
                return str(p)

    return None

def save_boxes_csv(boxes, out_csv):
    rows = []
    for b in boxes:
        rows.append({
            "cluster_id": b.get("cluster_id"),
            "t0": b.get("t0"),
            "t1": b.get("t1"),
            "duration_s": b.get("t1", 0) - b.get("t0", 0),
            "f0": b.get("f0"),
            "f1": b.get("f1"),
            "bandwidth_hz": b.get("f1", 0) - b.get("f0", 0),
            "peak_idx": b.get("peak_idx"),
            "support_left": b.get("support_left"),
            "support_right": b.get("support_right"),
            "members": ",".join(map(str, b.get("members", []))),
        })
    pd.DataFrame(rows).to_csv(out_csv, index=False)


def save_spectrogram_with_boxes(y, sr, boxes, gt_boxes, out_png, title, sec_limit=None):
    y2 = maybe_limit_time(y, sr, sec_limit)
    S = librosa.stft(y2, n_fft=2048, hop_length=512)
    S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)

    fig, ax = plt.subplots(figsize=(14, 6), dpi=FIG_DPI)
    img = librosa.display.specshow(
        S_db, sr=sr, hop_length=512, x_axis="time", y_axis="hz",
        fmin=0, fmax=FMAX, cmap="magma", ax=ax
    )
    ax.set_ylim(0, FMAX)
    ax.set_title(title)

    for b in boxes:
        if sec_limit is not None and b["t0"] > sec_limit:
            continue
        ax.add_patch(plt.Rectangle(
            (b["t0"], b["f0"]), b["t1"] - b["t0"], b["f1"] - b["f0"],
            fill=False, edgecolor="cyan", linewidth=1.3
        ))

    if gt_boxes is not None:
        for b in gt_boxes:
            if sec_limit is not None and b["t0"] > sec_limit:
                continue
            ax.add_patch(plt.Rectangle(
                (b["t0"], b["f0"]), b["t1"] - b["t0"], b["f1"] - b["f0"],
                fill=False, edgecolor="lime", linewidth=1.1, linestyle="--"
            ))

    plt.colorbar(img, ax=ax, format="%+2.0f dB")
    plt.tight_layout()
    fig.savefig(out_png, bbox_inches="tight")
    plt.close(fig)


def save_cluster_drifts_plot(cluster_drifts, cluster_members, cluster_labels_unique,
                             win_starts, edges_hz, out_png, title, sec_limit=None):
    t = win_starts.copy()
    mask = get_time_mask(t, sec_limit)
    n = len(cluster_drifts)
    fig, axes = plt.subplots(n, 1, figsize=(14, max(2.3 * n, 6)), sharex=True, dpi=FIG_DPI)
    if n == 1:
        axes = [axes]

    for ax, cd, members, c in zip(axes, cluster_drifts, cluster_members, cluster_labels_unique):
        lo = edges_hz[min(members)]
        hi = edges_hz[max(members) + 1]
        ax.plot(t[mask], cd[mask], linewidth=1.2)
        ax.set_ylabel("Drift")
        ax.set_title(f"Cluster {c}: {int(lo)}-{int(hi)} Hz | members={members}")
        ax.grid(alpha=0.25)

    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(title, y=1.0)
    plt.tight_layout()
    fig.savefig(out_png, bbox_inches="tight")
    plt.close(fig)


def gmm_boxes_from_cluster_drifts(cluster_drifts, cluster_members, cluster_labels_unique,
                                  edges_hz, win_starts, win_ends,
                                  min_event_posterior=0.50,
                                  smooth_labels_k=3,
                                  merge_gap_windows=2,
                                  min_event_windows=2,
                                  min_event_area_q=None,
                                  use_log_drift=False,
                                  random_state=0,
                                  verbose=False):
    all_boxes = []
    peak_info = []

    for cd, members, c in zip(cluster_drifts, cluster_members, cluster_labels_unique):
        f0 = float(edges_hz[min(members)])
        f1 = float(edges_hz[max(members) + 1])

        res = fit_gmm_event_state(
            cd,
            min_event_posterior=min_event_posterior,
            smooth_labels_k=smooth_labels_k,
            merge_gap_windows=merge_gap_windows,
            min_event_windows=min_event_windows,
            min_event_area_q=min_event_area_q,
            use_log_drift=use_log_drift,
            random_state=random_state,
        )

        labels = res["labels"]
        regions = res["regions"]
        cluster_boxes = []
        peaks = []

        for st, en in regions:
            segment = cd[st:en + 1]
            peak_idx = int(st + np.argmax(segment))
            peaks.append(peak_idx)
            cluster_boxes.append({
                "cluster_id": int(c),
                "t0": float(win_starts[st]),
                "t1": float(win_ends[en]),
                "f0": f0,
                "f1": f1,
                "peak_idx": int(peak_idx),
                "support_left": int(st),
                "support_right": int(en),
                "members": members,
            })

        cluster_boxes = merge_overlapping_time_boxes(cluster_boxes)
        all_boxes.extend(cluster_boxes)

        if verbose:
            means_str = np.array2string(res["means"], precision=3, separator=", ")
            weights_str = np.array2string(res["weights"], precision=3, separator=", ")
            event_fraction = float(np.mean(labels)) if len(labels) else 0.0
            print(
                f"Cluster {c} | {int(f0):4d}-{int(f1):5d} Hz | "
                f"thr≈{res['threshold']:.3f} | means={means_str} | "
                f"weights={weights_str} | event_fraction={event_fraction:.3f} | "
                f"regions={len(regions)} | boxes={len(cluster_boxes)}"
            )

        peak_info.append({
            "cluster_id": int(c),
            "freq_lo": f0,
            "freq_hi": f1,
            "n_peaks": int(len(peaks)),
            "n_boxes": int(len(cluster_boxes)),
            "peaks": peaks,
            "labels": labels,
            "posterior": res["posterior"],
            "regions": regions,
            "means": res["means"],
            "weights": res["weights"],
            "threshold": res["threshold"],
        })

    return all_boxes, peak_info


def score_boxes_against_event_curve(boxes, event_curve, win_starts, win_ends):
    activity = boxes_to_global_activity_curve(
        boxes=boxes, n_windows=len(win_starts),
        win_starts=win_starts, win_ends=win_ends,
        smooth_k=BOX_SMOOTH_ACTIVITY_K,
    )
    corr = safe_pearson(robust_zscore(activity), robust_zscore(event_curve))

    _, _, event_peaks, _ = detect_peaks_from_drift(
        event_curve, k_mad=EVENT_PEAK_K_MAD,
        min_sep_windows=EVENT_PEAK_MIN_SEP_WINDOWS,
    )

    pred_peaks = []
    for b in boxes:
        centre_t = 0.5 * (b["t0"] + b["t1"])
        pred_peaks.append(int(np.argmin(np.abs(win_starts - centre_t))))

    peak_scores = match_peaks_with_tolerance(event_peaks, pred_peaks, tol=PEAK_MATCH_TOL_WINDOWS)
    box_density = len(boxes) / max(1, len(win_starts))
    proxy = (
        PROXY_CORR_WEIGHT * corr
        + PROXY_PEAK_F1_WEIGHT * peak_scores["f1"]
        - PROXY_BOX_DENSITY_PENALTY * box_density
    )
    return {
        "proxy_score": float(proxy),
        "event_corr": float(corr),
        "event_peak_f1": float(peak_scores["f1"]),
        "event_peak_precision": float(peak_scores["precision"]),
        "event_peak_recall": float(peak_scores["recall"]),
        "box_density": float(box_density),
    }


def run_gmm_grid_search(cluster_drifts, cluster_members, uniq_clusters, edges_hz,
                        win_starts, win_ends, event_curve, gt_boxes):
    rows = []
    best = None

    for posterior in GRID_GMM_MIN_EVENT_POSTERIOR:
        for smooth_k in GRID_GMM_SMOOTH_LABELS_K:
            for gap in GRID_GMM_MERGE_GAP_WINDOWS:
                for min_w in GRID_GMM_MIN_EVENT_WINDOWS:
                    for use_log in GRID_GMM_USE_LOG_DRIFT:
                        for area_q in GRID_GMM_MIN_EVENT_AREA_Q:
                            boxes, _ = gmm_boxes_from_cluster_drifts(
                                cluster_drifts, cluster_members, uniq_clusters,
                                edges_hz, win_starts, win_ends,
                                min_event_posterior=posterior,
                                smooth_labels_k=smooth_k,
                                merge_gap_windows=gap,
                                min_event_windows=min_w,
                                min_event_area_q=area_q,
                                use_log_drift=use_log,
                                random_state=GMM_RANDOM_STATE,
                                verbose=False,
                            )

                            proxy = score_boxes_against_event_curve(boxes, event_curve, win_starts, win_ends)
                            gt_metrics = evaluate_predicted_boxes(boxes, gt_boxes, iou_thr=GT_IOU_THRESHOLD) if len(gt_boxes) else None

                            if gt_metrics is not None:
                                score = 0.65 * gt_metrics["f1"] + 0.20 * gt_metrics["mean_iou"] + 0.15 * proxy["event_corr"]
                            else:
                                score = proxy["proxy_score"]

                            row = {
                                "score": float(score),
                                "posterior": posterior,
                                "smooth_k": smooth_k,
                                "merge_gap": gap,
                                "min_event_windows": min_w,
                                "use_log_drift": use_log,
                                "min_event_area_q": area_q,
                                "n_boxes": len(boxes),
                                **proxy,
                            }
                            if gt_metrics is not None:
                                row.update({f"gt_{k}": v for k, v in gt_metrics.items()})
                            rows.append(row)

                            if best is None or row["score"] > best["score"]:
                                best = row

    return best, pd.DataFrame(rows).sort_values("score", ascending=False)


def compute_pipeline_for_file(wav_path):
    print(f"\n=== Processing {Path(wav_path).name} ===")
    y, sr = librosa.load(wav_path, sr=TARGET_SR, mono=True)
    y = apply_waveform_preprocessing(y, sr)
    duration_s = len(y) / sr

    win_starts, win_ends = make_fixed_windows(duration_s, WINDOW_S, HOP_S)

    repr_vectors = compute_window_repr_vectors(
        y=y, sr=sr, win_starts=win_starts, win_ends=win_ends,
        n_mels=N_MELS_FULL, fmin=FMIN, fmax=FMAX, use_pcen=USE_PCEN,
    )

    if DENOISE_METHOD == "median_specsub":
        repr_vectors = subtract_stationary_band_floor(
            repr_vectors, strength=SPEC_FLOOR_STRENGTH, floor_mode=SPEC_FLOOR_MODE
        )

    edges_hz = band_edges_hz(FMIN, FMAX, N_FINE_BANDS, scale=BAND_SCALE)
    mel_freqs = librosa.mel_frequencies(n_mels=N_MELS_FULL, fmin=FMIN, fmax=FMAX)
    band_vectors = split_mel_vectors_into_bands(repr_vectors, mel_freqs, edges_hz)
    band_drifts = compute_direct_band_drifts(band_vectors, smooth_k=DRIFT_SMOOTH_K)

    band_drifts_z, band_evidences = [], []
    for d in band_drifts:
        dz, ev, _, _ = detect_peaks_from_drift(d, k_mad=PEAK_K_MAD, min_sep_windows=PEAK_MIN_SEP_WINDOWS)
        band_drifts_z.append(dz)
        band_evidences.append(ev)

    band_descs = compute_band_frequency_descriptors(
        band_vectors, k_mad_activity=FREQ_DESC_ACTIVITY_K_MAD,
        normalize_subband=FREQ_DESC_NORMALIZE_SUBBAND,
        use_activity=FREQ_DESC_USE_ACTIVITY,
    )

    S_time = build_time_similarity_matrix(
        drifts=band_drifts_z, evidences=band_evidences,
        mode=SIMILARITY_MODE, max_lag=MAX_LAG_WINDOWS,
    )
    S_freq = build_frequency_similarity_matrix(band_descs)
    S_adj = build_adjacency_similarity_matrix(N_FINE_BANDS, tau=ADJ_TAU)
    S_total = combine_similarity_matrices(
        S_time=S_time, S_freq=S_freq, S_adj=S_adj,
        alpha=ALPHA_TIME, beta=BETA_FREQ, gamma=GAMMA_ADJ,
    )

    _, cluster_labels = cluster_bands_from_similarity(
        S_total, linkage_method=LINKAGE_METHOD,
        n_groups=N_GROUPS, distance_threshold=DISTANCE_THRESHOLD,
    )

    uniq_clusters, cluster_drifts, cluster_members = aggregate_cluster_drifts(
        drifts=band_drifts_z, cluster_labels=cluster_labels, agg=CLUSTER_AGG,
    )

    global_repr_vectors = compute_global_repr_vectors(
        y=y, sr=sr, win_starts=win_starts, win_ends=win_ends,
        n_mels=N_MELS_FULL, fmin=FMIN, fmax=FMAX,
    )

    if DENOISE_METHOD == "median_specsub":
        global_repr_vectors = subtract_stationary_band_floor(
            global_repr_vectors, strength=SPEC_FLOOR_STRENGTH, floor_mode=SPEC_FLOOR_MODE
        )

    event_curve = compute_spectrogram_event_curve(global_repr_vectors, smooth_k=EVENT_SMOOTH_K)

    return {
        "y": y, "sr": sr, "duration_s": duration_s,
        "win_starts": win_starts, "win_ends": win_ends,
        "edges_hz": edges_hz, "cluster_labels": cluster_labels,
        "uniq_clusters": uniq_clusters, "cluster_drifts": cluster_drifts,
        "cluster_members": cluster_members, "event_curve": event_curve,
    }



def get_file_output_paths(wav_path, out_dir, dataset_name):
    """
    Centralises output paths for one audio file so skip checks and saving
    always use the same folder/file naming convention.
    """
    wav_path = Path(wav_path)
    stem = wav_path.stem
    file_out_dir = Path(out_dir) / dataset_name / stem

    return {
        "file_out_dir": file_out_dir,
        "predicted_boxes_csv": file_out_dir / f"{stem}_predicted_boxes.csv",
        "done_marker": file_out_dir / f"{stem}_done.txt",
        "per_file_summary_csv": file_out_dir / f"{stem}_summary.csv",
        "grid_results_csv": file_out_dir / f"{stem}_grid_results.csv",
        "spectrogram_boxes_png": file_out_dir / f"{stem}_spectrogram_boxes.png",
        "cluster_drifts_png": file_out_dir / f"{stem}_cluster_drifts.png",
    }


def file_is_completed(wav_path, out_dir, dataset_name):
    """
    Returns True when this file appears to have completed successfully before.

    By default, this checks for the predicted boxes CSV. If
    REQUIRE_DONE_MARKER_FOR_SKIP=True, it also requires the done marker.
    """
    paths = get_file_output_paths(wav_path, out_dir, dataset_name)
    predicted_exists = paths["predicted_boxes_csv"].exists()
    marker_exists = paths["done_marker"].exists()

    if REQUIRE_DONE_MARKER_FOR_SKIP:
        return predicted_exists and marker_exists

    return predicted_exists or marker_exists


def build_skipped_row(wav_path, out_dir, gt_folder=None, dataset_name=None):
    """
    Builds a lightweight summary row for a skipped file.
    If a per-file summary exists, load it so previous metrics are preserved
    in the new batch_summary.csv.
    """
    wav_path = Path(wav_path)
    if dataset_name is None:
        dataset_name = "single_dataset"

    paths = get_file_output_paths(wav_path, out_dir, dataset_name)
    summary_csv = paths["per_file_summary_csv"]

    if summary_csv.exists():
        try:
            old = pd.read_csv(summary_csv)
            if len(old):
                row = old.iloc[0].to_dict()
                row["skipped_existing"] = True
                row["skip_reason"] = "existing_per_file_summary"
                return add_or_update_eval_metrics_for_existing_row(
                    row=row,
                    wav_path=wav_path,
                    out_dir=out_dir,
                    gt_folder=gt_folder,
                    dataset_name=dataset_name,
                )
        except Exception:
            pass

    gt_path = find_matching_gt(wav_path, gt_folder)
    row = {
        "dataset": dataset_name,
        "file": wav_path.name,
        "wav_path": str(wav_path),
        "gt_path": gt_path,
        "skipped_existing": True,
        "skip_reason": "existing_completed_outputs",
        "output_dir": str(paths["file_out_dir"]),
    }
    return add_or_update_eval_metrics_for_existing_row(
        row=row,
        wav_path=wav_path,
        out_dir=out_dir,
        gt_folder=gt_folder,
        dataset_name=dataset_name,
    )

def process_one_file(wav_path, out_dir, gt_folder=None, dataset_name=None):
    wav_path = Path(wav_path)
    stem = wav_path.stem

    if dataset_name is None:
        dataset_name = "single_dataset"

    # Keep outputs separated by dataset/site/session.
    paths = get_file_output_paths(wav_path, out_dir, dataset_name)
    file_out_dir = paths["file_out_dir"]
    file_out_dir.mkdir(parents=True, exist_ok=True)

    predicted_boxes_csv = paths["predicted_boxes_csv"]
    summary_marker = paths["done_marker"]
    per_file_summary_csv = paths["per_file_summary_csv"]

    if SKIP_COMPLETED_FILES and file_is_completed(wav_path, out_dir, dataset_name):
        print(f"Skipping completed file: {wav_path}")
        return build_skipped_row(
            wav_path=wav_path,
            out_dir=out_dir,
            gt_folder=gt_folder,
            dataset_name=dataset_name,
        )

    gt_path = find_matching_gt(wav_path, gt_folder)
    gt_boxes = try_load_ground_truth_boxes(gt_path) if gt_path is not None else []

    pipe = compute_pipeline_for_file(wav_path)

    params_used = {
        "posterior": GMM_MIN_EVENT_POSTERIOR,
        "smooth_k": GMM_SMOOTH_LABELS_K,
        "merge_gap": GMM_MERGE_GAP_WINDOWS,
        "min_event_windows": GMM_MIN_EVENT_WINDOWS,
        "use_log_drift": GMM_USE_LOG_DRIFT,
        "min_event_area_q": GMM_MIN_EVENT_AREA_Q,
    }

    grid_csv = None
    if RUN_PER_FILE_GRID_SEARCH:
        best, grid_df = run_gmm_grid_search(
            pipe["cluster_drifts"], pipe["cluster_members"], pipe["uniq_clusters"],
            pipe["edges_hz"], pipe["win_starts"], pipe["win_ends"],
            pipe["event_curve"], gt_boxes,
        )
        grid_csv = paths["grid_results_csv"]
        grid_df.to_csv(grid_csv, index=False)
        params_used = {
            "posterior": best["posterior"],
            "smooth_k": int(best["smooth_k"]),
            "merge_gap": int(best["merge_gap"]),
            "min_event_windows": int(best["min_event_windows"]),
            "use_log_drift": bool(best["use_log_drift"]),
            "min_event_area_q": best["min_event_area_q"] if not pd.isna(best["min_event_area_q"]) else None,
        }

    print("Running final GMM segmentation with params:", params_used)
    boxes, _ = gmm_boxes_from_cluster_drifts(
        pipe["cluster_drifts"], pipe["cluster_members"], pipe["uniq_clusters"],
        pipe["edges_hz"], pipe["win_starts"], pipe["win_ends"],
        min_event_posterior=params_used["posterior"],
        smooth_labels_k=params_used["smooth_k"],
        merge_gap_windows=params_used["merge_gap"],
        min_event_windows=params_used["min_event_windows"],
        min_event_area_q=params_used["min_event_area_q"],
        use_log_drift=params_used["use_log_drift"],
        random_state=GMM_RANDOM_STATE,
        verbose=True,
    )

    if SAVE_PREDICTED_BOX_CSVS:
        save_boxes_csv(boxes, predicted_boxes_csv)

    proxy = score_boxes_against_event_curve(boxes, pipe["event_curve"], pipe["win_starts"], pipe["win_ends"])
    gt_metrics = evaluate_predicted_boxes(
        boxes,
        gt_boxes,
        iou_thr=GT_IOU_THRESHOLD,
        win_starts=pipe["win_starts"],
        win_ends=pipe["win_ends"],
        duration_s=pipe["duration_s"],
        temporal_eval_hop_s=TEMPORAL_EVAL_HOP_S,
    ) if len(gt_boxes) else None

    activity_curve = boxes_to_global_activity_curve(
        boxes=boxes, n_windows=len(pipe["win_starts"]),
        win_starts=pipe["win_starts"], win_ends=pipe["win_ends"],
        smooth_k=BOX_SMOOTH_ACTIVITY_K,
    )
    global_cluster_drift = np.median(np.stack(pipe["cluster_drifts"], axis=0), axis=0)
    global_corr = safe_pearson(robust_zscore(global_cluster_drift), robust_zscore(pipe["event_curve"]))
    post_corr = safe_pearson(robust_zscore(activity_curve), robust_zscore(pipe["event_curve"]))

    if SAVE_PER_FILE_PLOTS:
        save_spectrogram_with_boxes(
            pipe["y"], pipe["sr"], boxes, gt_boxes if len(gt_boxes) else None,
            paths["spectrogram_boxes_png"],
            title=f"{dataset_name} / {stem}: predicted boxes (cyan)" + (" + GT (green)" if len(gt_boxes) else ""),
            sec_limit=PLOT_SECONDS_LIMIT,
        )

    if SAVE_CLUSTER_DRIFT_PLOTS:
        save_cluster_drifts_plot(
            pipe["cluster_drifts"], pipe["cluster_members"], pipe["uniq_clusters"],
            pipe["win_starts"], pipe["edges_hz"],
            paths["cluster_drifts_png"],
            title=f"{dataset_name} / {stem}: merged cluster drifts",
            sec_limit=PLOT_SECONDS_LIMIT,
        )

    row = {
        "dataset": dataset_name,
        "file": wav_path.name,
        "wav_path": str(wav_path),
        "gt_path": gt_path,
        "duration_s": pipe["duration_s"],
        "n_windows": len(pipe["win_starts"]),
        "n_clusters": len(pipe["uniq_clusters"]),
        "n_pred_boxes": len(boxes),
        "n_gt_boxes": len(gt_boxes),
        "global_drift_event_corr": global_corr,
        "box_activity_event_corr": post_corr,
        **{f"proxy_{k}": v for k, v in proxy.items()},
        **{f"param_{k}": v for k, v in params_used.items()},
        "grid_results_csv": str(grid_csv) if grid_csv is not None else None,
        "skipped_existing": False,
        "output_dir": str(file_out_dir),
    }

    if gt_metrics is not None:
        row.update({f"gt_{k}": v for k, v in gt_metrics.items()})
    else:
        row.update({
            "gt_tp": None, "gt_fp": None, "gt_fn": None,
            "gt_precision": None, "gt_recall": None,
            "gt_f1": None, "gt_mean_iou": None,
        })

    if SAVE_PER_FILE_SUMMARY:
        pd.DataFrame([row]).to_csv(per_file_summary_csv, index=False)

    summary_marker.write_text("done\n", encoding="utf-8")
    return row



# -----------------------------
# EVALUATION TABLES
# -----------------------------

def _sum_numeric(df, col):
    if col not in df.columns:
        return 0.0
    return float(pd.to_numeric(df[col], errors="coerce").fillna(0).sum())


def _mean_numeric(df, col):
    if col not in df.columns:
        return None
    vals = pd.to_numeric(df[col], errors="coerce").dropna()
    return float(vals.mean()) if len(vals) else None


def _weighted_mean_numeric(df, value_col, weight_col):
    if value_col not in df.columns or weight_col not in df.columns:
        return None
    vals = pd.to_numeric(df[value_col], errors="coerce")
    weights = pd.to_numeric(df[weight_col], errors="coerce").fillna(0)
    mask = vals.notna() & (weights > 0)
    if not mask.any():
        return None
    return float(np.average(vals[mask], weights=weights[mask]))


def _aggregate_eval_group(df, label):
    gt_files = int(df["n_gt_boxes"].notna().sum()) if "n_gt_boxes" in df.columns else 0
    files = int(len(df))
    n_pred = _sum_numeric(df, "n_pred_boxes")
    n_gt = _sum_numeric(df, "n_gt_boxes")
    tp = _sum_numeric(df, "gt_tp")
    fp = _sum_numeric(df, "gt_fp")
    fn = _sum_numeric(df, "gt_fn")

    strict_precision = _safe_div(tp, tp + fp)
    strict_recall = _safe_div(tp, tp + fn)
    strict_f1 = _safe_div(2 * strict_precision * strict_recall, strict_precision + strict_recall)

    # Temporal binary metrics are micro-averaged by summing windows.
    t_tp = _sum_numeric(df, "gt_temporal_tp_windows")
    t_fp = _sum_numeric(df, "gt_temporal_fp_windows")
    t_fn = _sum_numeric(df, "gt_temporal_fn_windows")
    t_tn = _sum_numeric(df, "gt_temporal_tn_windows")
    t_n = t_tp + t_fp + t_fn + t_tn
    t_precision = _safe_div(t_tp, t_tp + t_fp)
    t_recall = _safe_div(t_tp, t_tp + t_fn)
    t_f1 = _safe_div(2 * t_precision * t_recall, t_precision + t_recall)
    t_iou = _safe_div(t_tp, t_tp + t_fp + t_fn)
    t_accuracy = _safe_div(t_tp + t_tn, t_n)
    t_specificity = _safe_div(t_tn, t_tn + t_fp)
    t_bal_acc = 0.5 * (t_recall + t_specificity)

    matched_weight = "gt_tp"
    gt_weight = "n_gt_boxes"

    return {
        "scope": label,
        "files": files,
        "files_with_gt": gt_files,
        "pred_boxes": int(n_pred),
        "gt_boxes": int(n_gt),
        "pred_per_gt": _safe_div(n_pred, n_gt),
        "strict_tp": int(tp),
        "strict_fp": int(fp),
        "strict_fn": int(fn),
        "strict_precision": strict_precision,
        "strict_hit_rate": strict_recall,
        "strict_f1": strict_f1,
        "matched_iou_mean": _weighted_mean_numeric(df, "gt_mean_iou", matched_weight),
        "matched_temporal_iou_mean": _weighted_mean_numeric(df, "gt_mean_temporal_iou", matched_weight),
        "matched_frequency_iou_mean": _weighted_mean_numeric(df, "gt_mean_frequency_iou", matched_weight),
        "best_iou_mean": _weighted_mean_numeric(df, "gt_best_iou_mean", gt_weight),
        "best_temporal_iou_mean": _weighted_mean_numeric(df, "gt_best_temporal_iou_mean", gt_weight),
        "best_frequency_iou_mean": _weighted_mean_numeric(df, "gt_best_frequency_iou_mean", gt_weight),
        "temporal_coverage_mean": _weighted_mean_numeric(df, "gt_temporal_coverage_mean", gt_weight),
        "frequency_coverage_mean": _weighted_mean_numeric(df, "gt_frequency_coverage_mean", gt_weight),
        "iou_hit_rate_mean": _weighted_mean_numeric(df, "gt_iou_hit_rate", gt_weight),
        "any_overlap_hit_rate_mean": _weighted_mean_numeric(df, "gt_any_overlap_hit_rate", gt_weight),
        "temporal_accuracy": t_accuracy,
        "temporal_balanced_accuracy": t_bal_acc,
        "temporal_precision": t_precision,
        "temporal_hit_rate": t_recall,
        "temporal_f1": t_f1,
        "temporal_iou": t_iou,
    }


def _format_eval_table(df):
    out = df.copy()
    int_cols = [
        "files", "files_with_gt", "pred_boxes", "gt_boxes",
        "strict_tp", "strict_fp", "strict_fn",
    ]
    for c in int_cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0).astype(int)

    float_cols = [c for c in out.columns if c not in ["scope"] + int_cols]
    for c in float_cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").map(
                lambda x: "" if pd.isna(x) else f"{x:.3f}"
            )
    return out


def build_and_print_evaluation_tables(summary_df, out_dir):
    if summary_df is None or not len(summary_df):
        return None, None

    if "n_gt_boxes" not in summary_df.columns:
        print("\nNo GT evaluation columns found; skipping evaluation tables.")
        return None, None

    eval_df = summary_df.copy()
    eval_df["n_gt_boxes"] = pd.to_numeric(eval_df["n_gt_boxes"], errors="coerce")
    eval_df = eval_df[eval_df["n_gt_boxes"].fillna(0) > 0].copy()

    if not len(eval_df):
        print("\nNo files with GT boxes found; skipping evaluation tables.")
        return None, None

    overall = pd.DataFrame([_aggregate_eval_group(eval_df, "OVERALL")])
    per_site = pd.DataFrame([
        _aggregate_eval_group(g, str(site))
        for site, g in eval_df.groupby("dataset", dropna=False)
    ]).sort_values("scope")

    if SAVE_EVALUATION_TABLES:
        overall.to_csv(Path(out_dir) / "evaluation_overall.csv", index=False)
        per_site.to_csv(Path(out_dir) / "evaluation_by_site.csv", index=False)
        _format_eval_table(overall).to_csv(Path(out_dir) / "evaluation_overall_pretty.csv", index=False)
        _format_eval_table(per_site).to_csv(Path(out_dir) / "evaluation_by_site_pretty.csv", index=False)

    # A compact terminal table that is still readable.
    compact_cols = [
        "scope", "files", "pred_boxes", "gt_boxes", "pred_per_gt",
        "strict_precision", "strict_hit_rate", "strict_f1",
        "matched_iou_mean", "matched_temporal_iou_mean", "matched_frequency_iou_mean",
        "temporal_coverage_mean", "frequency_coverage_mean",
        "any_overlap_hit_rate_mean", "temporal_accuracy", "temporal_hit_rate",
    ]
    compact_overall = _format_eval_table(overall[[c for c in compact_cols if c in overall.columns]])
    compact_site = _format_eval_table(per_site[[c for c in compact_cols if c in per_site.columns]])

    print("\n=== EVALUATION SUMMARY: OVERALL ===")
    print(compact_overall.to_string(index=False))

    print("\n=== EVALUATION SUMMARY: PER SITE ===")
    print(compact_site.to_string(index=False))

    print("\nEvaluation table notes:")
    print("  strict_* = one-to-one GT-confirmed box matching; unmatched predictions count as false positives.")
    print("  GT coverage / overlap hit rate = GT-centred coverage metrics; less punitive when GT is incomplete.")
    print("  temporal_* = time-only binary activity metrics over analysis windows; useful for temporal alignment.")

    return overall, per_site


# -----------------------------
# MAIN: NESTED FOLDER SCAN
# -----------------------------

if __name__ == "__main__":
    out_dir = Path(OUTPUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)

    datasets = discover_dataset_folders(DATA_ROOT)

    # Fallback to single-file mode if no nested folders are found.
    if not datasets:
        if os.path.exists(WAV_PATH):
            fallback_gt_dir = Path(GT_PATH).parent if GT_PATH is not None else None
            datasets = [{
                "dataset_name": "single_file",
                "dataset_dir": Path(WAV_PATH).parent,
                "audio_dir": Path(WAV_PATH).parent,
                "gt_dir": fallback_gt_dir,
                "audio_files": [Path(WAV_PATH)],
                "n_audio_files": 1,
            }]
        else:
            raise FileNotFoundError(
                f"No nested dataset folders found in DATA_ROOT={DATA_ROOT}, "
                f"and WAV_PATH does not exist: {WAV_PATH}"
            )

    total_files = sum(len(d["audio_files"]) for d in datasets)

    print(f"Found {len(datasets)} dataset folders")
    print(f"Found {total_files} audio files total")
    print(f"Output directory: {out_dir.resolve()}")
    print(f"RUN_PER_FILE_GRID_SEARCH = {RUN_PER_FILE_GRID_SEARCH}")
    print(f"SKIP_COMPLETED_FILES = {SKIP_COMPLETED_FILES}")

    for dataset in datasets:
        print(
            f"  - {dataset['dataset_name']}: "
            f"{len(dataset['audio_files'])} audio files | "
            f"audio={dataset['audio_dir']} | "
            f"annotations={dataset['gt_dir']}"
        )

    summary_rows = []
    failures = []
    file_counter = 0

    for dataset in datasets:
        dataset_name = dataset["dataset_name"]
        audio_files = dataset["audio_files"]
        gt_dir = dataset["gt_dir"]

        print(f"\n=== Dataset: {dataset_name} ===")

        for wav_path in audio_files:
            file_counter += 1
            print(f"\n[{file_counter}/{total_files}] {wav_path}")

            try:
                row = process_one_file(
                    wav_path=wav_path,
                    out_dir=out_dir,
                    gt_folder=gt_dir,
                    dataset_name=dataset_name,
                )
                summary_rows.append(row)

                if row.get("skipped_existing"):
                    print(f"Skipped existing output: {row.get('output_dir')}")
                    continue

                if row.get("gt_f1") is not None and not pd.isna(row.get("gt_f1")):
                    print(
                        f"GT strict F1={row['gt_f1']:.3f} | P={row['gt_precision']:.3f} | "
                        f"hit={row['gt_hit_rate']:.3f} | IoU={row['gt_mean_iou']:.3f} | "
                        f"temp_cov={row.get('gt_temporal_coverage_mean', 0):.3f} | "
                        f"freq_cov={row.get('gt_frequency_coverage_mean', 0):.3f} | "
                        f"boxes={row['n_pred_boxes']}/{row['n_gt_boxes']}"
                    )
                else:
                    print(
                        f"No GT | proxy={row['proxy_proxy_score']:.3f} | "
                        f"event_corr={row['box_activity_event_corr']:.3f} | boxes={row['n_pred_boxes']}"
                    )

            except Exception as e:
                print(f"FAILED: {wav_path}: {e}")
                traceback.print_exc()
                failures.append({
                    "dataset": dataset_name,
                    "file": str(wav_path),
                    "error": repr(e),
                })

    summary_df = pd.DataFrame(summary_rows)
    summary_csv = out_dir / "batch_summary.csv"
    summary_df.to_csv(summary_csv, index=False)

    build_and_print_evaluation_tables(summary_df, out_dir)

    if len(summary_df) and "skipped_existing" in summary_df.columns:
        skipped_df = summary_df[summary_df["skipped_existing"] == True].copy()
        if len(skipped_df):
            skipped_csv = out_dir / "batch_skipped_existing.csv"
            skipped_df.to_csv(skipped_csv, index=False)
            print(f"\nSkipped existing files: {len(skipped_df)}. See {skipped_csv}")

    if len(summary_df):
        if "skipped_existing" in summary_df.columns:
            rankable_df = summary_df[summary_df["skipped_existing"] != True].copy()
        else:
            rankable_df = summary_df.copy()

        if len(rankable_df):
            if "gt_f1" in rankable_df.columns and rankable_df["gt_f1"].notna().any():
                ranked = rankable_df.sort_values(
                    ["gt_f1", "gt_mean_iou", "box_activity_event_corr"],
                    ascending=False,
                )
            else:
                ranked = rankable_df.sort_values(
                    ["box_activity_event_corr", "proxy_proxy_score"],
                    ascending=False,
                )

            ranked.to_csv(out_dir / "batch_summary_ranked.csv", index=False)

            print("\n=== TOP FILES ===")
            show_cols = [
                c for c in [
                    "dataset", "file", "gt_f1", "gt_recall", "gt_precision",
                    "gt_mean_iou", "box_activity_event_corr", "n_pred_boxes",
                    "n_gt_boxes", "output_dir"
                ] if c in ranked.columns
            ]
            print(ranked[show_cols].head(15).to_string(index=False))

    if failures:
        failures_csv = out_dir / "batch_failures.csv"
        pd.DataFrame(failures).to_csv(failures_csv, index=False)
        print(f"\nFailures: {len(failures)}. See {failures_csv}")

    print(f"\nDone. Summary: {summary_csv}")


<>:55: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<>:56: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<>:55: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<>:56: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
C:\Users\Dr. Tom\AppData\Local\Temp\ipykernel_27584\703614143.py:55: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
  DATA_ROOT = "D:\WABAD"
C:\Users\Dr. Tom\AppData\Local\Temp\ipykernel_27584\703614143.py:56: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not

Found 9 dataset folders
Found 711 audio files total
Output directory: D:\WABAD\batch_gmm_outputs
RUN_PER_FILE_GRID_SEARCH = False
SKIP_COMPLETED_FILES = True
  - BAM (1): 118 audio files | audio=D:\WABAD\BAM (1)\Recordings | annotations=D:\WABAD\BAM (1)\Raven Pro annotations
  - BIAL: 100 audio files | audio=D:\WABAD\BIAL\Recordings | annotations=D:\WABAD\BIAL\Raven Pro annotations
  - CLH (1): 61 audio files | audio=D:\WABAD\CLH (1)\Recordings | annotations=D:\WABAD\CLH (1)\Raven Pro annotations
  - DONG: 60 audio files | audio=D:\WABAD\DONG\Recordings | annotations=D:\WABAD\DONG\Raven Pro annotations
  - FEU: 69 audio files | audio=D:\WABAD\FEU\Recordings | annotations=D:\WABAD\FEU\Raven Pro annotations
  - LIM: 92 audio files | audio=D:\WABAD\LIM\Recordings | annotations=D:\WABAD\LIM\Raven Pro annotations
  - MAPIMI: 64 audio files | audio=D:\WABAD\MAPIMI\Recordings | annotations=D:\WABAD\MAPIMI\Raven Pro annotations
  - PITI (1): 48 audio files | audio=D:\WABAD\PITI (1)\Recordings 

In [ ]:
# ============================================================
# DATA-DRIVEN MULTI-BAND DRIFT GROUPING + TIME/FREQUENCY HYBRID SIMILARITY
#
# This version keeps the original time-based drift peak logic, but changes
# the band merging stage so that clustering is driven by a combined similarity:
#
#   S_total = alpha * S_time + beta * S_freq + gamma * S_adjacency
#
# where:
#   - S_time comes from the existing drift-based similarity
#   - S_freq comes from stable per-band spectral descriptors
#   - S_adjacency mildly prefers merging nearby bands
#
# The rest of the downstream pipeline is kept close to the original baseline.
# ============================================================

import gc
import os
import glob
import re
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib

# Non-interactive backend for restart-safe batch plotting.
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from scipy.stats import pearsonr, spearmanr
from sklearn.mixture import GaussianMixture

# -----------------------------
# INPUTS: SERF 60-SECOND CLIPS
# -----------------------------
# This version expects the output produced by SERF_split_60s_keep_labelled.py:
#
# DATA_ROOT/
#   Site_name/
#     Original_recording_name/
#       <clip_stem>.flac
#       <clip_stem>_annotations.csv
#
# Each audio file is already one retained 60-second clip. The companion CSV
# contains GT boxes with times relative to that clip, so this script must not
# split the audio again or align it against the original two-hour recording.
DATA_ROOT = Path(r"D:\WABAD\SERF\_serf_labelled_60s_audio")
OUTPUT_DIR = Path(r"D:\WABAD\SERF\_serf_60s_gmm_outputs")

# Optional single-clip fallback. Normally leave both as None.
WAV_PATH = None
GT_PATH = None

# Audio formats that may have been produced by the splitting script.
AUDIO_EXTENSIONS = (
    "*.wav", "*.WAV",
    "*.flac", "*.FLAC",
    "*.mp3", "*.MP3",
    "*.ogg", "*.OGG",
    "*.m4a", "*.M4A",
)
GT_EXTENSIONS = (".txt", ".csv", ".tsv")

# Search recursively through site/original-recording folders.
RECURSIVE_AUDIO_SEARCH = True
RECURSIVE_GT_SEARCH = True

# The splitter writes one companion CSV per retained clip. Keeping this True
# prevents unrelated audio from being processed accidentally.
REQUIRE_COMPANION_GT = True

# If True, skip files that already have completed output from a previous run.
# This makes the script restart-safe for large folders.
SKIP_COMPLETED_FILES = True

# Completion is checked using both the predicted boxes CSV and a small done marker.
# The done marker is written only after the file finishes successfully.
# If you have older outputs from before this marker existed, set this to False
# so the script can skip based on the predicted boxes CSV alone.
REQUIRE_DONE_MARKER_FOR_SKIP = False

# Write a per-file summary CSV inside each output folder. This helps preserve
# metrics for completed files even if the full batch later crashes.
SAVE_PER_FILE_SUMMARY = True

# If True, optimise GMM post-cluster parameters per file. Useful for exploration
# and picking promising examples, but it uses GT when available, so report fixed
# parameter results separately for generalisability claims.
RUN_PER_FILE_GRID_SEARCH = False
SAVE_PER_FILE_PLOTS = True
SAVE_CLUSTER_DRIFT_PLOTS = True
SAVE_PREDICTED_BOX_CSVS = True

# GT matching
GT_IOU_THRESHOLD = 0.30

# Evaluation controls.
# Strict metrics count unmatched predicted boxes as false positives.
# Coverage metrics focus on whether the GT annotations were found, which is
# useful when the GT is incomplete and the detector produces extra plausible boxes.
GT_TEMPORAL_HIT_IOU_THRESHOLD = 0.10
GT_FREQUENCY_HIT_IOU_THRESHOLD = 0.10
GT_ANY_OVERLAP_AS_HIT = True
TEMPORAL_EVAL_HOP_S = 0.125  # should normally match HOP_S below
RECOMPUTE_EVAL_FOR_SKIPPED_FILES = True
SAVE_EVALUATION_TABLES = True

# Grid-search values used only when RUN_PER_FILE_GRID_SEARCH=True
GRID_GMM_MIN_EVENT_POSTERIOR = [0.35, 0.40, 0.45, 0.50, 0.55]
GRID_GMM_SMOOTH_LABELS_K = [1, 3, 5]
GRID_GMM_MERGE_GAP_WINDOWS = [1, 2, 3, 5]
GRID_GMM_MIN_EVENT_WINDOWS = [1, 2]
GRID_GMM_USE_LOG_DRIFT = [False, True]
GRID_GMM_MIN_EVENT_AREA_Q = [None, 0.05, 0.10]

# When no GT exists, rank grid-search configurations using a proxy objective.
PROXY_CORR_WEIGHT = 0.80
PROXY_PEAK_F1_WEIGHT = 0.20
PROXY_BOX_DENSITY_PENALTY = 0.03

# SERF annotations extend above 10 kHz, so 44.1 kHz preserves GT up to 18 kHz.
TARGET_SR = 44100

# Micro-windowing
WINDOW_S = 0.25
HOP_S = 0.125

# Spectral representation used for per-band drift
N_FFT = 1024
HOP_LENGTH = 256
N_MELS_FULL = 128
FMIN = 200.0
FMAX = 18000.0

# Fine band decomposition
N_FINE_BANDS = 20
BAND_SCALE = "mel"   # "mel" or "linear"

# Drift smoothing
DRIFT_SMOOTH_K = 7

# Peak / boundary evidence extraction
PEAK_K_MAD = 2.5
PEAK_MIN_SEP_WINDOWS = 2

# Time similarity
SIMILARITY_MODE = "lagged_corr"   # "jaccard_binary" or "lagged_corr"
MAX_LAG_WINDOWS = 2

# New hybrid similarity weights
ALPHA_TIME = 0.70
BETA_FREQ = 0.20
GAMMA_ADJ = 0.10
ADJ_TAU = 2.5

# Frequency descriptor controls
FREQ_DESC_USE_ACTIVITY = True
FREQ_DESC_ACTIVITY_K_MAD = 2.0
FREQ_DESC_NORMALIZE_SUBBAND = True

# Hierarchical clustering
LINKAGE_METHOD = "average"
N_GROUPS = None
DISTANCE_THRESHOLD = 0.55

# Merged cluster drift aggregation
CLUSTER_AGG = "median"   # "median", "mean", "max"

# Plotting
PLOT_SECONDS_LIMIT = None
FIG_DPI = 120

# Spectrogram-event comparison
EVENT_SMOOTH_K = 7
EVENT_PEAK_K_MAD = 2.0
EVENT_PEAK_MIN_SEP_WINDOWS = 2
PEAK_MATCH_TOL_WINDOWS = 2

# Peak -> box controls
BOX_EXPAND_WINDOWS = 0
BOX_MIN_DURATION_WINDOWS = 1
BOX_SMOOTH_ACTIVITY_K = 5

# Post-cluster state segmentation controls
# This replaces threshold/peak-only detection for final boxes.
# Each merged cluster drift is treated as a 2-state problem:
# background vs event. A 2-component GMM is fitted per merged cluster.
POST_CLUSTER_DETECTOR = "gmm_state"
GMM_RANDOM_STATE = 0
GMM_MIN_EVENT_POSTERIOR = 0.50      # 0.50 = standard GMM decision boundary; raise to 0.60-0.70 for stricter boxes
GMM_SMOOTH_LABELS_K = 3             # odd integer; set 1 to disable short flicker smoothing
GMM_MERGE_GAP_WINDOWS = 2           # merge events separated by <= this many windows
GMM_MIN_EVENT_WINDOWS = 2           # remove very short events
GMM_MIN_EVENT_AREA_Q = None         # e.g. 0.10 to remove weakest 10% of event regions, None disables
GMM_USE_LOG_DRIFT = False           # True can help when one cluster has extreme spikes

# Optional preprocessing / denoising controls
DENOISE_METHOD = "hpss_percussive"
# options:
# "none"
# "spectral_gate"
# "hpss_percussive"
# "hpss_residual"
# "median_specsub"

USE_PCEN = True

# spectral gate params
NOISE_PROFILE_SEC = 2.0
SPEC_SUB_FLOOR = 0.1

# HPSS params
HPSS_KERNEL_SIZE = 31
HPSS_POWER = 2.0
HPSS_MARGIN = (1.0, 5.0)

# median spectral subtraction params
SPEC_FLOOR_MODE = "median"
SPEC_FLOOR_STRENGTH = 1.0

# -----------------------------
# UTILS
# -----------------------------

def moving_average(x: np.ndarray, k: int) -> np.ndarray:
    if k <= 1:
        return x.copy()
    pad = k // 2
    xp = np.pad(x, (pad, pad), mode="edge")
    w = np.ones(k, dtype=float) / k
    return np.convolve(xp, w, mode="valid")


def robust_mad(x: np.ndarray) -> float:
    med = np.median(x)
    return float(np.median(np.abs(x - med)) + 1e-12)


def robust_zscore(x: np.ndarray) -> np.ndarray:
    med = np.median(x)
    mad = robust_mad(x)
    return (x - med) / mad


def minmax01(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    mn = np.min(x)
    mx = np.max(x)
    if mx - mn < 1e-12:
        return np.zeros_like(x)
    return (x - mn) / (mx - mn)


def safe_pearson(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if np.std(x) < 1e-12 or np.std(y) < 1e-12:
        return 0.0
    r = pearsonr(x, y)[0]
    if np.isnan(r):
        return 0.0
    return float(r)


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na < 1e-12 or nb < 1e-12:
        return 0.0
    s = float(np.dot(a, b) / (na * nb))
    return max(0.0, min(1.0, 0.5 * (s + 1.0)))


def make_fixed_windows(duration_s: float, window_s: float, hop_s: float):
    starts = []
    ends = []
    t = 0.0
    while t + window_s <= duration_s + 1e-12:
        starts.append(t)
        ends.append(t + window_s)
        t += hop_s
    return np.array(starts, dtype=float), np.array(ends, dtype=float)


def nonmax_suppress_peaks(peaks, scores, radius):
    if radius <= 0 or len(peaks) <= 1:
        return sorted(peaks)
    peaks_sorted = sorted(peaks, key=lambda i: scores[i], reverse=True)
    blocked = np.zeros(len(scores), dtype=bool)
    kept = []
    for p in peaks_sorted:
        if blocked[p]:
            continue
        kept.append(p)
        lo = max(0, p - radius)
        hi = min(len(scores), p + radius + 1)
        blocked[lo:hi] = True
    return sorted(kept)


def detect_peaks_from_drift(drift: np.ndarray, k_mad: float, min_sep_windows: int):
    z = robust_zscore(drift)
    thr = np.median(z) + k_mad * robust_mad(z)

    peaks = []
    for i in range(1, len(z) - 1):
        if z[i] >= thr and z[i] >= z[i - 1] and z[i] >= z[i + 1]:
            peaks.append(i)

    peaks = nonmax_suppress_peaks(peaks, z, radius=min_sep_windows)

    evidence = np.zeros(len(z), dtype=int)
    evidence[peaks] = 1
    return z, evidence, peaks, thr


def band_edges_hz(fmin: float, fmax: float, n_bands: int, scale: str = "mel") -> np.ndarray:
    if scale == "mel":
        m0 = librosa.hz_to_mel(fmin)
        m1 = librosa.hz_to_mel(fmax)
        m_edges = np.linspace(m0, m1, n_bands + 1)
        return librosa.mel_to_hz(m_edges)
    elif scale == "linear":
        return np.linspace(fmin, fmax, n_bands + 1)
    else:
        raise ValueError("scale must be 'mel' or 'linear'")


# -----------------------------
# PREPROCESSING / DENOISING
# -----------------------------

def denoise_spectral_gate(y, sr, n_fft=1024, hop_length=256,
                          noise_profile_sec=2.0, floor=0.1):
    S = librosa.stft(y, n_fft=n_fft, hop_length=hop_length)
    mag = np.abs(S)
    phase = np.angle(S)

    noise_frames = max(1, int(round((noise_profile_sec * sr) / hop_length)))
    noise_frames = min(noise_frames, mag.shape[1])

    noise_mag = np.median(mag[:, :noise_frames], axis=1, keepdims=True)
    mag_denoised = mag - noise_mag
    mag_denoised = np.maximum(mag_denoised, floor * mag)

    S_denoised = mag_denoised * np.exp(1j * phase)
    y_denoised = librosa.istft(S_denoised, hop_length=hop_length, length=len(y))
    return y_denoised


def denoise_hpss_percussive(y, kernel_size=31, power=2.0, margin=(1.0, 5.0)):
    y_h, y_p = librosa.effects.hpss(
        y,
        kernel_size=kernel_size,
        power=power,
        margin=margin
    )
    return y_p


def denoise_hpss_residual(y, kernel_size=31, power=2.0, margin=(1.0, 5.0)):
    y_h, y_p = librosa.effects.hpss(
        y,
        kernel_size=kernel_size,
        power=power,
        margin=margin
    )
    residual = y - y_h
    return residual


def subtract_stationary_band_floor(X, strength=1.0, floor_mode="median"):
    X = np.asarray(X, dtype=float)

    if floor_mode == "median":
        bg = np.median(X, axis=0, keepdims=True)
    elif floor_mode == "mean":
        bg = np.mean(X, axis=0, keepdims=True)
    else:
        raise ValueError("floor_mode must be 'median' or 'mean'")

    out = X - strength * bg
    return np.maximum(out, 0.0)


def apply_waveform_preprocessing(y, sr):
    if DENOISE_METHOD == "none":
        return y

    if DENOISE_METHOD == "spectral_gate":
        print("Applying spectral gate denoising...")
        return denoise_spectral_gate(
            y=y,
            sr=sr,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            noise_profile_sec=NOISE_PROFILE_SEC,
            floor=SPEC_SUB_FLOOR
        )

    if DENOISE_METHOD == "hpss_percussive":
        print("Applying HPSS percussive emphasis...")
        return denoise_hpss_percussive(
            y,
            kernel_size=HPSS_KERNEL_SIZE,
            power=HPSS_POWER,
            margin=HPSS_MARGIN
        )

    if DENOISE_METHOD == "hpss_residual":
        print("Applying HPSS residual emphasis...")
        return denoise_hpss_residual(
            y,
            kernel_size=HPSS_KERNEL_SIZE,
            power=HPSS_POWER,
            margin=HPSS_MARGIN
        )

    if DENOISE_METHOD == "median_specsub":
        return y

    raise ValueError(f"Unsupported DENOISE_METHOD: {DENOISE_METHOD}")


# -----------------------------
# FEATURE EXTRACTION
# -----------------------------

def compute_window_repr_vectors(y, sr, win_starts, win_ends, n_mels, fmin, fmax,
                                use_pcen=False):
    out = []

    for t0, t1 in zip(win_starts, win_ends):
        s0 = int(round(t0 * sr))
        s1 = int(round(t1 * sr))
        seg = y[s0:s1]

        if len(seg) < 16:
            out.append(np.zeros(n_mels, dtype=np.float32))
            continue

        n_fft_eff = min(N_FFT, 2 ** int(np.floor(np.log2(len(seg)))))
        n_fft_eff = max(n_fft_eff, 128)
        hop_eff = min(HOP_LENGTH, max(32, n_fft_eff // 4))

        S = librosa.feature.melspectrogram(
            y=seg,
            sr=sr,
            n_fft=n_fft_eff,
            hop_length=hop_eff,
            n_mels=n_mels,
            fmin=fmin,
            fmax=fmax,
            power=2.0,
        )

        if use_pcen:
            rep = librosa.pcen(S * (2.0 ** 16), sr=sr, hop_length=hop_eff)
        else:
            rep = librosa.power_to_db(S, ref=np.max)

        vec = rep.mean(axis=1)
        out.append(vec.astype(np.float32))

    return np.stack(out, axis=0)


def split_mel_vectors_into_bands(logmel_vectors, mel_freqs, edges_hz):
    band_vectors = []
    for bi in range(len(edges_hz) - 1):
        lo, hi = edges_hz[bi], edges_hz[bi + 1]
        if bi < len(edges_hz) - 2:
            mask = (mel_freqs >= lo) & (mel_freqs < hi)
        else:
            mask = (mel_freqs >= lo) & (mel_freqs <= hi)

        if np.sum(mask) == 0:
            band_vectors.append(np.zeros((logmel_vectors.shape[0], 1), dtype=np.float32))
        else:
            band_vectors.append(logmel_vectors[:, mask])
    return band_vectors


def compute_direct_band_drifts(band_vectors, smooth_k):
    drifts = []
    for Xb in band_vectors:
        d = np.zeros(Xb.shape[0], dtype=float)
        for t in range(1, Xb.shape[0]):
            d[t] = np.linalg.norm(Xb[t] - Xb[t - 1])
        ds = moving_average(d, smooth_k)
        drifts.append(ds)
    return drifts


def compute_band_frequency_descriptors(band_vectors, k_mad_activity=2.0,
                                       normalize_subband=True,
                                       use_activity=True):
    """
    Stable per-band descriptor.

    Each band gets a compact summary vector built from:
    - median energy per sub-bin
    - variability per sub-bin (MAD)
    - global stats: median, MAD, max, flatness proxy, activity proportion,
      burstiness, temporal sparsity, mean pairwise sub-bin correlation
    """
    descs = []

    for Xb in band_vectors:
        X = np.asarray(Xb, dtype=float)
        if X.ndim == 1:
            X = X[:, None]

        if normalize_subband:
            Xn = np.zeros_like(X)
            for j in range(X.shape[1]):
                col = X[:, j]
                Xn[:, j] = robust_zscore(col)
            X_work = Xn
        else:
            X_work = X

        med_sub = np.median(X_work, axis=0)
        mad_sub = np.median(np.abs(X_work - med_sub[None, :]), axis=0)

        collapsed = np.mean(X_work, axis=1)
        collapsed_z = robust_zscore(collapsed)
        thr = np.median(collapsed_z) + k_mad_activity * robust_mad(collapsed_z)
        active = (collapsed_z >= thr).astype(float)

        if use_activity:
            activity_prop = float(np.mean(active))
            runs = np.diff(np.pad(active.astype(int), (1, 1), mode='constant'))
            starts = np.where(runs == 1)[0]
            ends = np.where(runs == -1)[0]
            durations = ends - starts
            mean_run = float(np.mean(durations)) if len(durations) > 0 else 0.0
            n_runs = float(len(durations))
        else:
            activity_prop = 0.0
            mean_run = 0.0
            n_runs = 0.0

        global_median = float(np.median(collapsed))
        global_mad = float(robust_mad(collapsed))
        global_max = float(np.max(collapsed))
        global_mean = float(np.mean(collapsed))
        global_std = float(np.std(collapsed))
        burstiness = float(global_std / (abs(global_mean) + 1e-12))
        sparsity = float(np.mean(np.abs(collapsed_z) < 0.5))

        # Flatness-like ratio over positive-shifted data
        xp = collapsed - np.min(collapsed) + 1e-6
        gmean = float(np.exp(np.mean(np.log(xp))))
        amean = float(np.mean(xp))
        flatness = float(gmean / (amean + 1e-12))

        # Correlation between sub-bins inside the band
        if X_work.shape[1] >= 2:
            C = np.corrcoef(X_work.T)
            vals = C[np.triu_indices_from(C, k=1)]
            vals = vals[np.isfinite(vals)]
            mean_subcorr = float(np.mean(vals)) if len(vals) > 0 else 0.0
        else:
            mean_subcorr = 1.0

        desc = np.concatenate([
            med_sub,
            mad_sub,
            np.array([
                global_median,
                global_mad,
                global_max,
                burstiness,
                sparsity,
                flatness,
                activity_prop,
                mean_run,
                n_runs,
                mean_subcorr,
            ], dtype=float)
        ])
        descs.append(desc)

    max_len = max(len(d) for d in descs)
    out = []
    for d in descs:
        if len(d) < max_len:
            pad = np.zeros(max_len - len(d), dtype=float)
            d = np.concatenate([d, pad])
        out.append(d)
    return np.stack(out, axis=0)


# -----------------------------
# SIMILARITY / CLUSTERING
# -----------------------------

def jaccard_similarity_binary(a: np.ndarray, b: np.ndarray) -> float:
    inter = np.sum((a > 0) & (b > 0))
    union = np.sum((a > 0) | (b > 0))
    if union == 0:
        return 0.0
    return float(inter / union)


def lagged_corr_similarity(a: np.ndarray, b: np.ndarray, max_lag: int = 2) -> float:
    a = robust_zscore(a)
    b = robust_zscore(b)
    best = -1.0
    for lag in range(-max_lag, max_lag + 1):
        if lag < 0:
            aa = a[-lag:]
            bb = b[:len(aa)]
        elif lag > 0:
            aa = a[:-lag]
            bb = b[lag:]
        else:
            aa = a
            bb = b

        if len(aa) < 3:
            continue

        c = np.corrcoef(aa, bb)[0, 1]
        if np.isnan(c):
            c = 0.0
        best = max(best, c)

    return float(max(best, 0.0))


def build_time_similarity_matrix(drifts, evidences, mode="jaccard_binary", max_lag=2):
    n = len(drifts)
    S = np.eye(n, dtype=float)

    for i in range(n):
        for j in range(i + 1, n):
            if mode == "jaccard_binary":
                sim = jaccard_similarity_binary(evidences[i], evidences[j])
            elif mode == "lagged_corr":
                sim = lagged_corr_similarity(drifts[i], drifts[j], max_lag=max_lag)
            else:
                raise ValueError("Unsupported similarity mode")

            S[i, j] = sim
            S[j, i] = sim

    return S


def build_frequency_similarity_matrix(band_descs):
    n = band_descs.shape[0]
    S = np.eye(n, dtype=float)

    # robust standardization across bands
    X = np.asarray(band_descs, dtype=float)
    Xs = np.zeros_like(X)
    for j in range(X.shape[1]):
        col = X[:, j]
        Xs[:, j] = robust_zscore(col)

    for i in range(n):
        for j in range(i + 1, n):
            sim = cosine_sim(Xs[i], Xs[j])
            S[i, j] = sim
            S[j, i] = sim
    return S


def build_adjacency_similarity_matrix(n_bands, tau=2.5):
    S = np.eye(n_bands, dtype=float)
    for i in range(n_bands):
        for j in range(i + 1, n_bands):
            d = abs(i - j)
            sim = float(np.exp(-d / tau))
            S[i, j] = sim
            S[j, i] = sim
    return S


def combine_similarity_matrices(S_time, S_freq, S_adj,
                                alpha=0.7, beta=0.2, gamma=0.1):
    wsum = alpha + beta + gamma
    if wsum <= 0:
        raise ValueError("alpha + beta + gamma must be > 0")
    St = alpha / wsum
    Sf = beta / wsum
    Sa = gamma / wsum
    S = St * S_time + Sf * S_freq + Sa * S_adj
    S = np.clip(S, 0.0, 1.0)
    np.fill_diagonal(S, 1.0)
    return S


def cluster_bands_from_similarity(S, linkage_method="average", n_groups=None, distance_threshold=0.55):
    D = 1.0 - S
    np.fill_diagonal(D, 0.0)
    condensed = squareform(D, checks=False)
    Z = linkage(condensed, method=linkage_method)

    if n_groups is not None:
        labels = fcluster(Z, n_groups, criterion="maxclust")
    else:
        labels = fcluster(Z, distance_threshold, criterion="distance")

    return Z, labels


def aggregate_cluster_drifts(drifts, cluster_labels, agg="median"):
    uniq = sorted(np.unique(cluster_labels))
    cluster_drifts = []
    cluster_members = []

    for c in uniq:
        idx = np.where(cluster_labels == c)[0]
        M = np.stack([drifts[i] for i in idx], axis=0)

        if agg == "median":
            cd = np.median(M, axis=0)
        elif agg == "mean":
            cd = np.mean(M, axis=0)
        elif agg == "max":
            cd = np.max(M, axis=0)
        else:
            raise ValueError("agg must be median, mean, or max")

        cluster_drifts.append(cd)
        cluster_members.append(idx.tolist())

    return uniq, cluster_drifts, cluster_members


def reorder_by_dendrogram(Z):
    dend = dendrogram(Z, no_plot=True)
    return dend["leaves"]


def cluster_band_ranges(edges_hz, cluster_labels):
    uniq = sorted(np.unique(cluster_labels))
    out = {}
    for c in uniq:
        idx = np.where(cluster_labels == c)[0]
        lo = edges_hz[np.min(idx)]
        hi = edges_hz[np.max(idx) + 1]
        out[c] = (lo, hi, idx.tolist())
    return out


# -----------------------------
# EVENT CURVE FROM SPECTROGRAM
# -----------------------------

def compute_global_repr_vectors(y, sr, win_starts, win_ends, n_mels, fmin, fmax):
    return compute_window_repr_vectors(
        y=y,
        sr=sr,
        win_starts=win_starts,
        win_ends=win_ends,
        n_mels=n_mels,
        fmin=fmin,
        fmax=fmax,
        use_pcen=USE_PCEN
    )


def compute_spectrogram_event_curve(global_repr_vectors, smooth_k=7):
    d = np.zeros(global_repr_vectors.shape[0], dtype=float)
    for t in range(1, global_repr_vectors.shape[0]):
        diff = global_repr_vectors[t] - global_repr_vectors[t - 1]
        d[t] = np.linalg.norm(diff)
    d = moving_average(d, smooth_k)
    return d


def match_peaks_with_tolerance(ref_peaks, test_peaks, tol=2):
    ref_peaks = sorted(ref_peaks)
    test_peaks = sorted(test_peaks)

    matched_ref = set()
    matched_test = set()

    for i, rp in enumerate(ref_peaks):
        best_j = None
        best_dist = None
        for j, tp in enumerate(test_peaks):
            if j in matched_test:
                continue
            dist = abs(rp - tp)
            if dist <= tol and (best_dist is None or dist < best_dist):
                best_j = j
                best_dist = dist
        if best_j is not None:
            matched_ref.add(i)
            matched_test.add(best_j)

    tp = len(matched_test)
    fp = len(test_peaks) - tp
    fn = len(ref_peaks) - tp

    precision = tp / (tp + fp + 1e-12)
    recall = tp / (tp + fn + 1e-12)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def best_lagged_corr(a, b, max_lag=2):
    a = robust_zscore(a)
    b = robust_zscore(b)

    best_corr = -1.0
    best_lag = 0

    for lag in range(-max_lag, max_lag + 1):
        if lag < 0:
            aa = a[-lag:]
            bb = b[:len(aa)]
        elif lag > 0:
            aa = a[:-lag]
            bb = b[lag:]
        else:
            aa = a
            bb = b

        if len(aa) < 3:
            continue

        c = np.corrcoef(aa, bb)[0, 1]
        if np.isnan(c):
            c = 0.0

        if c > best_corr:
            best_corr = c
            best_lag = lag

    return float(best_corr), int(best_lag)


def compute_cluster_vs_event_metrics(cluster_drifts, event_curve, max_lag=2,
                                     drift_peak_k_mad=2.5, drift_peak_sep=2,
                                     event_peak_k_mad=2.0, event_peak_sep=2,
                                     peak_match_tol=2):
    metrics = []

    ev_z, ev_bin, ev_peaks, ev_thr = detect_peaks_from_drift(
        event_curve,
        k_mad=event_peak_k_mad,
        min_sep_windows=event_peak_sep
    )

    for i, cd in enumerate(cluster_drifts):
        x = robust_zscore(cd)
        y = robust_zscore(event_curve)

        if np.std(x) < 1e-12 or np.std(y) < 1e-12:
            pear = 0.0
            spear = 0.0
        else:
            pear = pearsonr(x, y)[0]
            spear = spearmanr(x, y).statistic
            if np.isnan(pear):
                pear = 0.0
            if np.isnan(spear):
                spear = 0.0

        lagcorr, bestlag = best_lagged_corr(cd, event_curve, max_lag=max_lag)

        cd_z, cd_bin, cd_peaks, cd_thr = detect_peaks_from_drift(
            cd,
            k_mad=drift_peak_k_mad,
            min_sep_windows=drift_peak_sep
        )

        peak_scores = match_peaks_with_tolerance(
            ref_peaks=ev_peaks,
            test_peaks=cd_peaks,
            tol=peak_match_tol
        )

        metrics.append({
            "cluster_index": i,
            "pearson_r": float(pear),
            "spearman_rho": float(spear),
            "max_lag_corr": float(lagcorr),
            "best_lag_windows": int(bestlag),
            "n_cluster_peaks": int(len(cd_peaks)),
            "n_event_peaks": int(len(ev_peaks)),
            "peak_precision": float(peak_scores["precision"]),
            "peak_recall": float(peak_scores["recall"]),
            "peak_f1": float(peak_scores["f1"]),
            "cluster_peaks": cd_peaks,
            "event_peaks": ev_peaks,
            "cluster_drift_z": cd_z,
            "event_curve_z": ev_z,
        })

    return metrics


# -----------------------------
# PEAK -> BOX LOGIC
# -----------------------------

def find_peak_support(z, peak_idx, thr, expand_windows=0):
    n = len(z)

    left = peak_idx
    while left > 0 and z[left] >= thr:
        left -= 1
    if z[left] < thr and left < peak_idx:
        left += 1

    right = peak_idx
    while right < n - 1 and z[right] >= thr:
        right += 1
    if z[right] < thr and right > peak_idx:
        right -= 1

    left = max(0, left - expand_windows)
    right = min(n - 1, right + expand_windows)

    if right < left:
        left = max(0, peak_idx - 1)
        right = min(n - 1, peak_idx + 1)

    return left, right


def merge_overlapping_time_boxes(boxes):
    if len(boxes) == 0:
        return []

    boxes = sorted(boxes, key=lambda b: (b["cluster_id"], b["t0"], b["t1"]))
    merged = [boxes[0].copy()]

    for b in boxes[1:]:
        prev = merged[-1]
        same_cluster = (prev["cluster_id"] == b["cluster_id"])
        overlapping = b["t0"] <= prev["t1"]

        if same_cluster and overlapping:
            prev["t1"] = max(prev["t1"], b["t1"])
        else:
            merged.append(b.copy())

    return merged



def smooth_binary_labels(labels, k=3):
    """Simple majority filter for binary labels."""
    labels = np.asarray(labels, dtype=int)
    if k <= 1:
        return labels.copy()
    if k % 2 == 0:
        k += 1
    pad = k // 2
    padded = np.pad(labels, (pad, pad), mode="edge")
    out = np.zeros_like(labels)
    for i in range(len(labels)):
        window = padded[i:i + k]
        out[i] = 1 if np.mean(window) >= 0.5 else 0
    return out


def binary_labels_to_regions(labels, merge_gap_windows=2, min_event_windows=2):
    """Convert 0/1 labels into contiguous event regions."""
    labels = np.asarray(labels, dtype=int)
    regions = []
    in_event = False
    start = None

    for i, lab in enumerate(labels):
        if lab == 1 and not in_event:
            in_event = True
            start = i
        elif lab == 0 and in_event:
            regions.append([start, i - 1])
            in_event = False

    if in_event:
        regions.append([start, len(labels) - 1])

    # Merge nearby regions
    if merge_gap_windows is not None and merge_gap_windows >= 0 and len(regions) > 1:
        merged = [regions[0]]
        for st, en in regions[1:]:
            prev = merged[-1]
            gap = st - prev[1] - 1
            if gap <= merge_gap_windows:
                prev[1] = en
            else:
                merged.append([st, en])
        regions = merged

    # Drop very short regions
    regions = [[st, en] for st, en in regions if (en - st + 1) >= min_event_windows]
    return regions


def fit_gmm_event_state(drift,
                        min_event_posterior=0.50,
                        smooth_labels_k=3,
                        merge_gap_windows=2,
                        min_event_windows=2,
                        min_event_area_q=None,
                        use_log_drift=False,
                        random_state=0):
    """
    Per-cluster 2-state segmentation using a Gaussian mixture model.

    The lower-mean component is interpreted as background and the higher-mean
    component as event. This estimates the non-event/event state from each
    post-cluster drift curve rather than using a manually chosen drift threshold.
    """
    raw = np.asarray(drift, dtype=float)

    # Optional compression for highly spiky clusters.
    if use_log_drift:
        shift = np.min(raw)
        x = np.log1p(raw - shift)
    else:
        x = raw.copy()

    if len(x) < 5 or np.std(x) < 1e-12:
        labels = np.zeros(len(x), dtype=int)
        return {
            "labels": labels,
            "regions": [],
            "threshold": float(np.inf),
            "means": np.array([float(np.mean(x)), float(np.mean(x))]),
            "weights": np.array([1.0, 0.0]),
            "event_component": 1,
            "posterior": np.zeros(len(x), dtype=float),
            "raw_event_areas": [],
        }

    X = x.reshape(-1, 1)
    gmm = GaussianMixture(
        n_components=2,
        covariance_type="full",
        random_state=random_state,
        reg_covar=1e-6,
        n_init=5,
    )
    gmm.fit(X)

    means = gmm.means_.flatten()
    weights = gmm.weights_.flatten()
    event_component = int(np.argmax(means))
    bg_component = 1 - event_component

    posterior = gmm.predict_proba(X)[:, event_component]
    labels = (posterior >= min_event_posterior).astype(int)
    labels = smooth_binary_labels(labels, k=smooth_labels_k)

    regions = binary_labels_to_regions(
        labels,
        merge_gap_windows=merge_gap_windows,
        min_event_windows=min_event_windows,
    )

    # Optional region-level area filtering. This is intentionally weak by default.
    raw_event_areas = []
    for st, en in regions:
        local_base = min(raw[st], raw[en]) if en > st else raw[st]
        area = float(np.sum(np.maximum(raw[st:en + 1] - local_base, 0.0)))
        raw_event_areas.append(area)

    if min_event_area_q is not None and len(regions) > 0:
        area_gate = float(np.quantile(raw_event_areas, min_event_area_q))
        kept_regions = []
        kept_areas = []
        for region, area in zip(regions, raw_event_areas):
            if area >= area_gate:
                kept_regions.append(region)
                kept_areas.append(area)
        regions = kept_regions
        raw_event_areas = kept_areas

        labels2 = np.zeros_like(labels)
        for st, en in regions:
            labels2[st:en + 1] = 1
        labels = labels2

    # Approximate threshold in the same space used by the GMM.
    # This is diagnostic only; boxes are built from posterior labels/regions.
    threshold = float(np.mean(means))
    if use_log_drift:
        threshold = float(np.expm1(threshold) + np.min(raw))

    return {
        "labels": labels,
        "regions": regions,
        "threshold": threshold,
        "means": means,
        "weights": weights,
        "event_component": event_component,
        "bg_component": bg_component,
        "posterior": posterior,
        "raw_event_areas": raw_event_areas,
    }


def cluster_peaks_to_boxes(cluster_drifts, cluster_members, cluster_labels_unique,
                           edges_hz, win_starts, win_ends,
                           peak_k_mad, peak_min_sep_windows,
                           expand_windows=0, min_duration_windows=1):
    all_boxes = []
    peak_info = []

    print("\n=== POST-CLUSTER GMM STATE SEGMENTATION ===")

    for cd, members, c in zip(cluster_drifts, cluster_members, cluster_labels_unique):
        f0 = float(edges_hz[min(members)])
        f1 = float(edges_hz[max(members) + 1])

        res = fit_gmm_event_state(
            cd,
            min_event_posterior=GMM_MIN_EVENT_POSTERIOR,
            smooth_labels_k=GMM_SMOOTH_LABELS_K,
            merge_gap_windows=GMM_MERGE_GAP_WINDOWS,
            min_event_windows=max(GMM_MIN_EVENT_WINDOWS, min_duration_windows),
            min_event_area_q=GMM_MIN_EVENT_AREA_Q,
            use_log_drift=GMM_USE_LOG_DRIFT,
            random_state=GMM_RANDOM_STATE,
        )

        labels = res["labels"]
        regions = res["regions"]

        cluster_boxes = []
        peaks = []
        for st, en in regions:
            segment = cd[st:en + 1]
            peak_idx = int(st + np.argmax(segment))
            peaks.append(peak_idx)

            box = {
                "cluster_id": int(c),
                "t0": float(win_starts[st]),
                "t1": float(win_ends[en]),
                "f0": f0,
                "f1": f1,
                "peak_idx": int(peak_idx),
                "support_left": int(st),
                "support_right": int(en),
                "members": members
            }
            cluster_boxes.append(box)

        cluster_boxes = merge_overlapping_time_boxes(cluster_boxes)
        all_boxes.extend(cluster_boxes)

        means_str = np.array2string(res["means"], precision=3, separator=", ")
        weights_str = np.array2string(res["weights"], precision=3, separator=", ")
        event_fraction = float(np.mean(labels)) if len(labels) else 0.0
        print(
            f"Cluster {c} | {int(f0):4d}-{int(f1):5d} Hz | "
            f"mode=gmm_state | thr≈{res['threshold']:.3f} | "
            f"means={means_str} | weights={weights_str} | "
            f"event_fraction={event_fraction:.3f} | regions={len(regions)} | boxes={len(cluster_boxes)}"
        )

        peak_info.append({
            "cluster_id": int(c),
            "freq_lo": f0,
            "freq_hi": f1,
            "n_peaks": int(len(peaks)),
            "n_boxes": int(len(cluster_boxes)),
            "peaks": peaks,
            "z": labels.astype(float),
            "thr": res["threshold"],
            "labels": labels,
            "posterior": res["posterior"],
            "regions": regions,
            "means": res["means"],
            "weights": res["weights"],
        })

    return all_boxes, peak_info

def boxes_to_global_activity_curve(boxes, n_windows, win_starts, win_ends, smooth_k=1):
    activity = np.zeros(n_windows, dtype=float)

    for b in boxes:
        overlap = (win_starts < b["t1"]) & (win_ends > b["t0"])
        activity[overlap] = 1.0

    if smooth_k > 1:
        activity = moving_average(activity, smooth_k)

    return activity


# -----------------------------
# OPTIONAL GT LOADING + SIMPLE BOX METRICS
# -----------------------------

def try_load_ground_truth_boxes(path):
    """
    Load GT time-frequency boxes.

    SERF 60-second companion CSVs use:
      - clip_start_seconds
      - clip_end_seconds
      - low_frequency_hertz
      - high_frequency_hertz

    Raven-style aliases remain supported for backwards compatibility.
    """
    if path is None:
        return []

    path = Path(path)
    if not path.exists():
        print(f"GT file not found: {path}")
        return []

    try:
        if path.suffix.lower() in {".txt", ".tsv"}:
            df = pd.read_csv(path, sep="\t", engine="python")
        else:
            df = pd.read_csv(path)
    except Exception as exc:
        print(f"Could not read GT file {path}: {type(exc).__name__}: {exc}")
        return []

    cols = {str(c).lower().strip(): c for c in df.columns}

    def pick(names):
        for name in names:
            key = name.lower().strip()
            if key in cols:
                return cols[key]
        return None

    c_t0 = pick([
        "clip_start_seconds",
        "clip start seconds",
        "begin time (s)",
        "begin time",
        "start time (s)",
        "start time",
        "t0",
    ])
    c_t1 = pick([
        "clip_end_seconds",
        "clip end seconds",
        "end time (s)",
        "end time",
        "stop time (s)",
        "stop time",
        "t1",
    ])
    c_f0 = pick([
        "low_frequency_hertz",
        "low frequency hertz",
        "low freq (hz)",
        "low freq",
        "low frequency",
        "f0",
    ])
    c_f1 = pick([
        "high_frequency_hertz",
        "high frequency hertz",
        "high freq (hz)",
        "high freq",
        "high frequency",
        "f1",
    ])

    if any(column is None for column in [c_t0, c_t1, c_f0, c_f1]):
        print(
            "Could not identify GT columns in "
            f"{path.name}. Expected SERF clip-relative or Raven time/frequency columns."
        )
        return []

    numeric = pd.DataFrame({
        "t0": pd.to_numeric(df[c_t0], errors="coerce"),
        "t1": pd.to_numeric(df[c_t1], errors="coerce"),
        "f0": pd.to_numeric(df[c_f0], errors="coerce"),
        "f1": pd.to_numeric(df[c_f1], errors="coerce"),
    }).dropna()

    numeric = numeric[
        (numeric["t1"] > numeric["t0"])
        & (numeric["f1"] > numeric["f0"])
    ]

    boxes = []
    for row in numeric.itertuples(index=False):
        boxes.append({
            "t0": max(0.0, float(row.t0)),
            "t1": max(0.0, float(row.t1)),
            "f0": max(0.0, float(row.f0)),
            "f1": max(0.0, float(row.f1)),
        })
    return boxes


def interval_iou(a0, a1, b0, b1):
    inter = max(0.0, min(a1, b1) - max(a0, b0))
    union = max(a1, b1) - min(a0, b0)
    if union <= 0:
        return 0.0
    return inter / union


def box_iou(a, b):
    it = max(0.0, min(a["t1"], b["t1"]) - max(a["t0"], b["t0"]))
    ifr = max(0.0, min(a["f1"], b["f1"]) - max(a["f0"], b["f0"]))
    inter = it * ifr
    area_a = max(0.0, a["t1"] - a["t0"]) * max(0.0, a["f1"] - a["f0"])
    area_b = max(0.0, b["t1"] - b["t0"]) * max(0.0, b["f1"] - b["f0"])
    union = area_a + area_b - inter
    if union <= 0:
        return 0.0
    return inter / union


def _safe_div(n, d):
    return float(n / d) if d not in (0, 0.0, None) and not pd.isna(d) else 0.0


def _box_duration(b):
    return max(0.0, float(b.get("t1", 0.0)) - float(b.get("t0", 0.0)))


def _box_bandwidth(b):
    return max(0.0, float(b.get("f1", 0.0)) - float(b.get("f0", 0.0)))


def _box_area(b):
    return _box_duration(b) * _box_bandwidth(b)


def _intersection_length(a0, a1, b0, b1):
    return max(0.0, min(float(a1), float(b1)) - max(float(a0), float(b0)))


def _union_length_intervals(intervals):
    """Union length of [start, end] intervals."""
    clean = []
    for lo, hi in intervals:
        lo = float(lo)
        hi = float(hi)
        if hi > lo:
            clean.append((lo, hi))
    if not clean:
        return 0.0
    clean.sort()
    total = 0.0
    cur_lo, cur_hi = clean[0]
    for lo, hi in clean[1:]:
        if lo <= cur_hi:
            cur_hi = max(cur_hi, hi)
        else:
            total += cur_hi - cur_lo
            cur_lo, cur_hi = lo, hi
    total += cur_hi - cur_lo
    return float(total)


def _pair_alignment_metrics(pred_box, gt_box):
    """Returns time, frequency, and 2D box alignment for one prediction/GT pair."""
    t_inter = _intersection_length(pred_box["t0"], pred_box["t1"], gt_box["t0"], gt_box["t1"])
    f_inter = _intersection_length(pred_box["f0"], pred_box["f1"], gt_box["f0"], gt_box["f1"])

    pred_t = _box_duration(pred_box)
    gt_t = _box_duration(gt_box)
    pred_f = _box_bandwidth(pred_box)
    gt_f = _box_bandwidth(gt_box)

    t_union = pred_t + gt_t - t_inter
    f_union = pred_f + gt_f - f_inter

    inter_area = t_inter * f_inter
    pred_area = _box_area(pred_box)
    gt_area = _box_area(gt_box)
    union_area = pred_area + gt_area - inter_area

    pred_centre_t = 0.5 * (float(pred_box["t0"]) + float(pred_box["t1"]))
    gt_centre_t = 0.5 * (float(gt_box["t0"]) + float(gt_box["t1"]))
    pred_centre_f = 0.5 * (float(pred_box["f0"]) + float(pred_box["f1"]))
    gt_centre_f = 0.5 * (float(gt_box["f0"]) + float(gt_box["f1"]))

    return {
        "temporal_iou": _safe_div(t_inter, t_union),
        "frequency_iou": _safe_div(f_inter, f_union),
        "iou": _safe_div(inter_area, union_area),
        "temporal_intersection_s": float(t_inter),
        "frequency_intersection_hz": float(f_inter),
        "temporal_recall_coverage": _safe_div(t_inter, gt_t),
        "frequency_recall_coverage": _safe_div(f_inter, gt_f),
        "area_recall_coverage": _safe_div(inter_area, gt_area),
        "temporal_centre_error_s": abs(pred_centre_t - gt_centre_t),
        "frequency_centre_error_hz": abs(pred_centre_f - gt_centre_f),
    }


def _make_temporal_activity_curve(boxes, win_starts=None, win_ends=None, duration_s=None, hop_s=None):
    """
    Converts boxes to a 0/1 temporal activity curve.
    This is used for time-only accuracy/precision/recall. It ignores frequency.
    """
    if win_starts is None or win_ends is None:
        if duration_s is None or duration_s <= 0:
            max_t = 0.0
            for b in boxes:
                max_t = max(max_t, float(b.get("t1", 0.0)))
            duration_s = max_t
        hop_s = hop_s or TEMPORAL_EVAL_HOP_S
        if duration_s <= 0 or hop_s <= 0:
            return np.zeros(0, dtype=int)
        win_starts = np.arange(0.0, duration_s + 1e-9, hop_s)
        win_ends = win_starts + hop_s

    activity = np.zeros(len(win_starts), dtype=int)
    for b in boxes:
        overlap = (win_starts < float(b["t1"])) & (win_ends > float(b["t0"]))
        activity[overlap] = 1
    return activity


def _binary_curve_metrics(pred_curve, gt_curve):
    n = int(min(len(pred_curve), len(gt_curve)))
    if n <= 0:
        return {
            "temporal_n_windows": 0,
            "temporal_tp_windows": 0,
            "temporal_fp_windows": 0,
            "temporal_fn_windows": 0,
            "temporal_tn_windows": 0,
            "temporal_accuracy": None,
            "temporal_precision": None,
            "temporal_recall": None,
            "temporal_hit_rate": None,
            "temporal_f1": None,
            "temporal_iou": None,
            "temporal_balanced_accuracy": None,
        }

    pred = np.asarray(pred_curve[:n], dtype=bool)
    gt = np.asarray(gt_curve[:n], dtype=bool)

    tp = int(np.sum(pred & gt))
    fp = int(np.sum(pred & ~gt))
    fn = int(np.sum(~pred & gt))
    tn = int(np.sum(~pred & ~gt))

    precision = _safe_div(tp, tp + fp)
    recall = _safe_div(tp, tp + fn)
    specificity = _safe_div(tn, tn + fp)
    f1 = _safe_div(2 * precision * recall, precision + recall)
    jaccard = _safe_div(tp, tp + fp + fn)
    acc = _safe_div(tp + tn, n)
    bal_acc = 0.5 * (recall + specificity)

    return {
        "temporal_n_windows": n,
        "temporal_tp_windows": tp,
        "temporal_fp_windows": fp,
        "temporal_fn_windows": fn,
        "temporal_tn_windows": tn,
        "temporal_accuracy": acc,
        "temporal_precision": precision,
        "temporal_recall": recall,
        "temporal_hit_rate": recall,
        "temporal_f1": f1,
        "temporal_iou": jaccard,
        "temporal_balanced_accuracy": bal_acc,
    }


def evaluate_predicted_boxes(pred_boxes, gt_boxes, iou_thr=0.3,
                             win_starts=None, win_ends=None,
                             duration_s=None, temporal_eval_hop_s=None,
                             temporal_hit_iou_thr=None,
                             frequency_hit_iou_thr=None):
    """
    Evaluates predicted time-frequency boxes against Raven GT boxes.

    Important interpretation:
    - strict precision/recall/F1 use one-to-one IoU matching and count unmatched
      predictions as false positives.
    - GT coverage/hit metrics ask whether annotated GT boxes were found and are
      less punitive when the detector produces additional boxes not present in GT.
    - temporal_* metrics are time-only binary activity metrics over analysis windows.
      They are useful for temporal alignment but can be biased by incomplete GT.
    """
    n_pred = len(pred_boxes)
    n_gt = len(gt_boxes)

    if n_gt == 0:
        return None

    temporal_hit_iou_thr = GT_TEMPORAL_HIT_IOU_THRESHOLD if temporal_hit_iou_thr is None else temporal_hit_iou_thr
    frequency_hit_iou_thr = GT_FREQUENCY_HIT_IOU_THRESHOLD if frequency_hit_iou_thr is None else frequency_hit_iou_thr
    temporal_eval_hop_s = TEMPORAL_EVAL_HOP_S if temporal_eval_hop_s is None else temporal_eval_hop_s

    # Pairwise metrics for every prediction/GT pair.
    pair_rows = []
    for pi, pb in enumerate(pred_boxes):
        for gi, gb in enumerate(gt_boxes):
            m = _pair_alignment_metrics(pb, gb)
            pair_rows.append({"pred_idx": pi, "gt_idx": gi, **m})

    # Strict one-to-one matching by descending 2D IoU.
    matched_pred = set()
    matched_gt = set()
    matched_rows = []
    for r in sorted(pair_rows, key=lambda x: x["iou"], reverse=True):
        if r["iou"] < iou_thr:
            break
        if r["pred_idx"] in matched_pred or r["gt_idx"] in matched_gt:
            continue
        matched_pred.add(r["pred_idx"])
        matched_gt.add(r["gt_idx"])
        matched_rows.append(r)

    tp = len(matched_rows)
    fp = n_pred - tp
    fn = n_gt - tp

    precision = _safe_div(tp, tp + fp)
    recall = _safe_div(tp, tp + fn)
    f1 = _safe_div(2 * precision * recall, precision + recall)

    def mean_from(rows, key):
        vals = [float(r[key]) for r in rows if r.get(key) is not None and not pd.isna(r.get(key))]
        return float(np.mean(vals)) if vals else 0.0

    def median_from(rows, key):
        vals = [float(r[key]) for r in rows if r.get(key) is not None and not pd.isna(r.get(key))]
        return float(np.median(vals)) if vals else 0.0

    # Best prediction per GT. This is deliberately not one-to-one, because many
    # predicted boxes may map to the same GT or split one GT into smaller regions.
    best_by_gt = []
    for gi, gb in enumerate(gt_boxes):
        rows = [r for r in pair_rows if r["gt_idx"] == gi]
        if rows:
            best_iou_row = max(rows, key=lambda x: x["iou"])
            best_temporal_row = max(rows, key=lambda x: x["temporal_iou"])
            best_frequency_row = max(rows, key=lambda x: x["frequency_iou"])
        else:
            best_iou_row = {
                "iou": 0.0, "temporal_iou": 0.0, "frequency_iou": 0.0,
                "area_recall_coverage": 0.0, "temporal_centre_error_s": None,
                "frequency_centre_error_hz": None,
            }
            best_temporal_row = {"temporal_iou": 0.0}
            best_frequency_row = {"frequency_iou": 0.0}

        # Union-based GT coverage: this is better when one GT is split into
        # several predicted boxes.
        t_intervals = []
        f_intervals = []
        for pb in pred_boxes:
            t0 = max(float(pb["t0"]), float(gb["t0"]))
            t1 = min(float(pb["t1"]), float(gb["t1"]))
            f0 = max(float(pb["f0"]), float(gb["f0"]))
            f1 = min(float(pb["f1"]), float(gb["f1"]))
            if t1 > t0:
                t_intervals.append((t0, t1))
            if t1 > t0 and f1 > f0:
                f_intervals.append((f0, f1))

        temporal_coverage = _safe_div(_union_length_intervals(t_intervals), _box_duration(gb))
        frequency_coverage = _safe_div(_union_length_intervals(f_intervals), _box_bandwidth(gb))

        any_overlap_hit = any(
            (r["temporal_intersection_s"] > 0 and r["frequency_intersection_hz"] > 0)
            for r in rows
        )

        best_by_gt.append({
            "gt_idx": gi,
            "best_iou": float(best_iou_row.get("iou", 0.0)),
            "best_temporal_iou": float(best_temporal_row.get("temporal_iou", 0.0)),
            "best_frequency_iou": float(best_frequency_row.get("frequency_iou", 0.0)),
            "best_area_recall_coverage": float(best_iou_row.get("area_recall_coverage", 0.0)),
            "temporal_coverage": float(temporal_coverage),
            "frequency_coverage": float(frequency_coverage),
            "temporal_centre_error_s": best_iou_row.get("temporal_centre_error_s"),
            "frequency_centre_error_hz": best_iou_row.get("frequency_centre_error_hz"),
            "hit_iou": float(best_iou_row.get("iou", 0.0)) >= iou_thr,
            "hit_temporal": float(best_temporal_row.get("temporal_iou", 0.0)) >= temporal_hit_iou_thr,
            "hit_frequency": float(best_frequency_row.get("frequency_iou", 0.0)) >= frequency_hit_iou_thr,
            "hit_any_overlap": bool(any_overlap_hit),
        })

    pred_curve = _make_temporal_activity_curve(
        pred_boxes, win_starts=win_starts, win_ends=win_ends,
        duration_s=duration_s, hop_s=temporal_eval_hop_s,
    )
    gt_curve = _make_temporal_activity_curve(
        gt_boxes, win_starts=win_starts, win_ends=win_ends,
        duration_s=duration_s, hop_s=temporal_eval_hop_s,
    )
    temporal_metrics = _binary_curve_metrics(pred_curve, gt_curve)

    # Conservative precision penalises all extra predicted boxes. This is often
    # useful, but can be harsh when GT is incomplete.
    strict_precision = precision

    out = {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "hit_rate": recall,
        "f1": f1,
        "mean_iou": mean_from(matched_rows, "iou"),
        "median_iou": median_from(matched_rows, "iou"),
        "mean_temporal_iou": mean_from(matched_rows, "temporal_iou"),
        "mean_frequency_iou": mean_from(matched_rows, "frequency_iou"),
        "mean_temporal_centre_error_s": mean_from(matched_rows, "temporal_centre_error_s"),
        "mean_frequency_centre_error_hz": mean_from(matched_rows, "frequency_centre_error_hz"),
        "strict_precision": strict_precision,
        "pred_to_gt_box_ratio": _safe_div(n_pred, n_gt),
        "n_unmatched_predictions": fp,
        "n_unmatched_gt": fn,
        "best_iou_mean": mean_from(best_by_gt, "best_iou"),
        "best_temporal_iou_mean": mean_from(best_by_gt, "best_temporal_iou"),
        "best_frequency_iou_mean": mean_from(best_by_gt, "best_frequency_iou"),
        "temporal_coverage_mean": mean_from(best_by_gt, "temporal_coverage"),
        "frequency_coverage_mean": mean_from(best_by_gt, "frequency_coverage"),
        "area_recall_coverage_mean": mean_from(best_by_gt, "best_area_recall_coverage"),
        "iou_hit_rate": _safe_div(sum(bool(x["hit_iou"]) for x in best_by_gt), n_gt),
        "box_temporal_hit_rate": _safe_div(sum(bool(x["hit_temporal"]) for x in best_by_gt), n_gt),
        "box_frequency_hit_rate": _safe_div(sum(bool(x["hit_frequency"]) for x in best_by_gt), n_gt),
        "any_overlap_hit_rate": _safe_div(sum(bool(x["hit_any_overlap"]) for x in best_by_gt), n_gt),
        **temporal_metrics,
    }

    return out


def load_boxes_csv_as_boxes(path):
    """Loads a predicted-box CSV written by save_boxes_csv()."""
    path = Path(path)
    if not path.exists():
        return []
    df = pd.read_csv(path)
    required = {"t0", "t1", "f0", "f1"}
    if not required.issubset(df.columns):
        return []
    boxes = []
    for _, row in df.iterrows():
        try:
            boxes.append({
                "t0": float(row["t0"]),
                "t1": float(row["t1"]),
                "f0": float(row["f0"]),
                "f1": float(row["f1"]),
            })
        except Exception:
            continue
    return boxes


def add_or_update_eval_metrics_for_existing_row(row, wav_path, out_dir, gt_folder, dataset_name):
    """
    Recomputes evaluation metrics for skipped files using existing predicted boxes.
    This avoids rerunning the detector just to build the evaluation tables.
    """
    if not RECOMPUTE_EVAL_FOR_SKIPPED_FILES:
        return row

    paths = get_file_output_paths(wav_path, out_dir, dataset_name)
    pred_boxes = load_boxes_csv_as_boxes(paths["predicted_boxes_csv"])
    gt_path = find_matching_gt(wav_path, gt_folder)
    gt_boxes = try_load_ground_truth_boxes(gt_path) if gt_path is not None else []

    if not gt_boxes:
        return row

    duration_s = row.get("duration_s")
    try:
        duration_s = float(duration_s)
    except Exception:
        duration_s = None
    if duration_s is None or pd.isna(duration_s) or duration_s <= 0:
        duration_s = max(
            [0.0]
            + [float(b.get("t1", 0.0)) for b in pred_boxes]
            + [float(b.get("t1", 0.0)) for b in gt_boxes]
        )

    gt_metrics = evaluate_predicted_boxes(
        pred_boxes,
        gt_boxes,
        iou_thr=GT_IOU_THRESHOLD,
        duration_s=duration_s,
        temporal_eval_hop_s=TEMPORAL_EVAL_HOP_S,
    )

    row.update({
        "gt_path": gt_path,
        "duration_s": duration_s,
        "n_pred_boxes": len(pred_boxes),
        "n_gt_boxes": len(gt_boxes),
    })
    if gt_metrics is not None:
        row.update({f"gt_{k}": v for k, v in gt_metrics.items()})
    return row


# -----------------------------
# PLOTTING
# -----------------------------

def maybe_limit_time(y, sr, sec_limit):
    if sec_limit is None:
        return y
    n = int(round(sec_limit * sr))
    return y[:n]


def get_time_mask(win_starts, sec_limit):
    if sec_limit is None:
        return np.ones_like(win_starts, dtype=bool)
    return win_starts <= sec_limit


def plot_spectrogram(y, sr, sec_limit=None):
    y2 = maybe_limit_time(y, sr, sec_limit)

    S = librosa.stft(y2, n_fft=2048, hop_length=512)
    S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)

    fig, ax = plt.subplots(figsize=(14, 5), dpi=FIG_DPI)
    img = librosa.display.specshow(
        S_db,
        sr=sr,
        hop_length=512,
        x_axis="time",
        y_axis="hz",
        fmin=0,
        fmax=FMAX,
        cmap="magma",
        ax=ax
    )
    ax.set_title("Spectrogram")
    ax.set_ylim(0, FMAX)
    plt.colorbar(img, ax=ax, format="%+2.0f dB")
    plt.tight_layout()
    plt.show()


def plot_similarity_matrix(S, title="Similarity Matrix"):
    fig, ax = plt.subplots(figsize=(7, 6), dpi=FIG_DPI)
    im = ax.imshow(S, origin="lower", aspect="auto", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel("Band index")
    ax.set_ylabel("Band index")
    plt.colorbar(im, ax=ax, label="Similarity")
    plt.tight_layout()
    plt.show()


def plot_similarity_matrix_reordered(S, order, title="Reordered Similarity Matrix"):
    Sr = S[np.ix_(order, order)]
    fig, ax = plt.subplots(figsize=(7, 6), dpi=FIG_DPI)
    im = ax.imshow(Sr, origin="lower", aspect="auto", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel("Reordered band index")
    ax.set_ylabel("Reordered band index")
    plt.colorbar(im, ax=ax, label="Similarity")
    plt.tight_layout()
    plt.show()


def plot_dendrogram(Z):
    fig, ax = plt.subplots(figsize=(10, 4), dpi=FIG_DPI)
    dendrogram(Z, ax=ax)
    ax.set_title("Hierarchical Clustering of Bands")
    ax.set_xlabel("Band index")
    ax.set_ylabel("Distance")
    plt.tight_layout()
    plt.show()


def plot_cluster_bandmap(edges_hz, cluster_labels):
    uniq = sorted(np.unique(cluster_labels))
    fig, ax = plt.subplots(figsize=(10, 2.2), dpi=FIG_DPI)

    for bi in range(len(edges_hz) - 1):
        lo = edges_hz[bi]
        hi = edges_hz[bi + 1]
        c = cluster_labels[bi]
        ax.fill_between([lo, hi], 0, 1, alpha=0.7,
                        label=f"C{c}" if bi == np.where(cluster_labels == c)[0][0] else None)

    ax.set_xlim(edges_hz[0], edges_hz[-1])
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_xlabel("Frequency (Hz)")
    ax.set_title("Discovered Band Grouping")
    ax.legend(ncol=min(len(uniq), 8), loc="upper center", bbox_to_anchor=(0.5, -0.18))
    plt.tight_layout()
    plt.show()


def plot_cluster_drifts(cluster_drifts, cluster_members, cluster_labels_unique, win_starts, edges_hz, sec_limit=None):
    t = win_starts.copy()
    mask = get_time_mask(t, sec_limit)

    n = len(cluster_drifts)
    fig, axes = plt.subplots(n, 1, figsize=(14, max(2.5 * n, 6)), sharex=True, dpi=FIG_DPI)

    if n == 1:
        axes = [axes]

    for ax, cd, members, c in zip(axes, cluster_drifts, cluster_members, cluster_labels_unique):
        tt = t[mask]
        yy = cd[mask]

        lo = edges_hz[min(members)]
        hi = edges_hz[max(members) + 1]

        ax.plot(tt, yy, linewidth=1.4)
        ax.set_ylabel("Drift")
        ax.set_title(
            f"Cluster {c}: {int(lo)} - {int(hi)} Hz | members={members}"
        )
        ax.grid(alpha=0.25)

    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(f"Merged Cluster Drifts ({CLUSTER_AGG})", y=1.0)
    plt.tight_layout()
    plt.show()


def plot_all_cluster_drifts_overlay(cluster_drifts, cluster_members, cluster_labels_unique,
                                    win_starts, edges_hz, sec_limit=None,
                                    normalize=True, event_curve=None,
                                    activity_curve=None):
    t = win_starts.copy()
    mask = get_time_mask(t, sec_limit)
    tt = t[mask]

    fig, ax = plt.subplots(figsize=(14, 5), dpi=FIG_DPI)

    for cd, members, c in zip(cluster_drifts, cluster_members, cluster_labels_unique):
        lo = edges_hz[min(members)]
        hi = edges_hz[max(members) + 1]
        yy = cd[mask]
        if normalize:
            yy = minmax01(yy)
        ax.plot(tt, yy, linewidth=1.6, label=f"C{c}: {int(lo)}-{int(hi)} Hz")

    if event_curve is not None:
        ev = event_curve[mask]
        if normalize:
            ev = minmax01(ev)
        ax.plot(tt, ev, linewidth=2.5, linestyle="--", label="Spectrogram event curve")

    if activity_curve is not None:
        ac = activity_curve[mask]
        if normalize:
            ac = minmax01(ac)
        ax.plot(tt, ac, linewidth=2.0, linestyle=":", label="Box-derived activity")

    ax.set_title("Merged Cluster Drifts + Event Curve + Box Activity")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Normalized magnitude" if normalize else "Magnitude")
    ax.grid(alpha=0.25)
    ax.legend(loc="upper right", fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()


def plot_spectrogram_with_cluster_drifts(y, sr, win_starts, cluster_drifts, cluster_members,
                                         cluster_labels_unique, edges_hz,
                                         event_curve=None, activity_curve=None,
                                         sec_limit=None):
    y2 = maybe_limit_time(y, sr, sec_limit)
    S = librosa.stft(y2, n_fft=2048, hop_length=512)
    S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)

    t = win_starts.copy()
    mask = get_time_mask(t, sec_limit)
    tt = t[mask]

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                             gridspec_kw={"height_ratios": [3, 2]}, dpi=FIG_DPI)

    img = librosa.display.specshow(
        S_db,
        sr=sr,
        hop_length=512,
        x_axis="time",
        y_axis="hz",
        fmin=0,
        fmax=FMAX,
        cmap="magma",
        ax=axes[0]
    )
    axes[0].set_ylim(0, FMAX)
    axes[0].set_title("Spectrogram")
    plt.colorbar(img, ax=axes[0], format="%+2.0f dB")

    for cd, members, c in zip(cluster_drifts, cluster_members, cluster_labels_unique):
        lo = edges_hz[min(members)]
        hi = edges_hz[max(members) + 1]
        yy = minmax01(cd[mask])
        axes[1].plot(tt, yy, linewidth=1.5, label=f"C{c}: {int(lo)}-{int(hi)} Hz")

    if event_curve is not None:
        ev = minmax01(event_curve[mask])
        axes[1].plot(tt, ev, linewidth=2.5, linestyle="--", label="Spectrogram event curve")

    if activity_curve is not None:
        ac = minmax01(activity_curve[mask])
        axes[1].plot(tt, ac, linewidth=2.0, linestyle=":", label="Box-derived activity")

    axes[1].set_title("Merged Cluster Drifts vs Event Curve vs Box Activity")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Normalized magnitude")
    axes[1].grid(alpha=0.25)
    axes[1].legend(loc="upper right", fontsize=8, ncol=2)

    plt.tight_layout()
    plt.show()


def plot_spectrogram_with_discovered_boxes(y, sr, boxes, gt_boxes=None, sec_limit=None):
    y2 = maybe_limit_time(y, sr, sec_limit)
    S = librosa.stft(y2, n_fft=2048, hop_length=512)
    S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)

    fig, ax = plt.subplots(figsize=(14, 6), dpi=FIG_DPI)
    img = librosa.display.specshow(
        S_db,
        sr=sr,
        hop_length=512,
        x_axis="time",
        y_axis="hz",
        fmin=0,
        fmax=FMAX,
        cmap="magma",
        ax=ax
    )
    ax.set_ylim(0, FMAX)
    ax.set_title("Spectrogram with Discovered Peak-Derived Boxes")

    for b in boxes:
        if sec_limit is not None and b["t0"] > sec_limit:
            continue

        rect = plt.Rectangle(
            (b["t0"], b["f0"]),
            b["t1"] - b["t0"],
            b["f1"] - b["f0"],
            fill=False,
            edgecolor="cyan",
            linewidth=1.5
        )
        ax.add_patch(rect)

    if gt_boxes is not None:
        for b in gt_boxes:
            if sec_limit is not None and b["t0"] > sec_limit:
                continue
            rect = plt.Rectangle(
                (b["t0"], b["f0"]),
                b["t1"] - b["t0"],
                b["f1"] - b["f0"],
                fill=False,
                edgecolor="lime",
                linewidth=1.2,
                linestyle="--"
            )
            ax.add_patch(rect)

    plt.colorbar(img, ax=ax, format="%+2.0f dB")
    plt.tight_layout()
    plt.show()


def print_cluster_event_metrics(metrics, cluster_members, cluster_labels_unique, edges_hz):
    print("\n=== CLUSTER vs SPECTROGRAM EVENT METRICS ===")
    for m, members, c in zip(metrics, cluster_members, cluster_labels_unique):
        lo = edges_hz[min(members)]
        hi = edges_hz[max(members) + 1]

        print(
            f"Cluster {c} | {int(lo):4d}-{int(hi):5d} Hz | members={members}\n"
            f"  Pearson r       : {m['pearson_r']:.3f}\n"
            f"  Spearman rho    : {m['spearman_rho']:.3f}\n"
            f"  Max lag corr    : {m['max_lag_corr']:.3f}  (lag={m['best_lag_windows']} windows)\n"
            f"  Peak precision  : {m['peak_precision']:.3f}\n"
            f"  Peak recall     : {m['peak_recall']:.3f}\n"
            f"  Peak F1         : {m['peak_f1']:.3f}\n"
            f"  Cluster peaks   : {m['n_cluster_peaks']}\n"
            f"  Event peaks     : {m['n_event_peaks']}"
        )


def print_similarity_diagnostics(S_time, S_freq, S_adj, S_total):
    print("\n=== SIMILARITY MATRIX DIAGNOSTICS ===")
    for name, S in [("Time", S_time), ("Freq", S_freq), ("Adj", S_adj), ("Total", S_total)]:
        off = S[~np.eye(S.shape[0], dtype=bool)]
        print(f"{name:>5s} | min={off.min():.3f} mean={off.mean():.3f} max={off.max():.3f}")



# -----------------------------
# BATCH HELPERS
# -----------------------------

def normalise_folder_name(name):
    """Normalise folder names for stable output grouping."""
    return re.sub(r"[^a-z0-9]+", "_", str(name).lower()).strip("_")


def expected_companion_gt(wav_path):
    """Return the exact companion annotation path written by the SERF splitter."""
    wav_path = Path(wav_path)
    return wav_path.with_name(f"{wav_path.stem}_annotations.csv")


def find_audio_files(audio_folder):
    """Find SERF 60-second audio clips recursively."""
    audio_folder = Path(audio_folder)
    if not audio_folder.exists():
        return []

    paths = []
    for pattern in AUDIO_EXTENSIONS:
        if RECURSIVE_AUDIO_SEARCH:
            paths.extend(audio_folder.rglob(pattern))
        else:
            paths.extend(audio_folder.glob(pattern))

    clean = []
    for path in sorted(set(paths)):
        # Ignore interrupted ffmpeg temporary outputs.
        if ".partial" in path.stem.lower():
            continue
        if REQUIRE_COMPANION_GT and not expected_companion_gt(path).exists():
            continue
        clean.append(path)
    return clean


def discover_dataset_folders(data_root):
    """
    Discover SERF retained 60-second clips.

    Clips are grouped by their first folder below DATA_ROOT, which normally
    corresponds to site_name in the splitter output. Each clip's annotation CSV
    is stored beside the audio rather than in a separate annotation directory.
    """
    data_root = Path(data_root)
    if not data_root.exists():
        raise FileNotFoundError(f"DATA_ROOT does not exist: {data_root}")

    audio_files = find_audio_files(data_root)
    grouped = {}

    for audio_path in audio_files:
        try:
            relative = audio_path.relative_to(data_root)
            dataset_name = relative.parts[0] if len(relative.parts) > 1 else "SERF_60s"
        except ValueError:
            dataset_name = "SERF_60s"

        grouped.setdefault(dataset_name, []).append(audio_path)

    datasets = []
    for dataset_name, clips in sorted(grouped.items(), key=lambda item: item[0].lower()):
        datasets.append({
            "dataset_name": dataset_name,
            "dataset_dir": data_root / dataset_name,
            "audio_dir": data_root / dataset_name,
            # GT is companion-per-file, so no common GT directory is required.
            "gt_dir": None,
            "audio_files": sorted(clips),
            "n_audio_files": len(clips),
        })

    return datasets


def find_matching_gt(wav_path, gt_folder=None):
    """
    Find the GT file for a SERF 60-second clip.

    Primary match:
        clip.flac -> clip_annotations.csv

    A few Raven-style fallbacks are retained so the rest of the pipeline can
    still be used with older data if required.
    """
    wav_path = Path(wav_path)
    stem = wav_path.stem

    sibling_candidates = [
        expected_companion_gt(wav_path),
        wav_path.with_name(f"{stem}_annotation.csv"),
        wav_path.with_name(f"{stem}.csv"),
        wav_path.with_name(f"{stem}.txt"),
        wav_path.with_name(f"{stem}.tsv"),
        wav_path.with_name(f"{stem}_SelectionTable.txt"),
        wav_path.with_name(f"{stem}_Selection Table.txt"),
    ]
    for candidate in sibling_candidates:
        if candidate.exists():
            return str(candidate)

    if gt_folder is None:
        return None

    gt_dir = Path(gt_folder)
    if not gt_dir.exists():
        return None

    candidates = []
    for ext in GT_EXTENSIONS:
        candidates.extend([
            gt_dir / f"{stem}{ext}",
            gt_dir / f"{stem}_annotations{ext}",
            gt_dir / f"{stem}_annotation{ext}",
            gt_dir / f"{stem}_SelectionTable{ext}",
            gt_dir / f"{stem}_Selection Table{ext}",
            gt_dir / f"{stem}.SelectionTable{ext}",
        ])

    for candidate in candidates:
        if candidate.exists():
            return str(candidate)

    search_iter = gt_dir.rglob("*") if RECURSIVE_GT_SEARCH else gt_dir.glob("*")
    for path in search_iter:
        if path.is_file() and path.suffix.lower() in GT_EXTENSIONS:
            if stem.lower() in path.stem.lower():
                return str(path)

    return None


def save_boxes_csv(boxes, out_csv):
    rows = []
    for b in boxes:
        rows.append({
            "cluster_id": b.get("cluster_id"),
            "t0": b.get("t0"),
            "t1": b.get("t1"),
            "duration_s": b.get("t1", 0) - b.get("t0", 0),
            "f0": b.get("f0"),
            "f1": b.get("f1"),
            "bandwidth_hz": b.get("f1", 0) - b.get("f0", 0),
            "peak_idx": b.get("peak_idx"),
            "support_left": b.get("support_left"),
            "support_right": b.get("support_right"),
            "members": ",".join(map(str, b.get("members", []))),
        })
    pd.DataFrame(rows).to_csv(out_csv, index=False)


def save_spectrogram_with_boxes(y, sr, boxes, gt_boxes, out_png, title, sec_limit=None):
    y2 = maybe_limit_time(y, sr, sec_limit)
    S = librosa.stft(y2, n_fft=2048, hop_length=512)
    S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)

    fig, ax = plt.subplots(figsize=(14, 6), dpi=FIG_DPI)
    img = librosa.display.specshow(
        S_db, sr=sr, hop_length=512, x_axis="time", y_axis="hz",
        fmin=0, fmax=FMAX, cmap="magma", ax=ax
    )
    ax.set_ylim(0, FMAX)
    ax.set_title(title)

    for b in boxes:
        if sec_limit is not None and b["t0"] > sec_limit:
            continue
        ax.add_patch(plt.Rectangle(
            (b["t0"], b["f0"]), b["t1"] - b["t0"], b["f1"] - b["f0"],
            fill=False, edgecolor="cyan", linewidth=1.3
        ))

    if gt_boxes is not None:
        for b in gt_boxes:
            if sec_limit is not None and b["t0"] > sec_limit:
                continue
            ax.add_patch(plt.Rectangle(
                (b["t0"], b["f0"]), b["t1"] - b["t0"], b["f1"] - b["f0"],
                fill=False, edgecolor="lime", linewidth=1.1, linestyle="--"
            ))

    plt.colorbar(img, ax=ax, format="%+2.0f dB")
    plt.tight_layout()
    fig.savefig(out_png, bbox_inches="tight")
    plt.close(fig)


def save_cluster_drifts_plot(cluster_drifts, cluster_members, cluster_labels_unique,
                             win_starts, edges_hz, out_png, title, sec_limit=None):
    t = win_starts.copy()
    mask = get_time_mask(t, sec_limit)
    n = len(cluster_drifts)
    fig, axes = plt.subplots(n, 1, figsize=(14, max(2.3 * n, 6)), sharex=True, dpi=FIG_DPI)
    if n == 1:
        axes = [axes]

    for ax, cd, members, c in zip(axes, cluster_drifts, cluster_members, cluster_labels_unique):
        lo = edges_hz[min(members)]
        hi = edges_hz[max(members) + 1]
        ax.plot(t[mask], cd[mask], linewidth=1.2)
        ax.set_ylabel("Drift")
        ax.set_title(f"Cluster {c}: {int(lo)}-{int(hi)} Hz | members={members}")
        ax.grid(alpha=0.25)

    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(title, y=1.0)
    plt.tight_layout()
    fig.savefig(out_png, bbox_inches="tight")
    plt.close(fig)


def gmm_boxes_from_cluster_drifts(cluster_drifts, cluster_members, cluster_labels_unique,
                                  edges_hz, win_starts, win_ends,
                                  min_event_posterior=0.50,
                                  smooth_labels_k=3,
                                  merge_gap_windows=2,
                                  min_event_windows=2,
                                  min_event_area_q=None,
                                  use_log_drift=False,
                                  random_state=0,
                                  verbose=False):
    all_boxes = []
    peak_info = []

    for cd, members, c in zip(cluster_drifts, cluster_members, cluster_labels_unique):
        f0 = float(edges_hz[min(members)])
        f1 = float(edges_hz[max(members) + 1])

        res = fit_gmm_event_state(
            cd,
            min_event_posterior=min_event_posterior,
            smooth_labels_k=smooth_labels_k,
            merge_gap_windows=merge_gap_windows,
            min_event_windows=min_event_windows,
            min_event_area_q=min_event_area_q,
            use_log_drift=use_log_drift,
            random_state=random_state,
        )

        labels = res["labels"]
        regions = res["regions"]
        cluster_boxes = []
        peaks = []

        for st, en in regions:
            segment = cd[st:en + 1]
            peak_idx = int(st + np.argmax(segment))
            peaks.append(peak_idx)
            cluster_boxes.append({
                "cluster_id": int(c),
                "t0": float(win_starts[st]),
                "t1": float(win_ends[en]),
                "f0": f0,
                "f1": f1,
                "peak_idx": int(peak_idx),
                "support_left": int(st),
                "support_right": int(en),
                "members": members,
            })

        cluster_boxes = merge_overlapping_time_boxes(cluster_boxes)
        all_boxes.extend(cluster_boxes)

        if verbose:
            means_str = np.array2string(res["means"], precision=3, separator=", ")
            weights_str = np.array2string(res["weights"], precision=3, separator=", ")
            event_fraction = float(np.mean(labels)) if len(labels) else 0.0
            print(
                f"Cluster {c} | {int(f0):4d}-{int(f1):5d} Hz | "
                f"thr≈{res['threshold']:.3f} | means={means_str} | "
                f"weights={weights_str} | event_fraction={event_fraction:.3f} | "
                f"regions={len(regions)} | boxes={len(cluster_boxes)}"
            )

        peak_info.append({
            "cluster_id": int(c),
            "freq_lo": f0,
            "freq_hi": f1,
            "n_peaks": int(len(peaks)),
            "n_boxes": int(len(cluster_boxes)),
            "peaks": peaks,
            "labels": labels,
            "posterior": res["posterior"],
            "regions": regions,
            "means": res["means"],
            "weights": res["weights"],
            "threshold": res["threshold"],
        })

    return all_boxes, peak_info


def score_boxes_against_event_curve(boxes, event_curve, win_starts, win_ends):
    activity = boxes_to_global_activity_curve(
        boxes=boxes, n_windows=len(win_starts),
        win_starts=win_starts, win_ends=win_ends,
        smooth_k=BOX_SMOOTH_ACTIVITY_K,
    )
    corr = safe_pearson(robust_zscore(activity), robust_zscore(event_curve))

    _, _, event_peaks, _ = detect_peaks_from_drift(
        event_curve, k_mad=EVENT_PEAK_K_MAD,
        min_sep_windows=EVENT_PEAK_MIN_SEP_WINDOWS,
    )

    pred_peaks = []
    for b in boxes:
        centre_t = 0.5 * (b["t0"] + b["t1"])
        pred_peaks.append(int(np.argmin(np.abs(win_starts - centre_t))))

    peak_scores = match_peaks_with_tolerance(event_peaks, pred_peaks, tol=PEAK_MATCH_TOL_WINDOWS)
    box_density = len(boxes) / max(1, len(win_starts))
    proxy = (
        PROXY_CORR_WEIGHT * corr
        + PROXY_PEAK_F1_WEIGHT * peak_scores["f1"]
        - PROXY_BOX_DENSITY_PENALTY * box_density
    )
    return {
        "proxy_score": float(proxy),
        "event_corr": float(corr),
        "event_peak_f1": float(peak_scores["f1"]),
        "event_peak_precision": float(peak_scores["precision"]),
        "event_peak_recall": float(peak_scores["recall"]),
        "box_density": float(box_density),
    }


def run_gmm_grid_search(cluster_drifts, cluster_members, uniq_clusters, edges_hz,
                        win_starts, win_ends, event_curve, gt_boxes):
    rows = []
    best = None

    for posterior in GRID_GMM_MIN_EVENT_POSTERIOR:
        for smooth_k in GRID_GMM_SMOOTH_LABELS_K:
            for gap in GRID_GMM_MERGE_GAP_WINDOWS:
                for min_w in GRID_GMM_MIN_EVENT_WINDOWS:
                    for use_log in GRID_GMM_USE_LOG_DRIFT:
                        for area_q in GRID_GMM_MIN_EVENT_AREA_Q:
                            boxes, _ = gmm_boxes_from_cluster_drifts(
                                cluster_drifts, cluster_members, uniq_clusters,
                                edges_hz, win_starts, win_ends,
                                min_event_posterior=posterior,
                                smooth_labels_k=smooth_k,
                                merge_gap_windows=gap,
                                min_event_windows=min_w,
                                min_event_area_q=area_q,
                                use_log_drift=use_log,
                                random_state=GMM_RANDOM_STATE,
                                verbose=False,
                            )

                            proxy = score_boxes_against_event_curve(boxes, event_curve, win_starts, win_ends)
                            gt_metrics = evaluate_predicted_boxes(boxes, gt_boxes, iou_thr=GT_IOU_THRESHOLD) if len(gt_boxes) else None

                            if gt_metrics is not None:
                                score = 0.65 * gt_metrics["f1"] + 0.20 * gt_metrics["mean_iou"] + 0.15 * proxy["event_corr"]
                            else:
                                score = proxy["proxy_score"]

                            row = {
                                "score": float(score),
                                "posterior": posterior,
                                "smooth_k": smooth_k,
                                "merge_gap": gap,
                                "min_event_windows": min_w,
                                "use_log_drift": use_log,
                                "min_event_area_q": area_q,
                                "n_boxes": len(boxes),
                                **proxy,
                            }
                            if gt_metrics is not None:
                                row.update({f"gt_{k}": v for k, v in gt_metrics.items()})
                            rows.append(row)

                            if best is None or row["score"] > best["score"]:
                                best = row

    return best, pd.DataFrame(rows).sort_values("score", ascending=False)


def compute_pipeline_for_file(wav_path):
    print(f"\n=== Processing {Path(wav_path).name} ===")
    y, sr = librosa.load(wav_path, sr=TARGET_SR, mono=True)
    y = apply_waveform_preprocessing(y, sr)
    duration_s = len(y) / sr

    win_starts, win_ends = make_fixed_windows(duration_s, WINDOW_S, HOP_S)

    repr_vectors = compute_window_repr_vectors(
        y=y, sr=sr, win_starts=win_starts, win_ends=win_ends,
        n_mels=N_MELS_FULL, fmin=FMIN, fmax=FMAX, use_pcen=USE_PCEN,
    )

    if DENOISE_METHOD == "median_specsub":
        repr_vectors = subtract_stationary_band_floor(
            repr_vectors, strength=SPEC_FLOOR_STRENGTH, floor_mode=SPEC_FLOOR_MODE
        )

    edges_hz = band_edges_hz(FMIN, FMAX, N_FINE_BANDS, scale=BAND_SCALE)
    mel_freqs = librosa.mel_frequencies(n_mels=N_MELS_FULL, fmin=FMIN, fmax=FMAX)
    band_vectors = split_mel_vectors_into_bands(repr_vectors, mel_freqs, edges_hz)
    band_drifts = compute_direct_band_drifts(band_vectors, smooth_k=DRIFT_SMOOTH_K)

    band_drifts_z, band_evidences = [], []
    for d in band_drifts:
        dz, ev, _, _ = detect_peaks_from_drift(d, k_mad=PEAK_K_MAD, min_sep_windows=PEAK_MIN_SEP_WINDOWS)
        band_drifts_z.append(dz)
        band_evidences.append(ev)

    band_descs = compute_band_frequency_descriptors(
        band_vectors, k_mad_activity=FREQ_DESC_ACTIVITY_K_MAD,
        normalize_subband=FREQ_DESC_NORMALIZE_SUBBAND,
        use_activity=FREQ_DESC_USE_ACTIVITY,
    )

    S_time = build_time_similarity_matrix(
        drifts=band_drifts_z, evidences=band_evidences,
        mode=SIMILARITY_MODE, max_lag=MAX_LAG_WINDOWS,
    )
    S_freq = build_frequency_similarity_matrix(band_descs)
    S_adj = build_adjacency_similarity_matrix(N_FINE_BANDS, tau=ADJ_TAU)
    S_total = combine_similarity_matrices(
        S_time=S_time, S_freq=S_freq, S_adj=S_adj,
        alpha=ALPHA_TIME, beta=BETA_FREQ, gamma=GAMMA_ADJ,
    )

    _, cluster_labels = cluster_bands_from_similarity(
        S_total, linkage_method=LINKAGE_METHOD,
        n_groups=N_GROUPS, distance_threshold=DISTANCE_THRESHOLD,
    )

    uniq_clusters, cluster_drifts, cluster_members = aggregate_cluster_drifts(
        drifts=band_drifts_z, cluster_labels=cluster_labels, agg=CLUSTER_AGG,
    )

    global_repr_vectors = compute_global_repr_vectors(
        y=y, sr=sr, win_starts=win_starts, win_ends=win_ends,
        n_mels=N_MELS_FULL, fmin=FMIN, fmax=FMAX,
    )

    if DENOISE_METHOD == "median_specsub":
        global_repr_vectors = subtract_stationary_band_floor(
            global_repr_vectors, strength=SPEC_FLOOR_STRENGTH, floor_mode=SPEC_FLOOR_MODE
        )

    event_curve = compute_spectrogram_event_curve(global_repr_vectors, smooth_k=EVENT_SMOOTH_K)

    return {
        "y": y, "sr": sr, "duration_s": duration_s,
        "win_starts": win_starts, "win_ends": win_ends,
        "edges_hz": edges_hz, "cluster_labels": cluster_labels,
        "uniq_clusters": uniq_clusters, "cluster_drifts": cluster_drifts,
        "cluster_members": cluster_members, "event_curve": event_curve,
    }



def get_file_output_paths(wav_path, out_dir, dataset_name):
    """
    Centralises output paths for one audio file so skip checks and saving
    always use the same folder/file naming convention.
    """
    wav_path = Path(wav_path)
    stem = wav_path.stem
    file_out_dir = Path(out_dir) / dataset_name / stem

    return {
        "file_out_dir": file_out_dir,
        "predicted_boxes_csv": file_out_dir / f"{stem}_predicted_boxes.csv",
        "done_marker": file_out_dir / f"{stem}_done.txt",
        "per_file_summary_csv": file_out_dir / f"{stem}_summary.csv",
        "grid_results_csv": file_out_dir / f"{stem}_grid_results.csv",
        "spectrogram_boxes_png": file_out_dir / f"{stem}_spectrogram_boxes.png",
        "cluster_drifts_png": file_out_dir / f"{stem}_cluster_drifts.png",
    }


def file_is_completed(wav_path, out_dir, dataset_name):
    """
    Returns True when this file appears to have completed successfully before.

    By default, this checks for the predicted boxes CSV. If
    REQUIRE_DONE_MARKER_FOR_SKIP=True, it also requires the done marker.
    """
    paths = get_file_output_paths(wav_path, out_dir, dataset_name)
    predicted_exists = paths["predicted_boxes_csv"].exists()
    marker_exists = paths["done_marker"].exists()

    if REQUIRE_DONE_MARKER_FOR_SKIP:
        return predicted_exists and marker_exists

    return predicted_exists or marker_exists


def build_skipped_row(wav_path, out_dir, gt_folder=None, dataset_name=None):
    """
    Builds a lightweight summary row for a skipped file.
    If a per-file summary exists, load it so previous metrics are preserved
    in the new batch_summary.csv.
    """
    wav_path = Path(wav_path)
    if dataset_name is None:
        dataset_name = "single_dataset"

    paths = get_file_output_paths(wav_path, out_dir, dataset_name)
    summary_csv = paths["per_file_summary_csv"]

    if summary_csv.exists():
        try:
            old = pd.read_csv(summary_csv)
            if len(old):
                row = old.iloc[0].to_dict()
                row["skipped_existing"] = True
                row["skip_reason"] = "existing_per_file_summary"
                return add_or_update_eval_metrics_for_existing_row(
                    row=row,
                    wav_path=wav_path,
                    out_dir=out_dir,
                    gt_folder=gt_folder,
                    dataset_name=dataset_name,
                )
        except Exception:
            pass

    gt_path = find_matching_gt(wav_path, gt_folder)
    row = {
        "dataset": dataset_name,
        "file": wav_path.name,
        "wav_path": str(wav_path),
        "gt_path": gt_path,
        "skipped_existing": True,
        "skip_reason": "existing_completed_outputs",
        "output_dir": str(paths["file_out_dir"]),
    }
    return add_or_update_eval_metrics_for_existing_row(
        row=row,
        wav_path=wav_path,
        out_dir=out_dir,
        gt_folder=gt_folder,
        dataset_name=dataset_name,
    )

def process_one_file(wav_path, out_dir, gt_folder=None, dataset_name=None):
    wav_path = Path(wav_path)
    stem = wav_path.stem

    if dataset_name is None:
        dataset_name = "single_dataset"

    # Keep outputs separated by dataset/site/session.
    paths = get_file_output_paths(wav_path, out_dir, dataset_name)
    file_out_dir = paths["file_out_dir"]
    file_out_dir.mkdir(parents=True, exist_ok=True)

    predicted_boxes_csv = paths["predicted_boxes_csv"]
    summary_marker = paths["done_marker"]
    per_file_summary_csv = paths["per_file_summary_csv"]

    if SKIP_COMPLETED_FILES and file_is_completed(wav_path, out_dir, dataset_name):
        print(f"Skipping completed file: {wav_path}")
        return build_skipped_row(
            wav_path=wav_path,
            out_dir=out_dir,
            gt_folder=gt_folder,
            dataset_name=dataset_name,
        )

    gt_path = find_matching_gt(wav_path, gt_folder)
    gt_boxes = try_load_ground_truth_boxes(gt_path) if gt_path is not None else []

    pipe = compute_pipeline_for_file(wav_path)

    params_used = {
        "posterior": GMM_MIN_EVENT_POSTERIOR,
        "smooth_k": GMM_SMOOTH_LABELS_K,
        "merge_gap": GMM_MERGE_GAP_WINDOWS,
        "min_event_windows": GMM_MIN_EVENT_WINDOWS,
        "use_log_drift": GMM_USE_LOG_DRIFT,
        "min_event_area_q": GMM_MIN_EVENT_AREA_Q,
    }

    grid_csv = None
    if RUN_PER_FILE_GRID_SEARCH:
        best, grid_df = run_gmm_grid_search(
            pipe["cluster_drifts"], pipe["cluster_members"], pipe["uniq_clusters"],
            pipe["edges_hz"], pipe["win_starts"], pipe["win_ends"],
            pipe["event_curve"], gt_boxes,
        )
        grid_csv = paths["grid_results_csv"]
        grid_df.to_csv(grid_csv, index=False)
        params_used = {
            "posterior": best["posterior"],
            "smooth_k": int(best["smooth_k"]),
            "merge_gap": int(best["merge_gap"]),
            "min_event_windows": int(best["min_event_windows"]),
            "use_log_drift": bool(best["use_log_drift"]),
            "min_event_area_q": best["min_event_area_q"] if not pd.isna(best["min_event_area_q"]) else None,
        }

    print("Running final GMM segmentation with params:", params_used)
    boxes, _ = gmm_boxes_from_cluster_drifts(
        pipe["cluster_drifts"], pipe["cluster_members"], pipe["uniq_clusters"],
        pipe["edges_hz"], pipe["win_starts"], pipe["win_ends"],
        min_event_posterior=params_used["posterior"],
        smooth_labels_k=params_used["smooth_k"],
        merge_gap_windows=params_used["merge_gap"],
        min_event_windows=params_used["min_event_windows"],
        min_event_area_q=params_used["min_event_area_q"],
        use_log_drift=params_used["use_log_drift"],
        random_state=GMM_RANDOM_STATE,
        verbose=True,
    )

    if SAVE_PREDICTED_BOX_CSVS:
        save_boxes_csv(boxes, predicted_boxes_csv)

    proxy = score_boxes_against_event_curve(boxes, pipe["event_curve"], pipe["win_starts"], pipe["win_ends"])
    gt_metrics = evaluate_predicted_boxes(
        boxes,
        gt_boxes,
        iou_thr=GT_IOU_THRESHOLD,
        win_starts=pipe["win_starts"],
        win_ends=pipe["win_ends"],
        duration_s=pipe["duration_s"],
        temporal_eval_hop_s=TEMPORAL_EVAL_HOP_S,
    ) if len(gt_boxes) else None

    activity_curve = boxes_to_global_activity_curve(
        boxes=boxes, n_windows=len(pipe["win_starts"]),
        win_starts=pipe["win_starts"], win_ends=pipe["win_ends"],
        smooth_k=BOX_SMOOTH_ACTIVITY_K,
    )
    global_cluster_drift = np.median(np.stack(pipe["cluster_drifts"], axis=0), axis=0)
    global_corr = safe_pearson(robust_zscore(global_cluster_drift), robust_zscore(pipe["event_curve"]))
    post_corr = safe_pearson(robust_zscore(activity_curve), robust_zscore(pipe["event_curve"]))

    if SAVE_PER_FILE_PLOTS:
        save_spectrogram_with_boxes(
            pipe["y"], pipe["sr"], boxes, gt_boxes if len(gt_boxes) else None,
            paths["spectrogram_boxes_png"],
            title=f"{dataset_name} / {stem}: predicted boxes (cyan)" + (" + GT (green)" if len(gt_boxes) else ""),
            sec_limit=PLOT_SECONDS_LIMIT,
        )

    if SAVE_CLUSTER_DRIFT_PLOTS:
        save_cluster_drifts_plot(
            pipe["cluster_drifts"], pipe["cluster_members"], pipe["uniq_clusters"],
            pipe["win_starts"], pipe["edges_hz"],
            paths["cluster_drifts_png"],
            title=f"{dataset_name} / {stem}: merged cluster drifts",
            sec_limit=PLOT_SECONDS_LIMIT,
        )

    row = {
        "dataset": dataset_name,
        "file": wav_path.name,
        "wav_path": str(wav_path),
        "gt_path": gt_path,
        "duration_s": pipe["duration_s"],
        "n_windows": len(pipe["win_starts"]),
        "n_clusters": len(pipe["uniq_clusters"]),
        "n_pred_boxes": len(boxes),
        "n_gt_boxes": len(gt_boxes),
        "global_drift_event_corr": global_corr,
        "box_activity_event_corr": post_corr,
        **{f"proxy_{k}": v for k, v in proxy.items()},
        **{f"param_{k}": v for k, v in params_used.items()},
        "grid_results_csv": str(grid_csv) if grid_csv is not None else None,
        "skipped_existing": False,
        "output_dir": str(file_out_dir),
    }

    if gt_metrics is not None:
        row.update({f"gt_{k}": v for k, v in gt_metrics.items()})
    else:
        row.update({
            "gt_tp": None, "gt_fp": None, "gt_fn": None,
            "gt_precision": None, "gt_recall": None,
            "gt_f1": None, "gt_mean_iou": None,
        })

    if SAVE_PER_FILE_SUMMARY:
        pd.DataFrame([row]).to_csv(per_file_summary_csv, index=False)

    summary_marker.write_text("done\n", encoding="utf-8")

    # Release the audio and feature arrays before processing the next 60-second clip.
    del pipe
    plt.close("all")
    gc.collect()
    return row



# -----------------------------
# EVALUATION TABLES
# -----------------------------

def _sum_numeric(df, col):
    if col not in df.columns:
        return 0.0
    return float(pd.to_numeric(df[col], errors="coerce").fillna(0).sum())


def _mean_numeric(df, col):
    if col not in df.columns:
        return None
    vals = pd.to_numeric(df[col], errors="coerce").dropna()
    return float(vals.mean()) if len(vals) else None


def _weighted_mean_numeric(df, value_col, weight_col):
    if value_col not in df.columns or weight_col not in df.columns:
        return None
    vals = pd.to_numeric(df[value_col], errors="coerce")
    weights = pd.to_numeric(df[weight_col], errors="coerce").fillna(0)
    mask = vals.notna() & (weights > 0)
    if not mask.any():
        return None
    return float(np.average(vals[mask], weights=weights[mask]))


def _aggregate_eval_group(df, label):
    gt_files = int(df["n_gt_boxes"].notna().sum()) if "n_gt_boxes" in df.columns else 0
    files = int(len(df))
    n_pred = _sum_numeric(df, "n_pred_boxes")
    n_gt = _sum_numeric(df, "n_gt_boxes")
    tp = _sum_numeric(df, "gt_tp")
    fp = _sum_numeric(df, "gt_fp")
    fn = _sum_numeric(df, "gt_fn")

    strict_precision = _safe_div(tp, tp + fp)
    strict_recall = _safe_div(tp, tp + fn)
    strict_f1 = _safe_div(2 * strict_precision * strict_recall, strict_precision + strict_recall)

    # Temporal binary metrics are micro-averaged by summing windows.
    t_tp = _sum_numeric(df, "gt_temporal_tp_windows")
    t_fp = _sum_numeric(df, "gt_temporal_fp_windows")
    t_fn = _sum_numeric(df, "gt_temporal_fn_windows")
    t_tn = _sum_numeric(df, "gt_temporal_tn_windows")
    t_n = t_tp + t_fp + t_fn + t_tn
    t_precision = _safe_div(t_tp, t_tp + t_fp)
    t_recall = _safe_div(t_tp, t_tp + t_fn)
    t_f1 = _safe_div(2 * t_precision * t_recall, t_precision + t_recall)
    t_iou = _safe_div(t_tp, t_tp + t_fp + t_fn)
    t_accuracy = _safe_div(t_tp + t_tn, t_n)
    t_specificity = _safe_div(t_tn, t_tn + t_fp)
    t_bal_acc = 0.5 * (t_recall + t_specificity)

    matched_weight = "gt_tp"
    gt_weight = "n_gt_boxes"

    return {
        "scope": label,
        "files": files,
        "files_with_gt": gt_files,
        "pred_boxes": int(n_pred),
        "gt_boxes": int(n_gt),
        "pred_per_gt": _safe_div(n_pred, n_gt),
        "strict_tp": int(tp),
        "strict_fp": int(fp),
        "strict_fn": int(fn),
        "strict_precision": strict_precision,
        "strict_hit_rate": strict_recall,
        "strict_f1": strict_f1,
        "matched_iou_mean": _weighted_mean_numeric(df, "gt_mean_iou", matched_weight),
        "matched_temporal_iou_mean": _weighted_mean_numeric(df, "gt_mean_temporal_iou", matched_weight),
        "matched_frequency_iou_mean": _weighted_mean_numeric(df, "gt_mean_frequency_iou", matched_weight),
        "best_iou_mean": _weighted_mean_numeric(df, "gt_best_iou_mean", gt_weight),
        "best_temporal_iou_mean": _weighted_mean_numeric(df, "gt_best_temporal_iou_mean", gt_weight),
        "best_frequency_iou_mean": _weighted_mean_numeric(df, "gt_best_frequency_iou_mean", gt_weight),
        "temporal_coverage_mean": _weighted_mean_numeric(df, "gt_temporal_coverage_mean", gt_weight),
        "frequency_coverage_mean": _weighted_mean_numeric(df, "gt_frequency_coverage_mean", gt_weight),
        "iou_hit_rate_mean": _weighted_mean_numeric(df, "gt_iou_hit_rate", gt_weight),
        "any_overlap_hit_rate_mean": _weighted_mean_numeric(df, "gt_any_overlap_hit_rate", gt_weight),
        "temporal_accuracy": t_accuracy,
        "temporal_balanced_accuracy": t_bal_acc,
        "temporal_precision": t_precision,
        "temporal_hit_rate": t_recall,
        "temporal_f1": t_f1,
        "temporal_iou": t_iou,
    }


def _format_eval_table(df):
    out = df.copy()
    int_cols = [
        "files", "files_with_gt", "pred_boxes", "gt_boxes",
        "strict_tp", "strict_fp", "strict_fn",
    ]
    for c in int_cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0).astype(int)

    float_cols = [c for c in out.columns if c not in ["scope"] + int_cols]
    for c in float_cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").map(
                lambda x: "" if pd.isna(x) else f"{x:.3f}"
            )
    return out


def build_and_print_evaluation_tables(summary_df, out_dir):
    if summary_df is None or not len(summary_df):
        return None, None

    if "n_gt_boxes" not in summary_df.columns:
        print("\nNo GT evaluation columns found; skipping evaluation tables.")
        return None, None

    eval_df = summary_df.copy()
    eval_df["n_gt_boxes"] = pd.to_numeric(eval_df["n_gt_boxes"], errors="coerce")
    eval_df = eval_df[eval_df["n_gt_boxes"].fillna(0) > 0].copy()

    if not len(eval_df):
        print("\nNo files with GT boxes found; skipping evaluation tables.")
        return None, None

    overall = pd.DataFrame([_aggregate_eval_group(eval_df, "OVERALL")])
    per_site = pd.DataFrame([
        _aggregate_eval_group(g, str(site))
        for site, g in eval_df.groupby("dataset", dropna=False)
    ]).sort_values("scope")

    if SAVE_EVALUATION_TABLES:
        overall.to_csv(Path(out_dir) / "evaluation_overall.csv", index=False)
        per_site.to_csv(Path(out_dir) / "evaluation_by_site.csv", index=False)
        _format_eval_table(overall).to_csv(Path(out_dir) / "evaluation_overall_pretty.csv", index=False)
        _format_eval_table(per_site).to_csv(Path(out_dir) / "evaluation_by_site_pretty.csv", index=False)

    # A compact terminal table that is still readable.
    compact_cols = [
        "scope", "files", "pred_boxes", "gt_boxes", "pred_per_gt",
        "strict_precision", "strict_hit_rate", "strict_f1",
        "matched_iou_mean", "matched_temporal_iou_mean", "matched_frequency_iou_mean",
        "temporal_coverage_mean", "frequency_coverage_mean",
        "any_overlap_hit_rate_mean", "temporal_accuracy", "temporal_hit_rate",
    ]
    compact_overall = _format_eval_table(overall[[c for c in compact_cols if c in overall.columns]])
    compact_site = _format_eval_table(per_site[[c for c in compact_cols if c in per_site.columns]])

    print("\n=== EVALUATION SUMMARY: OVERALL ===")
    print(compact_overall.to_string(index=False))

    print("\n=== EVALUATION SUMMARY: PER SITE ===")
    print(compact_site.to_string(index=False))

    print("\nEvaluation table notes:")
    print("  strict_* = one-to-one GT-confirmed box matching; unmatched predictions count as false positives.")
    print("  GT coverage / overlap hit rate = GT-centred coverage metrics; less punitive when GT is incomplete.")
    print("  temporal_* = time-only binary activity metrics over analysis windows; useful for temporal alignment.")

    return overall, per_site


# -----------------------------
# MAIN: SERF 60-SECOND CLIP SCAN
# -----------------------------

if __name__ == "__main__":
    out_dir = Path(OUTPUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)

    datasets = discover_dataset_folders(DATA_ROOT)

    # Optional single-clip fallback.
    if not datasets:
        if WAV_PATH is not None and Path(WAV_PATH).exists():
            fallback_gt_dir = Path(GT_PATH).parent if GT_PATH is not None else None
            datasets = [{
                "dataset_name": "single_clip",
                "dataset_dir": Path(WAV_PATH).parent,
                "audio_dir": Path(WAV_PATH).parent,
                "gt_dir": fallback_gt_dir,
                "audio_files": [Path(WAV_PATH)],
                "n_audio_files": 1,
            }]
        else:
            requirement = (
                " with companion *_annotations.csv files"
                if REQUIRE_COMPANION_GT
                else ""
            )
            raise FileNotFoundError(
                f"No SERF 60-second audio clips{requirement} were found below "
                f"DATA_ROOT={DATA_ROOT}"
            )

    total_files = sum(len(d["audio_files"]) for d in datasets)

    print(f"Found {len(datasets)} SERF site groups")
    print(f"Found {total_files} audio files total")
    print(f"Output directory: {out_dir.resolve()}")
    print(f"RUN_PER_FILE_GRID_SEARCH = {RUN_PER_FILE_GRID_SEARCH}")
    print(f"SKIP_COMPLETED_FILES = {SKIP_COMPLETED_FILES}")

    for dataset in datasets:
        print(
            f"  - {dataset['dataset_name']}: "
            f"{len(dataset['audio_files'])} audio files | "
            f"audio={dataset['audio_dir']} | "
            f"GT=companion *_annotations.csv"
        )

    summary_rows = []
    failures = []
    file_counter = 0

    for dataset in datasets:
        dataset_name = dataset["dataset_name"]
        audio_files = dataset["audio_files"]
        gt_dir = dataset["gt_dir"]

        print(f"\n=== Dataset: {dataset_name} ===")

        for wav_path in audio_files:
            file_counter += 1
            print(f"\n[{file_counter}/{total_files}] {wav_path}")

            try:
                row = process_one_file(
                    wav_path=wav_path,
                    out_dir=out_dir,
                    gt_folder=gt_dir,
                    dataset_name=dataset_name,
                )
                summary_rows.append(row)

                if row.get("skipped_existing"):
                    print(f"Skipped existing output: {row.get('output_dir')}")
                    continue

                if row.get("gt_f1") is not None and not pd.isna(row.get("gt_f1")):
                    print(
                        f"GT strict F1={row['gt_f1']:.3f} | P={row['gt_precision']:.3f} | "
                        f"hit={row['gt_hit_rate']:.3f} | IoU={row['gt_mean_iou']:.3f} | "
                        f"temp_cov={row.get('gt_temporal_coverage_mean', 0):.3f} | "
                        f"freq_cov={row.get('gt_frequency_coverage_mean', 0):.3f} | "
                        f"boxes={row['n_pred_boxes']}/{row['n_gt_boxes']}"
                    )
                else:
                    print(
                        f"No GT | proxy={row['proxy_proxy_score']:.3f} | "
                        f"event_corr={row['box_activity_event_corr']:.3f} | boxes={row['n_pred_boxes']}"
                    )

            except Exception as e:
                print(f"FAILED: {wav_path}: {e}")
                traceback.print_exc()
                plt.close("all")
                gc.collect()
                failures.append({
                    "dataset": dataset_name,
                    "file": str(wav_path),
                    "error": repr(e),
                })

    summary_df = pd.DataFrame(summary_rows)
    summary_csv = out_dir / "batch_summary.csv"
    summary_df.to_csv(summary_csv, index=False)

    build_and_print_evaluation_tables(summary_df, out_dir)

    if len(summary_df) and "skipped_existing" in summary_df.columns:
        skipped_df = summary_df[summary_df["skipped_existing"] == True].copy()
        if len(skipped_df):
            skipped_csv = out_dir / "batch_skipped_existing.csv"
            skipped_df.to_csv(skipped_csv, index=False)
            print(f"\nSkipped existing files: {len(skipped_df)}. See {skipped_csv}")

    if len(summary_df):
        if "skipped_existing" in summary_df.columns:
            rankable_df = summary_df[summary_df["skipped_existing"] != True].copy()
        else:
            rankable_df = summary_df.copy()

        if len(rankable_df):
            if "gt_f1" in rankable_df.columns and rankable_df["gt_f1"].notna().any():
                ranked = rankable_df.sort_values(
                    ["gt_f1", "gt_mean_iou", "box_activity_event_corr"],
                    ascending=False,
                )
            else:
                ranked = rankable_df.sort_values(
                    ["box_activity_event_corr", "proxy_proxy_score"],
                    ascending=False,
                )

            ranked.to_csv(out_dir / "batch_summary_ranked.csv", index=False)

            print("\n=== TOP FILES ===")
            show_cols = [
                c for c in [
                    "dataset", "file", "gt_f1", "gt_recall", "gt_precision",
                    "gt_mean_iou", "box_activity_event_corr", "n_pred_boxes",
                    "n_gt_boxes", "output_dir"
                ] if c in ranked.columns
            ]
            print(ranked[show_cols].head(15).to_string(index=False))

    if failures:
        failures_csv = out_dir / "batch_failures.csv"
        pd.DataFrame(failures).to_csv(failures_csv, index=False)
        print(f"\nFailures: {len(failures)}. See {failures_csv}")

    print(f"\nDone. Summary: {summary_csv}")
